# Figure Helper Code

In [ ]:
an_color ="#E69F00"

In [ ]:
# === Artboard + panel helpers (2x2)with axis labels ===
from PIL import Image, ImageChops, ImageDraw, ImageFont
from collections import OrderedDict
from pathlib import Path
import math
import os
import re

FIG_W_IN = 6.7
FIG_H_IN = 8.0
DPI = 300

ML_IN = MR_IN = MT_IN = MB_IN = 0.25
ROW_GAP_IN = COL_GAP_IN = 0.08
LABEL_PAD_IN = 0.02

AXIS_PAD_LEFT_IN = 0.90
AXIS_PAD_TOP_IN  = 0.30

def inches_to_px(inches, dpi=DPI):
    return int(round(inches * dpi))

fig_w_px = inches_to_px(FIG_W_IN + AXIS_PAD_LEFT_IN, DPI)
fig_h_px = inches_to_px(FIG_H_IN + AXIS_PAD_TOP_IN, DPI)

ml = inches_to_px(AXIS_PAD_LEFT_IN + ML_IN, DPI)
mr = inches_to_px(MR_IN, DPI)
mt = inches_to_px(AXIS_PAD_TOP_IN + MT_IN, DPI)
mb = inches_to_px(MB_IN, DPI)

def load_font_safe(path: str, size: int):
    try:
        return ImageFont.truetype(path, size)
    except Exception:
        return ImageFont.load_default()

def text_size(draw, text, font):
    try:
        bbox = font.getbbox(text)
        return bbox[2] - bbox[0], bbox[3] - bbox[1]
    except Exception:
        return draw.textsize(text, font=font)
    


def normalize_rotation(rotate):
    if rotate is None:
        return 0.0
    if isinstance(rotate, str):
        key = rotate.strip().lower()
        if key in ("right", "r", "cw", "clockwise"):
            return -90.0
        if key in ("left", "l", "ccw", "counterclockwise", "counter-clockwise"):
            return 90.0
    try:
        rot = float(rotate)
    except Exception:
        return 0.0
    rot = rot % 360.0
    if rot > 180.0:
        rot -= 360.0
    return 0.0 if abs(rot) < 1e-9 else rot


def resolve_panel_rotation(view=None, rotate=None):
    view = view or {}
    return normalize_rotation(view.get("rotate", rotate))


def normalize_padding(padding_px):
    if isinstance(padding_px, dict):
        return tuple(int(max(0, padding_px.get(k, 0))) for k in ("left", "top", "right", "bottom"))
    pad = int(max(0, padding_px or 0))
    return (pad, pad, pad, pad)


def panel_padding(horizontal=0, vertical=0, left=None, right=None, top=None, bottom=None):
    horizontal = int(max(0, horizontal or 0))
    vertical = int(max(0, vertical or 0))
    return {
        "left": horizontal if left is None else int(max(0, left)),
        "right": horizontal if right is None else int(max(0, right)),
        "top": vertical if top is None else int(max(0, top)),
        "bottom": vertical if bottom is None else int(max(0, bottom)),
    }


def set_panel_padding(panels, panel_id, horizontal=0, vertical=0, left=None, right=None, top=None, bottom=None, color="white"):
    cfg = panels.setdefault(panel_id, {})
    cfg["padding_px"] = panel_padding(horizontal=horizontal, vertical=vertical, left=left, right=right, top=top, bottom=bottom)
    cfg["padding_color"] = color
    return panels


def load_image_with_adjustments(img_path, crop_bottom_px=0, rotate=None, padding_px=0, padding_color="white"):
    img = Image.open(img_path)
    if crop_bottom_px > 0:
        w, h = img.size
        crop_bottom_px = min(crop_bottom_px, h - 1)
        img = img.crop((0, 0, w, h - crop_bottom_px))
    rot = normalize_rotation(rotate)
    if rot:
        img = img.rotate(rot, expand=True, resample=Image.Resampling.BICUBIC)
    if img.mode not in ("L", "RGB"):
        img = img.convert("RGB")
    pad_left, pad_top, pad_right, pad_bottom = normalize_padding(padding_px)
    if pad_left or pad_top or pad_right or pad_bottom:
        padded = Image.new("RGB", (img.width + pad_left + pad_right, img.height + pad_top + pad_bottom), padding_color)
        padded.paste(img, (pad_left, pad_top))
        img = padded
    return img




def add_zoom_box_from_zoomed_panel(canvas, layout, panels, base_id, zoom_id, color="white", width=8):
    # Panel geometry
    x0, y0, pw_base, ph_base = layout[base_id]
    xz, yz, pw_zoom, ph_zoom = layout[zoom_id]

    # Panel configs
    cfg_base = panels[base_id]
    cfg_zoom = panels[zoom_id]
    view_base = cfg_base.get("view", {"zoom": 1.0, "x": 0.5, "y": 0.5})
    view_zoom = cfg_zoom.get("view", {"zoom": 1.0, "x": 0.5, "y": 0.5})
    crop_base = cfg_base.get("crop_bottom_px", 0)
    crop_zoom = cfg_zoom.get("crop_bottom_px", 0)
    rotate_base = resolve_panel_rotation(view_base, cfg_base.get("rotate"))
    rotate_zoom = resolve_panel_rotation(view_zoom, cfg_zoom.get("rotate"))
    path = cfg_zoom["path"]  # assumes same image; change if different

    # Image and scales
    img = load_image_with_adjustments(path, crop_bottom_px=crop_zoom, rotate=rotate_zoom)
    img_w, img_h = img.size

    scale_base = compute_panel_scale(path, pw_base, ph_base, view=view_base, crop_bottom_px=crop_base, rotate=rotate_base)
    scale_zoom = compute_panel_scale(path, pw_zoom, ph_zoom, view=view_zoom, crop_bottom_px=crop_zoom, rotate=rotate_zoom)

    # Base view crop origin in scaled coords
    extra_base_w = img_w * scale_base - pw_base
    extra_base_h = img_h * scale_base - ph_base
    crop_base_x = float(view_base.get("x", 0.5)) * max(0, extra_base_w)
    crop_base_y = float(view_base.get("y", 0.5)) * max(0, extra_base_h)

    # Zoom view crop origin in scaled coords
    extra_zoom_w = img_w * scale_zoom - pw_zoom
    extra_zoom_h = img_h * scale_zoom - ph_zoom
    crop_zoom_x = float(view_zoom.get("x", 0.5)) * max(0, extra_zoom_w)
    crop_zoom_y = float(view_zoom.get("y", 0.5)) * max(0, extra_zoom_h)

    # Region in image coords from zoom view
    region_img_w = pw_zoom / scale_zoom
    region_img_h = ph_zoom / scale_zoom
    region_img_x = crop_zoom_x / scale_zoom
    region_img_y = crop_zoom_y / scale_zoom

    # Map to base panel coords
    rect_w = region_img_w * scale_base
    rect_h = region_img_h * scale_base
    rect_x = region_img_x * scale_base - crop_base_x
    rect_y = region_img_y * scale_base - crop_base_y

    draw = ImageDraw.Draw(canvas)
    draw.rectangle(
        [x0 + rect_x, y0 + rect_y, x0 + rect_x + rect_w, y0 + rect_y + rect_h],
        outline=color,
        width=width,
    )
    return canvas


def set_panel_border(panels, panel_id, color="white", thickness=4):
    cfg = panels.setdefault(panel_id, {})
    cfg["border"] = {"color": color, "thickness": thickness}
    return panels


def add_panel_box(canvas, layout, panel_id, x, y, w, h, color="yellow", thickness=8):
    x0, y0, pw, ph = layout[panel_id]
    left = x0 + int(x * pw)
    top = y0 + int(y * ph)
    right = left + int(w * pw)
    bottom = top + int(h * ph)
    draw = ImageDraw.Draw(canvas)
    for i in range(int(thickness)):
        draw.rectangle([left - i, top - i, right + i, bottom + i], outline=color)
    return canvas


def add_panel_boxes(canvas, layout, boxes):
    for box in boxes:
        box = dict(box)
        panel_id = box.pop("panel_id")
        canvas = add_panel_box(canvas, layout, panel_id, **box)
    return canvas


def add_group_border(canvas, layout, panel_ids, color="black", thickness=4):
    boxes = [layout[panel_id] for panel_id in panel_ids]
    x0 = min(box[0] for box in boxes)
    y0 = min(box[1] for box in boxes)
    x1 = max(box[0] + box[2] for box in boxes)
    y1 = max(box[1] + box[3] for box in boxes)
    draw = ImageDraw.Draw(canvas)
    for i in range(int(thickness)):
        draw.rectangle([x0 - i, y0 - i, x1 + i, y1 + i], outline=color)
    return canvas


def make_channel_panel(path, view, border_color, label="", channel_label=None,
                       scale_bar_full_px=251, scale_bar_orig_um=20, scale_bar_new_um=15,
                       scale_bar_color="white", scale_bar_thickness=12,
                       label_color="white", label_font_size=72):
    annotations = []
    if channel_label:
        annotations.append(channel_label)
    return {
        "path": path,
        "view": view,
        "fit": "cover",
        "background": "white",
        "label": label,
        "label_color": label_color,
        "label_font_size": label_font_size,
        "scale_bar": {
            "full_px": scale_bar_full_px,
            "orig_um": scale_bar_orig_um,
            "new_um": scale_bar_new_um,
            "color": scale_bar_color,
            "thickness": scale_bar_thickness,
            "number": False,
        },
        "annotations": annotations,
        "border": {"color": border_color, "thickness": 12},
    }


def channel_text_label(text, x=0.60, y=0.83, size=60, color="white", anchor="left"):
    return {"text": text, "x": x, "y": y, "size": size, "color": color, "anchor": anchor}


def make_channel_label_style(x=0.94, y=0.60, size=60, anchor="right"):
    return {"x": x, "y": y, "size": size, "anchor": anchor}


def build_channel_group_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px, group_panel_ids):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    n_rows = (len(group_panel_ids) + 1) // 2
    row_h = (inner_h - row_gap_px * (n_rows - 1)) // n_rows
    row_heights = [row_h] * n_rows
    row_heights[-1] += inner_h - row_gap_px * (n_rows - 1) - sum(row_heights)

    group_w = (inner_w - col_gap_px) // 2
    group_w_last = inner_w - col_gap_px - group_w

    layout = {}
    y = mt

    def add_group(panel_ids, x, y, w, h):
        small_w = w // 2
        small_w_last = w - small_w
        small_h = h // 2
        small_h_last = h - small_h
        top_left_id, top_right_id, bottom_left_id, bottom_right_id = panel_ids
        layout[top_left_id] = (x, y, small_w, small_h)
        layout[top_right_id] = (x + small_w, y, small_w_last, small_h)
        layout[bottom_left_id] = (x, y + small_h, small_w, small_h_last)
        layout[bottom_right_id] = (x + small_w, y + small_h, small_w_last, small_h_last)

    for row_idx in range(n_rows):
        left_group = group_panel_ids[row_idx * 2]
        right_group = group_panel_ids[row_idx * 2 + 1] if row_idx * 2 + 1 < len(group_panel_ids) else None
        h = row_heights[row_idx]
        if right_group is None:
            x = ml + (inner_w - group_w) // 2
            add_group(left_group, x, y, group_w, h)
        else:
            add_group(left_group, ml, y, group_w, h)
            add_group(right_group, ml + group_w + col_gap_px, y, group_w_last, h)
        y += h + row_gap_px

    return layout


def build_variable_channel_group_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px, group_panel_ids):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    n_rows = (len(group_panel_ids) + 1) // 2
    row_h = (inner_h - row_gap_px * (n_rows - 1)) // n_rows
    row_heights = [row_h] * n_rows
    row_heights[-1] += inner_h - row_gap_px * (n_rows - 1) - sum(row_heights)

    group_w = (inner_w - col_gap_px) // 2
    group_w_last = inner_w - col_gap_px - group_w

    layout = {}
    y = mt

    def add_group(panel_ids, x, y, w, h):
        if len(panel_ids) == 4:
            small_w = w // 2
            small_w_last = w - small_w
            small_h = h // 2
            small_h_last = h - small_h
            top_left_id, top_right_id, bottom_left_id, bottom_right_id = panel_ids
            layout[top_left_id] = (x, y, small_w, small_h)
            layout[top_right_id] = (x + small_w, y, small_w_last, small_h)
            layout[bottom_left_id] = (x, y + small_h, small_w, small_h_last)
            layout[bottom_right_id] = (x + small_w, y + small_h, small_w_last, small_h_last)
            return

        if len(panel_ids) == 3:
            first_w = w // 3
            second_w = w // 3
            third_w = w - first_w - second_w
            widths = [first_w, second_w, third_w]
            panel_x = x
            for panel_id, panel_w in zip(panel_ids, widths):
                layout[panel_id] = (panel_x, y, panel_w, h)
                panel_x += panel_w
            return

        if len(panel_ids) == 1:
            layout[panel_ids[0]] = (x, y, w, h)
            return

        raise ValueError(f"Expected 1, 3, or 4 panels per group, got {len(panel_ids)}: {panel_ids}")

    for row_idx in range(n_rows):
        left_group = group_panel_ids[row_idx * 2]
        right_group = group_panel_ids[row_idx * 2 + 1] if row_idx * 2 + 1 < len(group_panel_ids) else None
        h = row_heights[row_idx]
        if right_group is None:
            x = ml + (inner_w - group_w) // 2
            add_group(left_group, x, y, group_w, h)
        else:
            add_group(left_group, ml, y, group_w, h)
            add_group(right_group, ml + group_w + col_gap_px, y, group_w_last, h)
        y += h + row_gap_px

    return layout


FRONS_BORDER_COLOR = "#B85C4F"
LABRUM_BORDER_COLOR = "#556E9B"
MAXILLA_BORDER_COLOR = "#8A9562"
LABIUM_BORDER_COLOR = "#d7c7a0"

HEAD_REGION_BORDER_COLORS = {
    "frons": FRONS_BORDER_COLOR,
    "labrum": LABRUM_BORDER_COLOR,
    "maxilla": MAXILLA_BORDER_COLOR,
    "labium": LABIUM_BORDER_COLOR,
}

FIGURE1_HEAD_REGION_BY_PANEL = {
    "F": "frons",
    "G": "labrum",
    "H": "maxilla",
    "I": "maxilla",
    "J": "labium",
    "K": "labium",
}


def set_head_region_border(panels, panel_id, region, thickness=12):
    return set_panel_border(panels, panel_id, HEAD_REGION_BORDER_COLORS[region], thickness)


def set_figure1_head_region_borders(panels, thickness=12):
    for panel_id, region in FIGURE1_HEAD_REGION_BY_PANEL.items():
        set_head_region_border(panels, panel_id, region, thickness)
    return panels


def set_panel_label_size(panels, panel_id, size, pad_px=None, color=None):
    cfg = panels.setdefault(panel_id, {})
    cfg["label_font_size"] = int(size)
    if pad_px is not None:
        cfg["label_pad_px"] = int(pad_px)
    if color is not None:
        cfg["label_color"] = color
    return panels



# Shared bounded caches cover every figure cell. File size and modification time are
# included in the keys so replacing a source image invalidates stale entries.
FIGURE_TILE_CACHE = globals().get("FIGURE_TILE_CACHE", OrderedDict())
FIGURE_SCALE_CACHE = globals().get("FIGURE_SCALE_CACHE", OrderedDict())
FIGURE_TILE_CACHE_MAX_ITEMS = 32
FIGURE_SCALE_CACHE_MAX_ITEMS = 256


def _freeze_cache_value(value):
    if isinstance(value, dict):
        return tuple(sorted((str(k), _freeze_cache_value(v)) for k, v in value.items()))
    if isinstance(value, (list, tuple)):
        return tuple(_freeze_cache_value(v) for v in value)
    return value


def _image_file_signature(img_path):
    resolved = os.path.normcase(os.path.abspath(os.fspath(img_path)))
    try:
        stat = os.stat(resolved)
        return resolved, stat.st_size, stat.st_mtime_ns
    except OSError:
        return resolved, None, None


def _bounded_cache_put(cache, key, value, max_items):
    cache[key] = value
    cache.move_to_end(key)
    while len(cache) > max_items:
        cache.popitem(last=False)


def clear_figure_caches():
    """Clear shared render caches after source-image or helper changes."""
    FIGURE_TILE_CACHE.clear()
    FIGURE_SCALE_CACHE.clear()


def paste_with_view(img_path, panel_w, panel_h, view=None, crop_bottom_px=0, rotate=None, padding_px=0, padding_color="white", fit="cover", background="white"):
    rotate = resolve_panel_rotation(view, rotate)
    key = (
        _image_file_signature(img_path), int(panel_w), int(panel_h),
        _freeze_cache_value(view or {}), int(crop_bottom_px), rotate,
        _freeze_cache_value(padding_px), str(padding_color), str(fit), str(background),
    )
    cached = FIGURE_TILE_CACHE.get(key)
    if cached is not None:
        FIGURE_TILE_CACHE.move_to_end(key)
        return cached.copy()

    img = load_image_with_adjustments(img_path, crop_bottom_px=crop_bottom_px, rotate=rotate, padding_px=padding_px, padding_color=padding_color)
    img_w, img_h = img.size
    fit = (fit or "cover").lower()
    base_scale = min(panel_w / img_w, panel_h / img_h) if fit == "contain" else max(panel_w / img_w, panel_h / img_h)
    view = view or {}
    zoom = float(view.get("zoom", 1.0))
    x_frac = max(0.0, min(1.0, float(view.get("x", 0.5))))
    y_frac = max(0.0, min(1.0, float(view.get("y", 0.5))))
    scale = base_scale * zoom
    new_w = int(round(img_w * scale)); new_h = int(round(img_h * scale))
    img_resized = img.resize((new_w, new_h), Image.Resampling.LANCZOS)
    if fit == "contain":
        tile = Image.new("RGB", (panel_w, panel_h), background)
        left = int(round((panel_w - new_w) * x_frac))
        top = int(round((panel_h - new_h) * y_frac))
        tile.paste(img_resized, (left, top))
    else:
        extra_w = max(0, new_w - panel_w); extra_h = max(0, new_h - panel_h)
        left = int(round(x_frac * extra_w)); top = int(round(y_frac * extra_h))
        tile = img_resized.crop((left, top, left + panel_w, top + panel_h))

    _bounded_cache_put(FIGURE_TILE_CACHE, key, tile.copy(), FIGURE_TILE_CACHE_MAX_ITEMS)
    return tile




def compute_panel_scale(img_path, panel_w, panel_h, view=None, crop_bottom_px=0, rotate=None, padding_px=0, padding_color="white"):
    rotate = resolve_panel_rotation(view, rotate)
    key = (
        _image_file_signature(img_path), int(panel_w), int(panel_h),
        _freeze_cache_value(view or {}), int(crop_bottom_px), rotate,
        _freeze_cache_value(padding_px), str(padding_color),
    )
    cached = FIGURE_SCALE_CACHE.get(key)
    if cached is not None:
        FIGURE_SCALE_CACHE.move_to_end(key)
        return cached

    img = load_image_with_adjustments(img_path, crop_bottom_px=crop_bottom_px, rotate=rotate, padding_px=padding_px, padding_color=padding_color)
    img_w, img_h = img.size
    base_scale = max(panel_w / img_w, panel_h / img_h)
    view = view or {}
    scale = base_scale * float(view.get("zoom", 1.0))
    _bounded_cache_put(FIGURE_SCALE_CACHE, key, scale, FIGURE_SCALE_CACHE_MAX_ITEMS)
    return scale


def draw_scale_bar(draw, x0, y0, panel_w, panel_h, bar_len_px, text,
                   color="white", thickness=5, font=None, margin_px=0):
    x_start = x0 + panel_w - margin_px - bar_len_px
    x_end = x_start + bar_len_px
    y_bar = y0 + margin_px
    draw.line([(x_start, y_bar), (x_end, y_bar)], fill=color, width=thickness)
    if text and font is not None:
        text_w, text_h = text_size(draw, text, font)
        text_x = x_start + (bar_len_px - text_w) // 2
        text_y = y_bar + thickness + 4
        draw.text((text_x, text_y), text, fill=color, font=font)

def block_arrow(x, y, angle=0, length=0.18, color="white",
                shaft_width=None, head_length=0.45, head_width=None,
                anchor="tip"):
    """Return a filled block-arrow annotation; x/y are the tip by default."""
    annotation = {
        "type": "block_arrow", "x": x, "y": y, "angle": angle,
        "length": length, "color": color, "anchor": anchor,
        "head_length": head_length,
    }
    if shaft_width is not None:
        annotation["shaft_width"] = shaft_width
    if head_width is not None:
        annotation["head_width"] = head_width
    return annotation


def draw_block_arrow(draw, tip_x, tip_y, length, angle=0, color="white",
                     shaft_width=None, head_length=0.45, head_width=None):
    """Draw one solid rectangular-shaft/triangular-head arrow."""
    length = max(1.0, float(length))
    shaft_width = max(1.0, float(shaft_width if shaft_width is not None else length * 0.25))
    head_length = float(head_length)
    if 0 < head_length <= 1:
        head_length *= length
    head_length = min(length, max(1.0, head_length))
    head_width = max(shaft_width, float(head_width if head_width is not None else length * 0.70))

    theta = math.radians(float(angle))
    ux, uy = math.cos(theta), math.sin(theta)
    vx, vy = -uy, ux
    neck_x, neck_y = tip_x - ux * head_length, tip_y - uy * head_length
    tail_x, tail_y = tip_x - ux * length, tip_y - uy * length
    half_shaft = shaft_width / 2.0
    half_head = head_width / 2.0
    points = [
        (tail_x + vx * half_shaft, tail_y + vy * half_shaft),
        (neck_x + vx * half_shaft, neck_y + vy * half_shaft),
        (neck_x + vx * half_head, neck_y + vy * half_head),
        (tip_x, tip_y),
        (neck_x - vx * half_head, neck_y - vy * half_head),
        (neck_x - vx * half_shaft, neck_y - vy * half_shaft),
        (tail_x - vx * half_shaft, tail_y - vy * half_shaft),
    ]
    draw.polygon([(round(x), round(y)) for x, y in points], fill=color)


def draw_annotations_for_panel(panel_cfg, draw, x0, y0, pw, ph, base_font, canvas=None):
    used_rgba_canvas = False
    for t in panel_cfg.get("annotations", []):
        kind = str(t.get("kind", t.get("type", "text"))).lower()
        tx = x0 + int(t.get("x", 0.5) * pw)
        ty = y0 + int(t.get("y", 0.5) * ph)
        color = t.get("color", "white")

        if kind in {"block_arrow", "solid_arrow"}:
            angle_degrees = float(t.get("angle", t.get("rotation", t.get("rotate", 0))))
            length = max(1, int(float(t.get("length", 0.18)) * min(pw, ph)))
            theta = math.radians(angle_degrees)
            anchor = str(t.get("anchor", "tip")).lower()
            tip_x, tip_y = tx, ty
            if anchor in {"center", "centre", "middle"}:
                tip_x += math.cos(theta) * length / 2.0
                tip_y += math.sin(theta) * length / 2.0
            elif anchor in {"tail", "start"}:
                tip_x += math.cos(theta) * length
                tip_y += math.sin(theta) * length
            draw_block_arrow(
                draw, tip_x, tip_y, length, angle=angle_degrees, color=color,
                shaft_width=t.get("shaft_width", t.get("width")),
                head_length=t.get("head_length", 0.45),
                head_width=t.get("head_width"),
            )
            continue

        if kind == "arrow":
            angle = math.radians(float(t.get("angle", t.get("rotation", t.get("rotate", 0)))))
            length = int(t.get("length", 0.18) * min(pw, ph))
            width = int(t.get("width", max(4, length * 0.08)))
            head_size = int(t.get("head_size", max(width * 3, length * 0.22)))
            start = bool(t.get("start", False))

            if start:
                x_start, y_start = tx, ty
                x_end = tx + int(math.cos(angle) * length)
                y_end = ty + int(math.sin(angle) * length)
            else:
                x_start = tx - int(math.cos(angle) * length / 2)
                y_start = ty - int(math.sin(angle) * length / 2)
                x_end = tx + int(math.cos(angle) * length / 2)
                y_end = ty + int(math.sin(angle) * length / 2)

            draw.line([x_start, y_start, x_end, y_end], fill=color, width=width)
            head_angle = float(t.get("head_angle", 165))
            left_angle = angle + math.radians(head_angle)
            right_angle = angle - math.radians(head_angle)
            p1 = (x_end, y_end)
            p2 = (x_end + int(math.cos(left_angle) * head_size), y_end + int(math.sin(left_angle) * head_size))
            p3 = (x_end + int(math.cos(right_angle) * head_size), y_end + int(math.sin(right_angle) * head_size))
            draw.polygon([p1, p2, p3], fill=color)
            continue

        text = t.get("text", "")
        if not text:
            continue
        size = int(t.get("size", 40))
        rotation = normalize_rotation(t.get("rotation", t.get("rotate", 0)))
        try:
            if isinstance(base_font, ImageFont.FreeTypeFont) and hasattr(base_font, "path"):
                font = ImageFont.truetype(base_font.path, size)
            else:
                font = base_font
        except Exception:
            font = base_font
        if rotation and canvas is not None:
            try:
                bbox = font.getbbox(text)
                text_w = bbox[2] - bbox[0]
                text_h = bbox[3] - bbox[1]
                offset_x = -bbox[0]
                offset_y = -bbox[1]
            except Exception:
                text_w, text_h = text_size(draw, text, font)
                offset_x = offset_y = 0
            pad = max(4, int(size * 0.35))
            txt_img = Image.new("RGBA", (text_w + 2 * pad, text_h + 2 * pad), (0, 0, 0, 0))
            ImageDraw.Draw(txt_img).text((pad + offset_x, pad + offset_y), text, fill=color, font=font)
            rotated = txt_img.rotate(rotation, expand=1, resample=Image.Resampling.BICUBIC)
            if canvas.mode != "RGBA":
                canvas = canvas.convert("RGBA")
                used_rgba_canvas = True
            paste_x = int(tx + txt_img.size[0] / 2 - rotated.size[0] / 2)
            paste_y = int(ty + txt_img.size[1] / 2 - rotated.size[1] / 2)
            canvas.paste(rotated, (paste_x, paste_y), rotated)
            draw = ImageDraw.Draw(canvas)
        else:
            anchor = str(t.get("anchor", t.get("align", "left"))).lower()
            if anchor in {"right", "r"}:
                text_w, _text_h = text_size(draw, text, font)
                tx -= text_w
            elif anchor in {"center", "centre", "middle", "c"}:
                text_w, _text_h = text_size(draw, text, font)
                tx -= text_w // 2
            draw.text((tx, ty), text, fill=color, font=font)
    return canvas.convert("RGB") if used_rgba_canvas else canvas


def build_mixed_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px, row_panel_orders, row_height_ratios=None, row_width_fracs=None):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    n_rows = len(row_panel_orders)
    usable_h = inner_h - row_gap_px * (n_rows - 1)
    if row_height_ratios is None:
        row_heights = [usable_h // n_rows] * n_rows
    else:
        ratio_total = sum(row_height_ratios)
        row_heights = [int(round(usable_h * r / ratio_total)) for r in row_height_ratios]
        row_heights[-1] += usable_h - sum(row_heights)
    if row_width_fracs is None:
        row_width_fracs = [1.0] * n_rows

    layout = {}
    y0 = mt
    for row_idx, row_panels in enumerate(row_panel_orders):
        n_cols = len(row_panels)
        row_w = int(round(inner_w * row_width_fracs[row_idx]))
        row_x0 = ml + (inner_w - row_w) // 2
        tile_w = (row_w - col_gap_px * (n_cols - 1)) // n_cols
        tile_h = row_heights[row_idx]
        x0 = row_x0
        for pid in row_panels:
            layout[pid] = (x0, y0, tile_w, tile_h)
            x0 += tile_w + col_gap_px
        y0 += tile_h + row_gap_px
    return layout


def render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels,
    dpi=DPI, label_font_path="arial.ttf", scale_font_path="arial.ttf",
    label_font_size=96, scale_font_size=75, label_pad_in=LABEL_PAD_IN,
    col_labels=None, row_labels=None, axis_font_size=70, axis_font_path="arial.ttf",
):
    canvas = Image.new("RGB", (fig_w_px, fig_h_px), "white")
    draw = ImageDraw.Draw(canvas)
    label_font = load_font_safe(label_font_path, label_font_size)
    scale_font = load_font_safe(scale_font_path, scale_font_size)
    axis_font = load_font_safe(axis_font_path, axis_font_size)
    label_pad_px = inches_to_px(label_pad_in, dpi)

    for panel_id, (x0, y0, pw, ph) in panel_layout.items():
        cfg = panels.get(panel_id, {})
        path = cfg.get("path")
        view = cfg.get("view", {"zoom": 1.0, "x": 0.5, "y": 0.5})
        crop_px = cfg.get("crop_bottom_px", 0)
        padding_px = cfg.get("padding_px", 0)
        padding_color = cfg.get("padding_color", "white")
        fit = cfg.get("fit", "cover")
        background = cfg.get("background", "white")
        rotate = resolve_panel_rotation(view, cfg.get("rotate"))

        if path:
            img = paste_with_view(path, pw, ph, view=view, crop_bottom_px=crop_px, rotate=rotate, padding_px=padding_px, padding_color=padding_color, fit=fit, background=background)
        else:
            img = Image.new("RGB", (pw, ph), "#dddddd")
        canvas.paste(img, (x0, y0))

        border = cfg.get("border")
        if border:
            bc = border.get("color", "black"); th = int(border.get("thickness", 4))
            for i in range(th):
                draw.rectangle([x0 - i, y0 - i, x0 + pw + i, y0 + ph + i], outline=bc)

        sb = cfg.get("scale_bar")
        if sb and path:
            full_px = float(sb.get("full_px", 0))
            orig_um = float(sb.get("orig_um", sb.get("new_um", 1)))
            new_um = float(sb.get("new_um", orig_um))
            color = sb.get("color", "white"); sb_th = int(sb.get("thickness", 5))
            show_number = bool(sb.get("number", True))
            scale_factor = compute_panel_scale(path, pw, ph, view=view, crop_bottom_px=crop_px, rotate=rotate, padding_px=padding_px, padding_color=padding_color)
            orig_um = orig_um if orig_um > 0 else 1.0
            px_per_um_panel = (full_px * scale_factor) / orig_um
            bar_len_panel_px = int(round(px_per_um_panel * new_um))
            bar_len_panel_px = max(10, min(bar_len_panel_px, int(pw * 0.9)))
            bar_text = f"{new_um:g} um" if show_number else None
            draw_scale_bar(draw, x0, y0, pw, ph, bar_len_panel_px, bar_text,
                           color=color, thickness=sb_th, font=scale_font,
                           margin_px=inches_to_px(0.08, dpi))

        canvas = draw_annotations_for_panel(cfg, draw, x0, y0, pw, ph, label_font, canvas=canvas) or canvas
        draw = ImageDraw.Draw(canvas)
        label_color = cfg.get("label_color", "black")
        label_text = cfg.get("label", panel_id)
        panel_label_font_size = cfg.get("label_font_size")
        panel_label_font = label_font if panel_label_font_size is None else load_font_safe(label_font_path, int(panel_label_font_size))
        panel_label_pad_px = int(cfg.get("label_pad_px", label_pad_px))
        panel_label_pad_x_px = int(cfg.get("label_pad_x_px", panel_label_pad_px))
        panel_label_pad_y_px = int(cfg.get("label_pad_y_px", panel_label_pad_px))
        draw.text(
            (x0 + panel_label_pad_x_px, y0 + panel_label_pad_y_px),
            label_text,
            fill=label_color,
            font=panel_label_font,
        )

    # Axis label offsets (match row/col distances)
    col_margin_px = inches_to_px(0.30, dpi)
    row_margin_px = col_margin_px

    if col_labels:
        # Find the row with the most panels and use its column centers
        rows = {}
        for pid, (x0, y0, pw, ph) in panel_layout.items():
            rows.setdefault(y0, []).append((x0, pw))
        max_row = max(rows.values(), key=lambda r: len(r))
        col_centers = [x0 + pw // 2 for x0, pw in sorted(max_row, key=lambda t: t[0])]

        y_top = mt - col_margin_px
        for i, cx in enumerate(col_centers):
            key = f"col{i}"
            if key not in col_labels:
                continue
            w, h = text_size(draw, col_labels[key], axis_font)
            draw.text((cx - w/2, y_top - h/2), col_labels[key], fill="black", font=axis_font)

    if row_labels:
        # Use sorted unique row y positions for row centers. Keys can be "row1" or spans like "row1-row2".
        rows = {}
        for pid, (x0, y0, pw, ph) in panel_layout.items():
            rows.setdefault(y0, ph)
        y_sorted = sorted(rows.keys())

        def row_label_bounds(key):
            key = str(key).strip().lower().replace(" ", "")
            if "-" in key:
                left, right = key.split("-", 1)
            else:
                left = right = key
            if not left.startswith("row") or not right.startswith("row"):
                return None
            try:
                start_idx = int(left[3:])
                end_idx = int(right[3:])
            except Exception:
                return None
            if start_idx > end_idx:
                start_idx, end_idx = end_idx, start_idx
            if start_idx < 0 or end_idx >= len(y_sorted):
                return None
            top = y_sorted[start_idx]
            bottom = y_sorted[end_idx] + rows[y_sorted[end_idx]]
            return top, bottom

        x_left = ml - row_margin_px
        pad = int(axis_font_size * 0.8)
        canvas_rgba = canvas.convert("RGBA")
        for key, text in row_labels.items():
            bounds = row_label_bounds(key)
            if bounds is None or not text:
                continue
            top, bottom = bounds
            ty = (top + bottom) / 2

            w, h = text_size(ImageDraw.Draw(canvas_rgba), text, axis_font)
            txt_img = Image.new("RGBA", (w + 2*pad, h + 2*pad), (0, 0, 0, 0))
            ImageDraw.Draw(txt_img).text((pad, pad), text, font=axis_font, fill="black")
            rot = txt_img.rotate(90, expand=1)
            rx, ry = rot.size
            paste_x = int(x_left - rx / 2)
            paste_y = int(ty - ry / 2)
            canvas_rgba.paste(rot, (paste_x, paste_y), rot)
        canvas = canvas_rgba.convert("RGB")

    return canvas

# Finished canvases are retained by section title for the final batch export.
FIGURE_CANVASES = globals().get("FIGURE_CANVASES", OrderedDict())
FIGURE_CANVAS_TRIM_OPTIONS = globals().get("FIGURE_CANVAS_TRIM_OPTIONS", {})
FIGURE_SCALE_BARS = globals().get("FIGURE_SCALE_BARS", OrderedDict())


def trim_canvas_to_artboard(
    canvas, background="white", left_grace_in=0.0, dpi=DPI, preserve_artboard=False,
):
    """Crop outer background unless the full artboard should be retained."""
    rgb = canvas.convert("RGB")
    if preserve_artboard:
        return rgb
    bg = Image.new("RGB", rgb.size, background)
    bbox = ImageChops.difference(rgb, bg).getbbox()
    if not bbox:
        return rgb
    cropped = rgb.crop(bbox)
    left_grace_px = max(0, inches_to_px(left_grace_in, dpi))
    if left_grace_px == 0:
        return cropped
    padded = Image.new("RGB", (cropped.width + left_grace_px, cropped.height), background)
    padded.paste(cropped, (left_grace_px, 0))
    return padded


def register_figure_canvas(
    section_title, canvas, left_grace_in=0.0, panel_configs=None, panel_layout=None,
    preserve_artboard=False,
):
    """Retain a final canvas and scale bars for panels in its final layout."""
    section_title = str(section_title).strip()
    trim_options = {
        "left_grace_in": float(left_grace_in),
        "preserve_artboard": bool(preserve_artboard),
    }
    trimmed = trim_canvas_to_artboard(canvas, **trim_options)
    FIGURE_CANVASES[section_title] = trimmed
    FIGURE_CANVAS_TRIM_OPTIONS[section_title] = trim_options

    panel_configs = panel_configs or {}
    displayed_panel_ids = panel_layout.keys() if panel_layout is not None else panel_configs.keys()
    scale_bars = OrderedDict()
    for config_panel_id in displayed_panel_ids:
        config = panel_configs.get(config_panel_id)
        if not isinstance(config, dict):
            continue
        scale_bar = config.get("scale_bar")
        if not isinstance(scale_bar, dict) or scale_bar.get("new_um") is None:
            continue
        display_panel_id = config.get("label", config_panel_id)
        display_panel_id = str(display_panel_id).strip() or str(config_panel_id)
        scale_bars[display_panel_id] = scale_bar["new_um"]
    FIGURE_SCALE_BARS[section_title] = scale_bars
    return trimmed


def _join_panel_ids(panel_ids):
    panel_ids = [str(panel_id) for panel_id in panel_ids]
    if len(panel_ids) == 1:
        return panel_ids[0]
    if len(panel_ids) == 2:
        return f"{panel_ids[0]} and {panel_ids[1]}"
    return f"{', '.join(panel_ids[:-1])}, and {panel_ids[-1]}"


def _collapse_equal_subpanel_scale_bars(panel_scale_bars):
    """Collapse A1/A2/... to A when all displayed subpanel values agree."""
    items = list(panel_scale_bars.items())
    collapsed = OrderedDict()
    handled = set()
    for panel_id, new_um in items:
        if panel_id in handled:
            continue
        match = re.fullmatch(r"([A-Za-z]+)(\d+)", str(panel_id))
        if not match:
            collapsed[panel_id] = new_um
            handled.add(panel_id)
            continue
        base_id = match.group(1)
        family = []
        for candidate_id, candidate_um in items:
            candidate_match = re.fullmatch(r"([A-Za-z]+)(\d+)", str(candidate_id))
            if candidate_match and candidate_match.group(1) == base_id:
                family.append((candidate_id, candidate_um))
        handled.update(candidate_id for candidate_id, _candidate_um in family)
        if len(family) > 1 and all(value == family[0][1] for _candidate_id, value in family):
            collapsed[base_id] = family[0][1]
        else:
            collapsed.update(family)
    return collapsed


def format_scale_bar_summary(panel_scale_bars):
    """Group displayed panels with equal scale-bar values into manuscript text."""
    panel_scale_bars = _collapse_equal_subpanel_scale_bars(panel_scale_bars)
    grouped = OrderedDict()
    for panel_id, new_um in panel_scale_bars.items():
        grouped.setdefault(new_um, []).append(panel_id)
    if not grouped:
        return "Scale bars: none."
    entries = []
    for new_um, panel_ids in grouped.items():
        value_text = f"{new_um:g}" if isinstance(new_um, (int, float)) else str(new_um)
        entries.append(f"{_join_panel_ids(panel_ids)}: {value_text} µm")
    return "Scale bars: " + "; ".join(entries) + "."


def print_all_scale_bar_summaries():
    """Print scale-bar summaries in registered figure-section order."""
    if not FIGURE_CANVASES:
        raise RuntimeError("No figures are registered. Run the figure cells first.")
    for index, section_title in enumerate(FIGURE_CANVASES):
        if index:
            print()
        print(section_title)
        print(format_scale_bar_summary(FIGURE_SCALE_BARS.get(section_title, {})))


def _safe_export_filename(section_title):
    name = re.sub(r'[<>:"/\\|?*]+', "_", str(section_title)).strip(" .")
    return re.sub(r"\s+", " ", name) or "figure"


def export_registered_canvases(
    output_dir=r"D:\larval chemosensory\Written\Round 5\Exports Figures",
    source_dpi=DPI,
    target_dpi=600,
):
    """Export all registered canvases at target DPI without changing geometry."""
    if not FIGURE_CANVASES:
        raise RuntimeError("No canvases are registered. Run the figure cells before Export.")
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)
    scale = float(target_dpi) / float(source_dpi)
    written = []
    for section_title, canvas in FIGURE_CANVASES.items():
        trim_options = FIGURE_CANVAS_TRIM_OPTIONS.get(section_title, {})
        final_canvas = trim_canvas_to_artboard(canvas, **trim_options)
        if scale != 1.0:
            final_canvas = final_canvas.resize(
                (int(round(final_canvas.width * scale)), int(round(final_canvas.height * scale))),
                Image.Resampling.LANCZOS,
            )
        export_path = output_path / f"{_safe_export_filename(section_title)}.png"
        final_canvas.save(export_path, format="PNG", dpi=(target_dpi, target_dpi))
        written.append(export_path)
    return written



In [ ]:
print("fig_w_px", fig_w_px, "ml", ml, "AXIS_PAD_LEFT_IN", AXIS_PAD_LEFT_IN)


In [ ]:
def build_mixed_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px, row_panel_orders, row_height_ratios=None, row_width_fracs=None):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    n_rows = len(row_panel_orders)
    usable_h = inner_h - row_gap_px * (n_rows - 1)
    if row_height_ratios is None:
        row_heights = [usable_h // n_rows] * n_rows
    else:
        ratio_total = sum(row_height_ratios)
        row_heights = [int(round(usable_h * r / ratio_total)) for r in row_height_ratios]
        row_heights[-1] += usable_h - sum(row_heights)
    if row_width_fracs is None:
        row_width_fracs = [1.0] * n_rows

    layout = {}
    y0 = mt
    for row_idx, row_panels in enumerate(row_panel_orders):
        n_cols = len(row_panels)
        row_w = int(round(inner_w * row_width_fracs[row_idx]))
        row_x0 = ml + (inner_w - row_w) // 2
        tile_w = (row_w - col_gap_px * (n_cols - 1)) // n_cols
        tile_h = row_heights[row_idx]
        x0 = row_x0
        for pid in row_panels:
            layout[pid] = (x0, y0, tile_w, tile_h)
            x0 += tile_w + col_gap_px
        y0 += tile_h + row_gap_px
    return layout


def build_figure3_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px):
    inner_w_px = fig_w_px - ml - mr
    inner_h_px = fig_h_px - mt - mb

    usable_h = inner_h_px - row_gap_px * 2
    row_height_ratios = (0.5, 0.3, 0.3)
    ratio_total = sum(row_height_ratios)
    row1_h = int(round(usable_h * row_height_ratios[0] / ratio_total))
    row2_h = int(round(usable_h * row_height_ratios[1] / ratio_total))
    row3_h = usable_h - row1_h - row2_h

    layout = {}

    y_row1 = mt
    row1_w = int(round(inner_w_px * 0.70))
    x_a = ml + (inner_w_px - row1_w) // 2
    layout['A'] = (x_a, y_row1, row1_w, row1_h)

    y_row2 = y_row1 + row1_h + row_gap_px
    usable_w_row2 = inner_w_px - 3 * col_gap_px
    unit_w = usable_w_row2 // 6
    b_w = unit_w
    c_w = unit_w
    d_w = 2 * unit_w
    e_w = usable_w_row2 - b_w - c_w - d_w

    x_b = ml
    x_c = x_b + b_w + col_gap_px
    x_d = x_c + c_w + col_gap_px
    x_e = x_d + d_w + col_gap_px

    layout['B'] = (x_b, y_row2, b_w, row2_h)
    layout['C'] = (x_c, y_row2, c_w, row2_h)
    layout['D'] = (x_d, y_row2, d_w, row2_h)
    layout['E'] = (x_e, y_row2, e_w, row2_h)

    y_row3 = y_row2 + row2_h + row_gap_px
    usable_w_row3 = inner_w_px - col_gap_px
    f_w = usable_w_row3 // 2
    g_w = usable_w_row3 - f_w

    x_f = ml
    x_g = x_f + f_w + col_gap_px

    layout['F'] = (x_f, y_row3, f_w, row3_h)
    layout['G'] = (x_g, y_row3, g_w, row3_h)

    return layout

def build_figure3_five_row_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb

    usable_h = inner_h - row_gap_px * 4
    row_h = usable_h // 5
    row_heights = [row_h] * 5
    row_heights[-1] += usable_h - sum(row_heights)

    left_w = (inner_w - col_gap_px) // 2
    right_w = inner_w - col_gap_px - left_w
    x_left = ml
    x_right = ml + left_w + col_gap_px

    layout = {}
    y = mt
    rows = [
        ("A", "B"),
        ("C", "D"),
        ("E", None),
        ("H", "I"),
        ("J", "K"),
    ]

    for row_idx, (left_id, right_id) in enumerate(rows):
        h = row_heights[row_idx]
        layout[left_id] = (x_left, y, left_w, h)

        if row_idx == 2:
            split_gap = row_gap_px
            top_h = (h - split_gap) // 2
            bottom_h = h - split_gap - top_h
            layout["F"] = (x_right, y, right_w, top_h)
            layout["G"] = (x_right, y + top_h + split_gap, right_w, bottom_h)
        else:
            layout[right_id] = (x_right, y, right_w, h)

        y += h + row_gap_px

    return layout

def build_figure3_test_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    usable_h = inner_h - row_gap_px * 3
    row_height_ratios = (0.28, 0.28, 0.28, 0.16)
    ratio_total = sum(row_height_ratios)
    row_heights = [int(round(usable_h * r / ratio_total)) for r in row_height_ratios]
    row_heights[-1] += usable_h - sum(row_heights)

    layout = {}

    def add_grouped_row(panel_groups, y, h):
        flat_panel_ids = [panel_id for group in panel_groups for panel_id in group]
        n_panels = len(flat_panel_ids)
        n_group_gaps = len(panel_groups) - 1
        panel_w = (inner_w - col_gap_px * n_group_gaps) // n_panels
        extra_w = inner_w - col_gap_px * n_group_gaps - panel_w * n_panels
        x = ml
        panel_idx = 0
        for group_idx, group in enumerate(panel_groups):
            for panel_id in group:
                w = panel_w + (1 if panel_idx < extra_w else 0)
                layout[panel_id] = (x, y, w, h)
                x += w
                panel_idx += 1
            if group_idx < len(panel_groups) - 1:
                x += col_gap_px

    y = mt
    add_grouped_row((("A", "B", "C"), ("D", "E")), y, row_heights[0])

    y += row_heights[0] + row_gap_px
    add_grouped_row((("F",), ("G", "H", "I", "J")), y, row_heights[1])

    y += row_heights[1] + row_gap_px
    add_grouped_row((("K", "L"), ("M", "N", "O")), y, row_heights[2])

    y += row_heights[2] + row_gap_px
    add_grouped_row((("P", "Q"),), y, row_heights[3])

    return layout

def apply_panel_tuning(panels, panel_tuning):
    for panel_id, overrides in panel_tuning.items():
        cfg = panels[panel_id]
        overrides = dict(overrides)

        if "view" in overrides:
            view = dict(cfg.get("view", {}))
            view.update(overrides.pop("view"))
            cfg["view"] = view

        if "scale_bar" in overrides:
            scale_bar = dict(cfg.get("scale_bar", {}))
            scale_bar.update(overrides.pop("scale_bar"))
            cfg["scale_bar"] = scale_bar

        if "add_annotations" in overrides:
            cfg.setdefault("annotations", []).extend(overrides.pop("add_annotations"))

        cfg.update(overrides)

    return panels


def add_row_side_labels(canvas, layout, row_labels, font_size=60, color="black", pad_in=0.04, font_path="arial.ttf", dpi=DPI, rotation=0):
    font = load_font_safe(font_path, font_size)
    pad_px = inches_to_px(pad_in, dpi)

    if canvas.mode != "RGBA":
        canvas = canvas.convert("RGBA")

    for row_label, panel_ids in row_labels:
        x0 = min(layout[panel_id][0] for panel_id in panel_ids)
        y0 = min(layout[panel_id][1] for panel_id in panel_ids)
        row_bottom = max(layout[panel_id][1] + layout[panel_id][3] for panel_id in panel_ids)
        row_h = row_bottom - y0

        measure_draw = ImageDraw.Draw(canvas)
        text_w, text_h = text_size(measure_draw, row_label, font)
        pad = max(4, int(font_size * 0.35))
        text_img = Image.new("RGBA", (text_w + 2 * pad, text_h + 2 * pad), (0, 0, 0, 0))
        ImageDraw.Draw(text_img).text((pad, pad), row_label, fill=color, font=font)
        text_img = text_img.rotate(rotation, expand=True, resample=Image.Resampling.BICUBIC)

        paste_x = x0 - text_img.size[0] - pad_px
        paste_y = y0 + (row_h - text_img.size[1]) // 2
        canvas.paste(text_img, (int(paste_x), int(paste_y)), text_img)

    return canvas.convert("RGB")


def resolve_row_label_panel_ids(row_panel_ids, row_labels):
    resolved = []

    for row_key, label in row_labels.items():
        if not label:
            continue

        keys = []
        if row_key in row_panel_ids:
            keys = [row_key]
        elif "-" in row_key:
            start_key, end_key = row_key.split("-", 1)
            if start_key.startswith("row") and end_key.startswith("row"):
                try:
                    start_idx = int(start_key[3:])
                    end_idx = int(end_key[3:])
                except ValueError:
                    keys = []
                else:
                    step = 1 if end_idx >= start_idx else -1
                    keys = [f"row{i}" for i in range(start_idx, end_idx + step, step)]

        panel_ids = tuple(
            panel_id
            for key in keys
            for panel_id in row_panel_ids.get(key, ())
        )
        if panel_ids:
            resolved.append((label, panel_ids))

    return resolved


def row_keys_from_span(row_panel_ids, span_key):
    if span_key in row_panel_ids:
        return [span_key]
    if "-" not in span_key:
        return []

    start_key, end_key = span_key.split("-", 1)
    if not (start_key.startswith("row") and end_key.startswith("row")):
        return []

    try:
        start_idx = int(start_key[3:])
        end_idx = int(end_key[3:])
    except ValueError:
        return []

    step = 1 if end_idx >= start_idx else -1
    return [
        key
        for key in (f"row{i}" for i in range(start_idx, end_idx + step, step))
        if key in row_panel_ids
    ]


def add_row_span_bracket(canvas, layout, row_panel_ids, span_key, color="black",
                         line_width=6, x_offset_in=0.18, tick_length_in=0.12,
                         dpi=DPI):
    keys = row_keys_from_span(row_panel_ids, span_key)
    if not keys:
        return canvas

    if canvas.mode != "RGBA":
        canvas = canvas.convert("RGBA")

    draw = ImageDraw.Draw(canvas)
    x0 = min(
        layout[panel_id][0]
        for key in keys
        for panel_id in row_panel_ids[key]
    )
    x_line = int(x0 - inches_to_px(x_offset_in, dpi))
    tick_len = int(inches_to_px(tick_length_in, dpi))

    row_centers = []
    for key in keys:
        panel_ids = row_panel_ids[key]
        y0 = min(layout[panel_id][1] for panel_id in panel_ids)
        y1 = max(layout[panel_id][1] + layout[panel_id][3] for panel_id in panel_ids)
        row_centers.append(int((y0 + y1) / 2))

    draw.line((x_line, row_centers[0], x_line, row_centers[-1]), fill=color, width=line_width)
    for y in row_centers:
        draw.line((x_line, y, x_line + tick_len, y), fill=color, width=line_width)

    return canvas.convert("RGB")


def build_centered_group_rows_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px, row_panel_groups, row_height_ratios=None):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    n_rows = len(row_panel_groups)
    usable_h = inner_h - row_gap_px * (n_rows - 1)

    if row_height_ratios is None:
        row_heights = [usable_h // n_rows] * n_rows
    else:
        ratio_total = sum(row_height_ratios)
        row_heights = [int(round(usable_h * r / ratio_total)) for r in row_height_ratios]
    row_heights[-1] += usable_h - sum(row_heights)

    max_panels = max(len(panel_ids) for panel_ids, _border_color in row_panel_groups)
    panel_w = inner_w // max_panels

    layout = {}
    y = mt
    for row_idx, (panel_ids, _border_color) in enumerate(row_panel_groups):
        h = row_heights[row_idx]
        row_w = panel_w * len(panel_ids)
        x = ml + (inner_w - row_w) // 2
        for panel_id in panel_ids:
            layout[panel_id] = (x, y, panel_w, h)
            x += panel_w
        y += h + row_gap_px

    return layout


def build_figure35_separate_channels_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb

    row_h = (inner_h - row_gap_px * 4) // 5
    row_heights = [row_h] * 5
    row_heights[-1] += inner_h - row_gap_px * 4 - sum(row_heights)

    left_w = (inner_w - col_gap_px) // 2
    right_w = inner_w - col_gap_px - left_w
    x_left = ml
    x_right = ml + left_w + col_gap_px

    panel_ids = list("ABCDEFGHIJKLMNOPQRSTUVWXY")
    layout = {}
    y = mt

    for row_idx in range(5):
        h = row_heights[row_idx]
        row_ids = panel_ids[row_idx * 5:(row_idx + 1) * 5]
        left_id, top_left_id, top_right_id, bottom_left_id, bottom_right_id = row_ids

        layout[left_id] = (x_left, y, left_w, h)

        small_w = (right_w - col_gap_px) // 2
        small_w_last = right_w - col_gap_px - small_w
        small_h = (h - row_gap_px) // 2
        small_h_last = h - row_gap_px - small_h

        layout[top_left_id] = (x_right, y, small_w, small_h)
        layout[top_right_id] = (x_right + small_w + col_gap_px, y, small_w_last, small_h)
        layout[bottom_left_id] = (x_right, y + small_h + row_gap_px, small_w, small_h_last)
        layout[bottom_right_id] = (x_right + small_w + col_gap_px, y + small_h + row_gap_px, small_w_last, small_h_last)

        y += h + row_gap_px

    return layout


# Figure 1


## Figure 1: 4th SEM Larval Chemosensory Organs


In [ ]:
thickness_border = 12
scale_bar_thickness = 12

dome_color = "black"
papilla_color = "black"
rod_color = "black"

panels = {
    "A": {
        "path": r"C:\Users\hejaz\Downloads\L4 1_0008 (1).tif",
        "view": {"zoom": 1.0, "x": 0.6, "y": 0.6},
        "crop_bottom_px": 270,
        "label_color": "white",
        "scale_bar": {"full_px": 781.37, "orig_um": 50, "new_um": 80, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
    },
    "B": {
        "path": r"D:\larval chemosensory\cartoonfig1.png",
        "view": {"zoom": 1.05, "x": 0.56, "y": 1.2},
        "fit": "contain",
        "background": "white",
        "label_color": "black",
    },
    "C": {
        "path": r"C:\Users\hejaz\Downloads\22-03_Daniel_L4-2_0010.tif",
        "view": {"zoom": 1.4, "x": 0.18, "y": 0.5},
        "label_color": "white",
        "scale_bar": {"full_px": 1002, "orig_um": 5, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "Papilla", "x": 0.10, "y": 0.80, "color": "white", "size": 70},
        ],
    },
    "D": {
        "path": r"C:\Users\hejaz\Downloads\22-03_Daniel_L4-1_0004.tif",
        "view": {"zoom": 1.4, "x": 0.1, "y": 0.0},
        "label_color": "white",
        "scale_bar": {"full_px": 620, "orig_um": 2, "new_um": 4, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "Rod", "x": 0.10, "y": 0.80, "color": "white", "size": 70},
        ],
    },
    "E": {
        "path": r"C:\Users\hejaz\Downloads\22-03_Daniel_L3-1_0008.tif",
        "view": {"zoom": 4.0, "x": 0.49, "y": 0.4},
        "label_color": "white",
        "scale_bar": {"full_px": 860, "orig_um": 5, "new_um": 2, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "Dome", "x": 0.10, "y": 0.80, "color": "white", "size": 70},
        ],
    },
    "F": {
        "path": r"C:\Users\hejaz\Downloads\22-03_Daniel_L3-1_0011.tif",
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.1},
        "crop_bottom_px": 260,
        "label_color": "white",
        "border": {"color": FRONS_BORDER_COLOR, "thickness": thickness_border},
        "scale_bar": {"full_px": 548, "orig_um": 2, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        # "annotations": [
        #     {"text": "▼", "x": 0.23, "y": 0.26, "color": papilla_color, "size": 70},
        #     {"text": "▼", "x": 0.66, "y": 0.38, "color": papilla_color, "size": 70},
        #     {"text": "▼", "x": 0.57, "y": 0.52, "color": papilla_color, "size": 70},
        # ],
    },
    "G": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\22-03_Daniel_L4-1_0009.tif",
        "view": {"zoom": 1.2, "x": 0.27, "y": 1.0},
        "crop_bottom_px": 250,
        "label_color": "white",
        "border": {"color": LABRUM_BORDER_COLOR, "thickness": thickness_border},
        "scale_bar": {"full_px": 756, "orig_um": 10, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "*", "x": 0.09, "y": 0.30, "color": papilla_color, "size": 135},
            {"text": "*", "x": 0.18, "y": 0.30, "color": papilla_color, "size": 135},
            {"text": "*", "x": 0.01, "y": 0.29, "color": papilla_color, "size": 135},
            {"text": "*", "x": 0.48, "y": 0.33, "color": papilla_color, "size": 135},
        ],
    },
    "H": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\22-03_Daniel_L3_0009.tif",
        "view": {"zoom": 1.0, "x": 0.1, "y": 0.40},
        "crop_bottom_px": 190,
        "label_color": "white",
        "border": {"color": MAXILLA_BORDER_COLOR, "thickness": thickness_border},
        "scale_bar": {"full_px": 576, "orig_um": 2, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        # "annotations": [
        #     {"text": "▼", "x": 0.18, "y": 0.22, "color": papilla_color, "size": 70},
        #     {"text": "▼", "x": 0.47, "y": 0.41, "color": papilla_color, "size": 70},
        #     {"text": "▼", "x": 0.55, "y": 0.16, "color": dome_color, "size": 70},
        #     {"text": "▼", "x": 0.35, "y": 0.60, "color": dome_color, "size": 70},
        #     {"text": "▼", "x": 0.26, "y": -0.05, "color": rod_color, "size": 70, "rotation": 90},
        # ],
    },
    "I": {
        "path": r"C:\Users\hejaz\Downloads\22-03_Daniel_L4-2_0006.tif",
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5},
        "crop_bottom_px": 255,
        "label_color": "black",
        "border": {"color": MAXILLA_BORDER_COLOR, "thickness": thickness_border},
        "scale_bar": {"full_px": 712, "orig_um": 10, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "*", "x": 0.67, "y": 0.685, "color": papilla_color, "size": 135},
        ],
    },
    "J": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\22-03_Daniel_L4-2_0005.tif",
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5},
        "crop_bottom_px": 255,
        "label_color": "white",
        "scale_bar": {"full_px": 474, "orig_um": 2, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "*", "x": 0.50, "y": 0.5, "color": papilla_color, "size": 135},
            {"text": "1", "x": 0.37, "y": 0.6, "color": dome_color, "size": 75},
            {"text": "2", "x": 0.4, "y": 0.3, "color": dome_color, "size": 75},
            {"text": "▼", "x": 0.65, "y": 0.14, "color": rod_color, "size": 70},
        ],
    },
    "K": {
        "path": r"C:\Users\hejaz\Downloads\22-03_Daniel_L4-2_0003 (1).tif",
        "view": {"zoom": 3.5, "x": 0.75, "y": 0.85},
        "crop_bottom_px": 180,
        "label_color": "white",
        "scale_bar": {"full_px": 520, "orig_um": 10, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "*", "x": 0.40, "y": 0.40, "color": papilla_color, "size": 135},
        ],
    },
}

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = inches_to_px(COL_GAP_IN, DPI)

row_panel_orders = [
    ("A", "B"),
    ("C", "D", "E"),
    ("F", "G", "H"),
    ("I", "J", "K"),
]

figure1_row_labels = {
    # row0 = A/B, row1 = C/D/E, row2 = F/G/H, row3 = I/J/K
    "row1": "Sensilla Examples",
    "row2-row3": "Sensilla in Different Head Regions",
}

panel_layout = build_mixed_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px,
    row_panel_orders,
    row_height_ratios=[1.4, 1.0, 1.0, 1.0],
)

def tighten_figure1_panel_layout(layout, panels, row_panel_orders):
    adjusted = dict(layout)

    # Close horizontal gaps only when both neighboring panels are borderless.
    for row_ids in row_panel_orders:
        for left_id, right_id in zip(row_ids, row_ids[1:]):
            if panels[left_id].get("border") or panels[right_id].get("border"):
                continue
            lx, ly, lw, lh = adjusted[left_id]
            rx, ry, rw, rh = adjusted[right_id]
            gap = rx - (lx + lw)
            if gap <= 0:
                continue
            left_extra = gap // 2
            right_extra = gap - left_extra
            adjusted[left_id] = (lx, ly, lw + left_extra, lh)
            adjusted[right_id] = (rx - right_extra, ry, rw + right_extra, rh)

    # Close vertical gaps by expanding the rows equally toward each other.
    for top_row, bottom_row in zip(row_panel_orders, row_panel_orders[1:]):
        top_bottom = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in top_row)
        bottom_top = min(adjusted[panel_id][1] for panel_id in bottom_row)
        gap = bottom_top - top_bottom
        if gap <= 0:
            continue
        top_extra = gap // 2
        bottom_extra = gap - top_extra
        for panel_id in top_row:
            x, y, w, h = adjusted[panel_id]
            adjusted[panel_id] = (x, y, w, h + top_extra)
        for panel_id in bottom_row:
            x, y, w, h = adjusted[panel_id]
            adjusted[panel_id] = (x, y - bottom_extra, w, h + bottom_extra)

    # Keep the three-panel rows sharing the same panel edges.
    three_panel_rows = [row_ids for row_ids in row_panel_orders if len(row_ids) == 3]
    if three_panel_rows:
        target_left = min(adjusted[row_ids[0]][0] for row_ids in three_panel_rows)
        target_right = max(adjusted[row_ids[-1]][0] + adjusted[row_ids[-1]][2] for row_ids in three_panel_rows)
        for row_ids in three_panel_rows:
            left_id = row_ids[0]
            right_id = row_ids[-1]
            x, y, w, h = adjusted[left_id]
            left_delta = x - target_left
            if left_delta > 0:
                adjusted[left_id] = (target_left, y, w + left_delta, h)
            x, y, w, h = adjusted[right_id]
            right_delta = target_right - (x + w)
            if right_delta > 0:
                adjusted[right_id] = (x, y, w + right_delta, h)

    # Match the first two rows to the visible outer edge of the bordered rows.
    bordered_rows = [
        row_ids for row_ids in row_panel_orders
        if any(panels[panel_id].get("border") for panel_id in row_ids)
    ]
    if bordered_rows:
        visible_left = min(
            adjusted[panel_id][0] - int(panels[panel_id].get("border", {}).get("thickness", 0))
            for row_ids in bordered_rows
            for panel_id in row_ids
        )
        visible_right = max(
            adjusted[panel_id][0] + adjusted[panel_id][2] + int(panels[panel_id].get("border", {}).get("thickness", 0))
            for row_ids in bordered_rows
            for panel_id in row_ids
        )
        for row_ids in row_panel_orders[:2]:
            left_id = row_ids[0]
            right_id = row_ids[-1]
            x, y, w, h = adjusted[left_id]
            left_delta = x - visible_left
            if left_delta > 0:
                adjusted[left_id] = (visible_left, y, w + left_delta, h)
            x, y, w, h = adjusted[right_id]
            right_delta = visible_right - (x + w)
            if right_delta > 0:
                adjusted[right_id] = (x, y, w + right_delta, h)
    return adjusted

panel_layout = tighten_figure1_panel_layout(panel_layout, panels, row_panel_orders)


def add_figure1_panel_gaps(layout, row_panel_orders, gap_px=6):
    adjusted = dict(layout)
    gap_px = int(gap_px)

    row_extents = []
    for row_ids in row_panel_orders:
        row_top = min(adjusted[panel_id][1] for panel_id in row_ids)
        row_bottom = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in row_ids)
        row_extents.append((row_top, row_bottom, row_bottom - row_top))

    figure_top = row_extents[0][0]
    figure_bottom = row_extents[-1][1]
    available_h = (figure_bottom - figure_top) - gap_px * (len(row_panel_orders) - 1)
    total_original_h = sum(row_h for _row_top, _row_bottom, row_h in row_extents)
    row_heights = [
        int(round(available_h * row_h / total_original_h))
        for _row_top, _row_bottom, row_h in row_extents
    ]
    row_heights[-1] += available_h - sum(row_heights)

    y = figure_top
    for row_idx, row_ids in enumerate(row_panel_orders):
        row_ids = tuple(row_ids)
        row_h = row_heights[row_idx]
        row_left = min(adjusted[panel_id][0] for panel_id in row_ids)
        row_right = max(adjusted[panel_id][0] + adjusted[panel_id][2] for panel_id in row_ids)
        row_w = row_right - row_left

        available_w = row_w - gap_px * (len(row_ids) - 1)
        original_widths = [adjusted[panel_id][2] for panel_id in row_ids]
        total_original_w = sum(original_widths)
        widths = [
            int(round(available_w * panel_w / total_original_w))
            for panel_w in original_widths
        ]
        widths[-1] += available_w - sum(widths)

        x = row_left
        for panel_id, panel_w in zip(row_ids, widths):
            adjusted[panel_id] = (x, y, panel_w, row_h)
            x += panel_w + gap_px

        y += row_h + gap_px

    return adjusted


panel_layout = add_figure1_panel_gaps(panel_layout, row_panel_orders, gap_px=6)


def align_figure1_region_edges(layout):
    adjusted = dict(layout)
    c_left = adjusted["C"][0]
    e_right = adjusted["E"][0] + adjusted["E"][2]

    for panel_id in ("F", "I"):
        x, y, w, h = adjusted[panel_id]
        left_delta = x - c_left
        if left_delta > 0:
            adjusted[panel_id] = (c_left, y, w + left_delta, h)

    for panel_id in ("H", "K"):
        x, y, w, h = adjusted[panel_id]
        right_delta = e_right - (x + w)
        if right_delta > 0:
            adjusted[panel_id] = (x, y, w + right_delta, h)

    for bottom_id, top_id in (("I", "F"), ("J", "G"), ("K", "H")):
        _x, y, _w, h = adjusted[bottom_id]
        top_x, _top_y, top_w, _top_h = adjusted[top_id]
        adjusted[bottom_id] = (top_x, y, top_w, h)

    return adjusted


panel_layout = align_figure1_region_edges(panel_layout)

figure1_region_border_groups = [
    (("F",), FRONS_BORDER_COLOR),
    (("G",), LABRUM_BORDER_COLOR),
    (("H",), MAXILLA_BORDER_COLOR),
    (("I",), MAXILLA_BORDER_COLOR),
    (("J", "K"), LABIUM_BORDER_COLOR),
]

for panel_ids, _border_color in figure1_region_border_groups:
    for panel_id in panel_ids:
        panels[panel_id].pop("border", None)


def add_clipped_group_border(canvas, layout, panel_ids, color="black", thickness=12, max_gap_share_px=0, reference_panel_ids=None):
    panel_ids = tuple(panel_ids)
    boxes = [layout[panel_id] for panel_id in panel_ids]
    group_left = min(box[0] for box in boxes)
    group_top = min(box[1] for box in boxes)
    group_right = max(box[0] + box[2] for box in boxes)
    group_bottom = max(box[1] + box[3] for box in boxes)

    reference_left = reference_right = None
    if reference_panel_ids:
        reference_boxes = [layout[panel_id] for panel_id in reference_panel_ids]
        reference_left = min(box[0] for box in reference_boxes)
        reference_right = max(box[0] + box[2] for box in reference_boxes)

    def nearest_gap_on_side(side):
        gaps = []
        for other_id, (ox, oy, ow, oh) in layout.items():
            if other_id in panel_ids:
                continue
            other_left = ox
            other_top = oy
            other_right = ox + ow
            other_bottom = oy + oh
            x_overlap = min(group_right, other_right) - max(group_left, other_left)
            y_overlap = min(group_bottom, other_bottom) - max(group_top, other_top)
            if side == "left" and y_overlap > 0 and other_right <= group_left:
                gaps.append(group_left - other_right)
            elif side == "right" and y_overlap > 0 and other_left >= group_right:
                gaps.append(other_left - group_right)
            elif side == "top" and x_overlap > 0 and other_bottom <= group_top:
                gaps.append(group_top - other_bottom)
            elif side == "bottom" and x_overlap > 0 and other_top >= group_bottom:
                gaps.append(other_top - group_bottom)
        return min(gaps) if gaps else None

    left_gap = nearest_gap_on_side("left")
    right_gap = nearest_gap_on_side("right")
    top_gap = nearest_gap_on_side("top")
    bottom_gap = nearest_gap_on_side("bottom")

    left_share = max(0, min(int(max_gap_share_px), left_gap // 2)) if left_gap is not None else 0
    right_share = max(0, min(int(max_gap_share_px), right_gap // 2)) if right_gap is not None else 0
    top_share = max(0, min(int(max_gap_share_px), top_gap // 2)) if top_gap is not None else 0
    bottom_share = max(0, min(int(max_gap_share_px), bottom_gap // 2)) if bottom_gap is not None else 0

    x0 = group_left - left_share
    x1 = group_right + right_share - 1
    if reference_left is not None and left_gap is None and group_left <= reference_left + max_gap_share_px:
        x0 = reference_left
    if reference_right is not None and right_gap is None and group_right >= reference_right - max_gap_share_px:
        x1 = reference_right - 1

    y0 = group_top - top_share
    y1 = group_bottom + bottom_share - 1

    draw = ImageDraw.Draw(canvas)
    thickness = int(thickness)
    draw.rectangle([x0, y0, x1, y0 + thickness - 1], fill=color)
    draw.rectangle([x0, y1 - thickness + 1, x1, y1], fill=color)
    draw.rectangle([x0, y0, x0 + thickness - 1, y1], fill=color)
    draw.rectangle([x1 - thickness + 1, y0, x1, y1], fill=color)
    return canvas


canvas = render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels, dpi=DPI,
    scale_font_size=60,
    row_labels=figure1_row_labels,
    axis_font_size=60,
)

draw = ImageDraw.Draw(canvas)
a_x, a_y, a_w, _a_h = panel_layout["A"]
title_font_size = 54
species_font = load_font_safe("ariali.ttf", title_font_size)
title_font = load_font_safe("arial.ttf", title_font_size)
species_text = "Ooceraea biroi"
suffix_text = " 4th instar larval head"
species_w, species_h = text_size(draw, species_text, species_font)
suffix_w, suffix_h = text_size(draw, suffix_text, title_font)
title_w = species_w + suffix_w
title_h = max(species_h, suffix_h)
title_x = a_x + (a_w - title_w) // 2
title_y = max(8, a_y - title_h - inches_to_px(0.06, DPI))
#draw.text((title_x, title_y), species_text, fill="black", font=species_font)
#draw.text((title_x + species_w, title_y), suffix_text, fill="black", font=title_font)

for panel_ids, border_color in figure1_region_border_groups:
    canvas = add_clipped_group_border(
        canvas,
        panel_layout,
        panel_ids,
        color=border_color,
        thickness=thickness_border,
        # Preserve the full 6 px panel gap instead of extending borders into it.
        max_gap_share_px=0,
        reference_panel_ids=("C", "D", "E"),
    )

canvas = register_figure_canvas('4th SEM Larval Chemosensory Organs', canvas, left_grace_in=0.04, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

# Figure 2

## Figure 2: Orco Organs and Cells


In [ ]:
border_thickness = 14
scale_bar_thickness = 12

panels = {
    "A": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2021-2023\Documents\Kronauer Lab\Confocal\DH_008_04142022\larva2\MAX_l4 dapi orco647aldapil.png",
        "view": {"zoom": 1.06, "x": 0.90, "y": 0.2},
        "crop_bottom_px": 50,
        "fit": "contain",
        "background": "black",
        "label_color": "white",
        "border": {"color": "#000000", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 40, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            {"text": "α-Orco", "x": 0.03, "y": 0.91, "size": 75, "color": "magenta"},
            {"text": "Autofluorescence", "x": 0.02, "y": 0.95, "size": 75, "color": "white"},
            {"text": "Labium", "x": 0.40, "y": 0.28, "size": 80, "color": "#E69F00"},
            {"text": "Maxilla", "x": -0.075, "y": 0.36, "size": 80, "color": "#E69F00", "rotation": 90},
        ],
    },
    "B": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2021-2023\Documents\Kronauer Lab\Confocal\DH_008_04142022\larva3\MAX_l4 dapi orco647all3.png",
        "view": {"zoom": 2.7, "x": 0.33, "y": 0.165},
        "label_color": "white",
        "border": {"color": "#8A9562", "thickness": border_thickness},
        "scale_bar": {"full_px": 49, "orig_um": 10, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            {"text": "Maxillary", "x": 0.18, "y": 0.77, "size": 65, "color": "#8A9562"},
            {"text": "Palp", "x": 0.36, "y": 0.87, "size": 65, "color": "#8A9562"},
        ],
    },
    "C": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\Experiment 2025-01-27\Experiment 2025-01-27\Snap-319.png",
        "view": {"zoom": 8, "x": 0.80, "y": 0.45},
        "label_color": "white",
        "scale_bar": {"full_px": 638, "orig_um": 50, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            #{"text": "OrW1", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "Composite", "x": 0.05, "y": 0.72, "size": 65, "color": "white"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
    },
    "D": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\Experiment 2025-01-27\Experiment 2025-01-27\Snap-319DAPI.png",
        "view": {"zoom": 8, "x": 0.80, "y": 0.45},
        "label_color": "white",
        "annotations": [
            #{"text": "OrW1", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            #{"text": "Orco", "x": 0.05, "y": 0.72, "size": 70, "color": "magenta"},
            {"text": "DAPI", "x": 0.05, "y": 0.72, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 638, "orig_um": 50, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "E": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\Experiment 2025-01-27\Experiment 2025-01-27\Snap-319ORCO.png",
        "view": {"zoom": 8, "x": 0.80, "y": 0.45},
        "label_color": "white",
            "annotations": [
            #{"text": "OrW1", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "Orco", "x": 0.05, "y": 0.72, "size": 70, "color": "magenta"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 638, "orig_um": 50, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "F": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\Experiment 2025-01-27\Experiment 2025-01-27\Snap-319W1.png",
        "view": {"zoom": 8, "x": 0.80, "y": 0.45},
        "label_color": "white",
        "label_color": "white",
            "annotations": [
            {"text": "OrW1", "x": 0.05, "y": 0.70, "size": 70, "color": "cyan"},
            #{"text": "Orco", "x": 0.05, "y": 0.72, "size": 70, "color": "magenta"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 638, "orig_um": 50, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "G": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2021-2023\Documents\Kronauer Lab\Confocal\DH_008_04142022\larva2\MAX_l4 dapi orco647all_labialpalp_closeup.png",
        "view": {"zoom": 6.0, "x": 0.77, "y": 0.31},
        "label_color": "white",
        "border": {"color": "#d7c7a0", "thickness": border_thickness},
        "scale_bar": {"full_px": 62, "orig_um": 10, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            {"text": "Labial", "x": 0.05, "y": 0.70, "size": 70, "color": "#d7c7a0"},
            {"text": "Palp", "x": 0.59, "y": 0.70, "size": 70, "color": "#d7c7a0"},
        ],
    },
    "H": {
        "path": r"D:\larval chemosensory\Experiment-1459\ROI 01 - labium - z004 of 8\Experiment-1459_z003_roi01_combined_ch0_E3_ch2_Orco_ch3_DAPI.png",
        "view": {"zoom": 1.3, "x": 0.9, "y": 0.4},
        "label_color": "white",
            "scale_bar": {"full_px": 276, "orig_um": 10, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "Composite", "x": 0.05, "y": 0.70, "size": 65, "color": "white"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
    },
    "I": {
        "path": r"D:\larval chemosensory\Experiment-1459\ROI 01 - labium - z004 of 8\Experiment-1459_z003_roi01_ch3_DAPI.png",
        "view": {"zoom": 1.3, "x": 0.9, "y": 0.4},
        "label_color": "white",
        "annotations": [
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "DAPI", "x": 0.05, "y": 0.70, "size": 70, "color": "white"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 276, "orig_um": 10, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "J": {
        "path": r"D:\larval chemosensory\Experiment-1459\ROI 01 - labium - z004 of 8\Experiment-1459_z003_roi01_ch2_Orco.png",
        "view": {"zoom": 1.3, "x": 0.9, "y": 0.4},
        "label_color": "white",
        "annotations": [
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "Orco", "x": 0.05, "y": 0.70, "size": 70, "color": "magenta"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 276, "orig_um": 10, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "K": {
        "path": r"D:\larval chemosensory\Experiment-1459\ROI 01 - labium - z004 of 8\Experiment-1459_z003_roi01_ch0_E3.png",
        "view": {"zoom": 1.3, "x": 0.9, "y": 0.4},
        "label_color": "white",
        "annotations": [
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "OrE3", "x": 0.05, "y": 0.70, "size": 70, "color": "cyan"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 276, "orig_um": 10, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
}


orco_copy_even_panel_tuning = {
    # Use this block to tune images and annotations in this layout only.
}

panels = apply_panel_tuning(panels, orco_copy_even_panel_tuning)

# Remove panel-level borders; consistent outward borders are added after rendering.
for panel_id in ("A", "B", "C", "D", "E", "F", "G", "H", "I", "J", "K"):
    panels[panel_id].pop("border", None)

border_thickness_px = 14
panel_gap_px = 6
# Separate bordered groups need room for two outward borders plus the same 6 px visible gap.
group_gap_px = panel_gap_px + 2 * border_thickness_px
row_gap_px = group_gap_px
stack_gap_px = panel_gap_px
figure2_orco_copy_even_fig_h_px = fig_h_px


def add_orco_copy_even_outside_group_border(canvas, layout, panel_ids, color="black", thickness=4):
    boxes = [layout[panel_id] for panel_id in panel_ids]
    x0 = min(box[0] for box in boxes)
    y0 = min(box[1] for box in boxes)
    x1 = max(box[0] + box[2] for box in boxes)
    y1 = max(box[1] + box[3] for box in boxes)
    draw = ImageDraw.Draw(canvas)
    for offset in range(int(thickness)):
        # Start one pixel beyond the panel bounds so no image pixels are covered.
        draw.rectangle(
            [x0 - 1 - offset, y0 - 1 - offset, x1 + offset, y1 + offset],
            outline=color,
        )
    return canvas


def build_figure2_orco_copy_even_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, panel_gap_px, group_gap_px
):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    usable_h = inner_h - row_gap_px
    row1_height_fraction = 0.75
    row1_h = int(round(usable_h * row1_height_fraction))
    row2_h = usable_h - row1_h
    row1_y = mt
    row2_y = row1_y + row1_h + row_gap_px

    layout = {}

    # Row 1: A at left; former C-G (now real B-F) stack vertically at right.
    top_available_w = inner_w - group_gap_px
    original_stack_fraction = 0.30
    stack_width_reduction_fraction = 0.35
    stack_fraction = original_stack_fraction * (1.0 - stack_width_reduction_fraction)
    a_width_fraction = 1.0 - stack_fraction
    a_w = int(round(top_available_w * a_width_fraction))
    stack_w = top_available_w - a_w
    stack_x = ml + a_w + group_gap_px
    layout["A"] = (ml, row1_y, a_w, row1_h)

    available_stack_h = row1_h - 4 * panel_gap_px
    b_height_fraction = 0.36
    b_h = int(round(available_stack_h * b_height_fraction))
    remaining_stack_h = available_stack_h - b_h
    smaller_heights = [remaining_stack_h // 4] * 4
    smaller_heights[-1] += remaining_stack_h - sum(smaller_heights)
    stack_heights = [b_h, *smaller_heights]
    stack_y = row1_y
    for panel_id, panel_h in zip(("B", "C", "D", "E", "F"), stack_heights):
        layout[panel_id] = (stack_x, stack_y, stack_w, panel_h)
        stack_y += panel_h + panel_gap_px

    # Row 2: former H-L, now real G-K, at equal widths.
    row2_ids = ("G", "H", "I", "J", "K")
    available_row2_w = inner_w - panel_gap_px * (len(row2_ids) - 1)
    row2_widths = [available_row2_w // len(row2_ids)] * len(row2_ids)
    row2_widths[-1] += available_row2_w - sum(row2_widths)
    x = ml
    for panel_index, (panel_id, panel_w) in enumerate(zip(row2_ids, row2_widths)):
        layout[panel_id] = (x, row2_y, panel_w, row2_h)
        if panel_index < len(row2_ids) - 1:
            x += panel_w + panel_gap_px

    return layout


panel_layout = build_figure2_orco_copy_even_layout(
    fig_w_px,
    figure2_orco_copy_even_fig_h_px,
    ml,
    mr,
    mt,
    mb,
    row_gap_px,
    panel_gap_px,
    group_gap_px,
)

# Match every row-2 panel to the height of the small D-F reference panels.
figure2_orco_copy_even_row2_ids = ("G", "H", "I", "J", "K")
figure2_orco_copy_even_row2_h = panel_layout["D"][3]
for panel_id in figure2_orco_copy_even_row2_ids:
    x, y, w, _old_h = panel_layout[panel_id]
    panel_layout[panel_id] = (x, y, w, figure2_orco_copy_even_row2_h)

# Shorten the canvas to the adjusted row-2 bottom plus the standard margin.
figure2_orco_copy_even_row2_bottom = max(
    panel_layout[panel_id][1] + panel_layout[panel_id][3]
    for panel_id in figure2_orco_copy_even_row2_ids
)
figure2_orco_copy_even_fig_h_px = figure2_orco_copy_even_row2_bottom + mb

# Cache resized panel tiles and scale calculations across repeated runs of this cell.
# The source modification time is part of each key, so edited image files refresh automatically.
import os

figure2_orco_copy_even_tile_cache = globals().get("figure2_orco_copy_even_tile_cache", {})
figure2_orco_copy_even_scale_cache = globals().get("figure2_orco_copy_even_scale_cache", {})


def _figure2_orco_copy_even_freeze(value):
    if isinstance(value, dict):
        return tuple(sorted((key, _figure2_orco_copy_even_freeze(item)) for key, item in value.items()))
    if isinstance(value, (list, tuple)):
        return tuple(_figure2_orco_copy_even_freeze(item) for item in value)
    return value


def _figure2_orco_copy_even_source_stamp(img_path):
    try:
        return os.path.getmtime(os.fspath(img_path))
    except OSError:
        return None


_figure2_orco_copy_even_base_paste_with_view = paste_with_view
_figure2_orco_copy_even_base_compute_panel_scale = compute_panel_scale


def _figure2_orco_copy_even_cached_paste_with_view(
    img_path, panel_w, panel_h, view=None, crop_bottom_px=0, rotate=None,
    padding_px=0, padding_color="white", fit="cover", background="white",
):
    key = (
        os.fspath(img_path),
        _figure2_orco_copy_even_source_stamp(img_path),
        int(panel_w),
        int(panel_h),
        _figure2_orco_copy_even_freeze(view),
        int(crop_bottom_px),
        rotate,
        _figure2_orco_copy_even_freeze(padding_px),
        padding_color,
        fit,
        background,
    )
    cached = figure2_orco_copy_even_tile_cache.get(key)
    if cached is not None:
        return cached.copy()
    tile = _figure2_orco_copy_even_base_paste_with_view(
        img_path,
        panel_w,
        panel_h,
        view=view,
        crop_bottom_px=crop_bottom_px,
        rotate=rotate,
        padding_px=padding_px,
        padding_color=padding_color,
        fit=fit,
        background=background,
    )
    if len(figure2_orco_copy_even_tile_cache) >= 32:
        figure2_orco_copy_even_tile_cache.pop(next(iter(figure2_orco_copy_even_tile_cache)))
    figure2_orco_copy_even_tile_cache[key] = tile.copy()
    return tile


def _figure2_orco_copy_even_cached_compute_panel_scale(
    img_path, panel_w, panel_h, view=None, crop_bottom_px=0, rotate=None,
    padding_px=0, padding_color="white",
):
    key = (
        os.fspath(img_path),
        _figure2_orco_copy_even_source_stamp(img_path),
        int(panel_w),
        int(panel_h),
        _figure2_orco_copy_even_freeze(view),
        int(crop_bottom_px),
        rotate,
        _figure2_orco_copy_even_freeze(padding_px),
        padding_color,
    )
    if key not in figure2_orco_copy_even_scale_cache:
        if len(figure2_orco_copy_even_scale_cache) >= 64:
            figure2_orco_copy_even_scale_cache.pop(next(iter(figure2_orco_copy_even_scale_cache)))
        figure2_orco_copy_even_scale_cache[key] = _figure2_orco_copy_even_base_compute_panel_scale(
            img_path,
            panel_w,
            panel_h,
            view=view,
            crop_bottom_px=crop_bottom_px,
            rotate=rotate,
            padding_px=padding_px,
            padding_color=padding_color,
        )
    return figure2_orco_copy_even_scale_cache[key]


# Temporarily use the cached wrappers only for this figure, then restore the shared helpers.
paste_with_view = _figure2_orco_copy_even_cached_paste_with_view
compute_panel_scale = _figure2_orco_copy_even_cached_compute_panel_scale
try:
    canvas = render_artboard(
        fig_w_px,
        figure2_orco_copy_even_fig_h_px,
        panel_layout,
        panels,
        dpi=DPI,
    )
finally:
    paste_with_view = _figure2_orco_copy_even_base_paste_with_view
    compute_panel_scale = _figure2_orco_copy_even_base_compute_panel_scale

canvas = add_orco_copy_even_outside_group_border(
    canvas, panel_layout, ("A",), color="#000000", thickness=border_thickness_px
)
canvas = add_orco_copy_even_outside_group_border(
    canvas, panel_layout, ("B", "C", "D", "E", "F"),
    color=MAXILLA_BORDER_COLOR, thickness=border_thickness_px,
)
canvas = add_orco_copy_even_outside_group_border(
    canvas, panel_layout, ("G", "H", "I", "J", "K"),
    color=LABIUM_BORDER_COLOR, thickness=border_thickness_px,
)

canvas = register_figure_canvas('Orco Organs and Cells copy even heights', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure 2: Orco Organs and Cells New


In [ ]:
border_thickness = 14
scale_bar_thickness = 12

from copy import deepcopy

figure2_orco_new_source_panels = {
    "A": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2021-2023\Documents\Kronauer Lab\Confocal\DH_008_04142022\larva2\MAX_l4 dapi orco647aldapil.png",
        "view": {"zoom": 1.06, "x": 0.90, "y": 0.5},
        "crop_bottom_px": 50,
        "fit": "contain",
        "background": "black",
        "label_color": "white",
        "border": {"color": "#000000", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 40, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            {"text": "α-Orco", "x": 0.03, "y": 0.91, "size": 75, "color": "magenta"},
            {"text": "Autofluorescence", "x": 0.02, "y": 0.95, "size": 75, "color": "white"},
            {"text": "Labium", "x": 0.40, "y": 0.28, "size": 80, "color": "#E69F00"},
            {"text": "Maxilla", "x": -0.095, "y": 0.39, "size": 80, "color": "#E69F00", "rotation": 90},
        ],
    },
    "B": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2021-2023\Documents\Kronauer Lab\Confocal\DH_008_04142022\larva3\MAX_l4 dapi orco647all3.png",
        "view": {"zoom": 3.0, "x": 0.33, "y": 0.23, "rotate": 4},
        "label_color": "white",
        "border": {"color": "#8A9562", "thickness": border_thickness},
        "scale_bar": {"full_px": 49, "orig_um": 10, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            block_arrow(x=0.52,y=0.23,angle=90,length=0.18,color="white",shaft_width=12,head_length=0.45,head_width=32),
            {"text": "Maxillary", "x": 0.18, "y": 0.77, "size": 80, "color": "#8A9562"},
            {"text": "Palp", "x": 0.36, "y": 0.87, "size": 80, "color": "#8A9562"},
        ],
    },
    "C": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\Experiment 2025-01-27\Experiment 2025-01-27\Snap-319.png",
        "view": {"zoom": 8, "x": 0.80, "y": 0.45},
        "label_color": "white",
        "scale_bar": {"full_px": 638, "orig_um": 50, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            #{"text": "OrW1", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "Composite", "x": 0.05, "y": 0.72, "size": 65, "color": "white"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
    },
    "D": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\Experiment 2025-01-27\Experiment 2025-01-27\Snap-319DAPI.png",
        "view": {"zoom": 8, "x": 0.80, "y": 0.45},
        "label_color": "white",
        "annotations": [
            #{"text": "OrW1", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            #{"text": "Orco", "x": 0.05, "y": 0.72, "size": 70, "color": "magenta"},
            {"text": "DAPI", "x": 0.05, "y": 0.72, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 638, "orig_um": 50, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "E": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\Experiment 2025-01-27\Experiment 2025-01-27\Snap-319ORCO.png",
        "view": {"zoom": 8, "x": 0.80, "y": 0.45},
        "label_color": "white",
            "annotations": [
            #{"text": "OrW1", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "Orco", "x": 0.05, "y": 0.72, "size": 70, "color": "magenta"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 638, "orig_um": 50, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "F": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\Experiment 2025-01-27\Experiment 2025-01-27\Snap-319W1.png",
        "view": {"zoom": 8, "x": 0.80, "y": 0.45},
        "label_color": "white",
        "label_color": "white",
            "annotations": [
            {"text": "OrW1", "x": 0.05, "y": 0.70, "size": 70, "color": "cyan"},
            #{"text": "Orco", "x": 0.05, "y": 0.72, "size": 70, "color": "magenta"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 638, "orig_um": 50, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "G": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2021-2023\Documents\Kronauer Lab\Confocal\DH_008_04142022\larva2\MAX_l4 dapi orco647all_labialpalp_closeup.png",
        "view": {"zoom": 6.0, "x": 0.75, "y": 0.31},
        "label_color": "white",
        "border": {"color": "#d7c7a0", "thickness": border_thickness},
        "scale_bar": {"full_px": 62, "orig_um": 10, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            block_arrow(x=0.38,y=0.4,angle=300,length=0.18,color="white",shaft_width=12,head_length=0.45,head_width=32),
            block_arrow(x=0.47,y=0.60,angle=300,length=0.18,color="white",shaft_width=12,head_length=0.45,head_width=32),
            {"text": "Labial", "x": 0.25, "y": 0.77, "size": 80, "color": "#d7c7a0"},
            {"text": "Palp", "x": 0.31, "y": 0.87, "size": 80, "color": "#d7c7a0"},
        ],
    },
    "H": {
        "path": r"D:\larval chemosensory\Experiment-1459\ROI 01 - labium - z004 of 8\Experiment-1459_z003_roi01_combined_ch0_E3_ch2_Orco_ch3_DAPI.png",
        "view": {"zoom": 1.3, "x": 0.9, "y": 0.4},
        "label_color": "white",
            "scale_bar": {"full_px": 276, "orig_um": 10, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "Composite", "x": 0.05, "y": 0.70, "size": 65, "color": "white"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
    },
    "I": {
        "path": r"D:\larval chemosensory\Experiment-1459\ROI 01 - labium - z004 of 8\Experiment-1459_z003_roi01_ch3_DAPI.png",
        "view": {"zoom": 1.3, "x": 0.9, "y": 0.4},
        "label_color": "white",
        "annotations": [
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "DAPI", "x": 0.05, "y": 0.70, "size": 70, "color": "white"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 276, "orig_um": 10, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "J": {
        "path": r"D:\larval chemosensory\Experiment-1459\ROI 01 - labium - z004 of 8\Experiment-1459_z003_roi01_ch2_Orco.png",
        "view": {"zoom": 1.3, "x": 0.9, "y": 0.4},
        "label_color": "white",
        "annotations": [
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "Orco", "x": 0.05, "y": 0.70, "size": 70, "color": "magenta"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 276, "orig_um": 10, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
    "K": {
        "path": r"D:\larval chemosensory\Experiment-1459\ROI 01 - labium - z004 of 8\Experiment-1459_z003_roi01_ch0_E3.png",
        "view": {"zoom": 1.3, "x": 0.9, "y": 0.4},
        "label_color": "white",
        "annotations": [
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 70, "color": "cyan"},
            {"text": "OrE3", "x": 0.05, "y": 0.70, "size": 70, "color": "cyan"},
            #{"text": "DAPI", "x": 0.05, "y": 0.60, "size": 70, "color": "white"},
        ],
        "scale_bar": {"full_px": 276, "orig_um": 10, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
    },
}


# Relabel by reading order. In particular, the former G panel is now panel C.
figure2_orco_new_source_order = ("A", "B", "G", "C", "D", "E", "F", "H", "I", "J", "K")
panels = {
    new_panel_id: deepcopy(figure2_orco_new_source_panels[source_panel_id])
    for new_panel_id, source_panel_id in zip(
        tuple("ABCDEFGHIJK"), figure2_orco_new_source_order
    )
}

figure2_orco_new_panel_tuning = {
    # Use this block to tune images and annotations in this layout only.
}
panels = apply_panel_tuning(panels, figure2_orco_new_panel_tuning)

# Draw consistent group borders outside the image areas after rendering.
for panel_id in tuple("ABCDEFGHIJK"):
    panels[panel_id].pop("border", None)

figure2_orco_new_border_thickness_px = 14
figure2_orco_new_panel_gap_px = 6
# Separate bordered groups need room for two outward borders and a 6 px visible gap.
figure2_orco_new_group_gap_px = (
    figure2_orco_new_panel_gap_px + 2 * figure2_orco_new_border_thickness_px
)
figure2_orco_new_row_gap_px = figure2_orco_new_group_gap_px


def add_figure2_orco_new_outside_group_border(
    canvas, layout, panel_ids, color="black", thickness=4
):
    boxes = [layout[panel_id] for panel_id in panel_ids]
    x0 = min(box[0] for box in boxes)
    y0 = min(box[1] for box in boxes)
    x1 = max(box[0] + box[2] for box in boxes)
    y1 = max(box[1] + box[3] for box in boxes)
    draw = ImageDraw.Draw(canvas)
    for offset in range(int(thickness)):
        draw.rectangle(
            [x0 - 1 - offset, y0 - 1 - offset, x1 + offset, y1 + offset],
            outline=color,
        )
    return canvas


def build_figure2_orco_new_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, panel_gap_px, group_gap_px
):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb

    # Three visual rows: a tall overview row followed by two equal cell rows.
    usable_h = inner_h - 2 * row_gap_px
    top_h = int(round(usable_h * 0.76))
    lower_rows_h = usable_h - top_h
    row2_h = lower_rows_h // 2
    row3_h = lower_rows_h - row2_h
    row1_y = mt
    row2_y = row1_y + top_h + row_gap_px
    row3_y = row2_y + row2_h + row_gap_px

    layout = {}

    # Row 1: large A at left; B and C (former G) stacked at right.
    top_available_w = inner_w - group_gap_px
    right_stack_fraction = 0.285
    stack_w = int(round(top_available_w * right_stack_fraction))
    a_w = top_available_w - stack_w
    stack_x = ml + a_w + group_gap_px
    layout["A"] = (ml, row1_y, a_w, top_h)

    stack_available_h = top_h - group_gap_px
    b_h = stack_available_h // 2
    c_h = stack_available_h - b_h
    layout["B"] = (stack_x, row1_y, stack_w, b_h)
    layout["C"] = (stack_x, row1_y + b_h + group_gap_px, stack_w, c_h)

    # Row 2: former C-F, now D-G.
    row2_ids = ("D", "E", "F", "G")
    row2_available_w = inner_w - panel_gap_px * (len(row2_ids) - 1)
    row2_widths = [row2_available_w // len(row2_ids)] * len(row2_ids)
    row2_widths[-1] += row2_available_w - sum(row2_widths)
    x = ml
    for panel_index, (panel_id, panel_w) in enumerate(zip(row2_ids, row2_widths)):
        layout[panel_id] = (x, row2_y, panel_w, row2_h)
        if panel_index < len(row2_ids) - 1:
            x += panel_w + panel_gap_px

    # Row 3: former H-K, retaining H-K after the reading-order relabel.
    row3_ids = ("H", "I", "J", "K")
    row3_available_w = inner_w - panel_gap_px * (len(row3_ids) - 1)
    row3_widths = [row3_available_w // len(row3_ids)] * len(row3_ids)
    row3_widths[-1] += row3_available_w - sum(row3_widths)
    x = ml
    for panel_index, (panel_id, panel_w) in enumerate(zip(row3_ids, row3_widths)):
        layout[panel_id] = (x, row3_y, panel_w, row3_h)
        if panel_index < len(row3_ids) - 1:
            x += panel_w + panel_gap_px

    return layout


figure2_orco_new_fig_h_px = fig_h_px
panel_layout = build_figure2_orco_new_layout(
    fig_w_px,
    figure2_orco_new_fig_h_px,
    ml,
    mr,
    mt,
    mb,
    figure2_orco_new_row_gap_px,
    figure2_orco_new_panel_gap_px,
    figure2_orco_new_group_gap_px,
)

canvas = render_artboard(
    fig_w_px,
    figure2_orco_new_fig_h_px,
    panel_layout,
    panels,
    dpi=DPI,
)

canvas = add_figure2_orco_new_outside_group_border(
    canvas,
    panel_layout,
    ("A",),
    color="#000000",
    thickness=figure2_orco_new_border_thickness_px,
)
canvas = add_figure2_orco_new_outside_group_border(
    canvas,
    panel_layout,
    ("B",),
    color=MAXILLA_BORDER_COLOR,
    thickness=figure2_orco_new_border_thickness_px,
)
canvas = add_figure2_orco_new_outside_group_border(
    canvas,
    panel_layout,
    ("C",),
    color=LABIUM_BORDER_COLOR,
    thickness=figure2_orco_new_border_thickness_px,
)
canvas = add_figure2_orco_new_outside_group_border(
    canvas,
    panel_layout,
    ("D", "E", "F", "G"),
    color=MAXILLA_BORDER_COLOR,
    thickness=figure2_orco_new_border_thickness_px,
)
canvas = add_figure2_orco_new_outside_group_border(
    canvas,
    panel_layout,
    ("H", "I", "J", "K"),
    color=LABIUM_BORDER_COLOR,
    thickness=figure2_orco_new_border_thickness_px,
)

canvas = register_figure_canvas(
    "Figure 2: Orco Organs and Cells New",
    canvas,
    panel_configs=panels,
    panel_layout=panel_layout,
)

display(canvas)


# Figure 3


## Figure 3: Ir25a sensory organ and cells


In [ ]:
panels = {
    "A": {
        "path": r"D:\larval chemosensory\MIP_P22_outline.png",
        "scale_bar": {"full_px": 271, "orig_um": 50, "new_um": 100, "color": "white", "thickness": 12, "number": False},
        "view": {"zoom": 1.8, "x": 0.57, "y": 0.25, "rotate":55},
          'annotations': [
                {"text": "α-GFP", "x": 0.80, "y": 0.88, "size": 60, "color": "white"},
                {"text": "Ir25a>GCaMP", "x": 0.70, "y": 0.93, "size": 60, "color": "white"},
                {'text': 'Labrum', 'x': 0.43, 'y': 0.4, 'size': 50, 'color': 'white'},
                {'text': 'Maxillary', 'x': 0.07, 'y': 0.28, 'size': 50, 'rotate': 90, 'color': 'white'},
                {'text': 'Palp', 'x': 0.15, 'y': 0.28, 'size': 50, 'rotate': 90, 'color': 'white'},
                {'text': 'Galea', 'x': 0.12, 'y': 0.08, 'size': 50, 'rotate': 45, 'color': 'white'},
                {'text': 'CB', 'x': 0.43, 'y': 0.84, 'size': 105, 'color': 'white'},
                {'text': 'Antn', 'x': 0.26, 'y': 0.84, 'size': 50, 'color': 'white'},
                {'text': 'SEG', 'x': 0.44, 'y': 0.69, 'size': 50, 'rotate': 0, 'color': 'white'},
                 {'text': 'Labium', 'x': 0.44, 'y': 0.02, 'size': 50, 'rotate': 0, 'color': 'white'}     
                     ],
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label_color": "white",
    },

    "B":     {'path': 'C:\\Users\\hejaz\\OneDrive - The Rockefeller University\\2024-Present\\skeletonize\\MIP_Z62_75_mediumBoost.png',
     'view': {'zoom': 1.4, 'x': 0.33, 'y': 0.75},
     'label_color': 'white',
     'scale_bar': {'full_px': 300, 'orig_um': 100, 'new_um': 25, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [{'text': 'Native', 'x': 0.075, 'y': 0.67, 'size': 50, 'color': 'white'},
                     {'text': 'GCaMP', 'x': 0.05, 'y': 0.80, 'size': 50, 'color': 'white'}],
     'border': {'color': '#B85C4F', 'thickness': 12}},

    "C":     {'path': 'D:\\P22_ir25a_20250421\\P22_ir25a_20250421\\ZSeries-04212025-1801-004\\MIP_slices3-5_enhanced.png',
     'view': {'zoom': 1.0, 'x': 0.5, 'y': 0.45},
     'label_color': 'white',
    'annotations': [{'text': '*', 'x': 0.575, 'y': 0.05, 'size': 70, 'color': '#E69F00'},
                     {'text': '*', 'x': 0.275, 'y': 0.24, 'size': 70, 'color': '#E69F00'},
                     {'text': '*', 'x': 0.18, 'y': 0.38, 'size': 70, 'color': '#E69F00'},
                     {'text': '*', 'x': 0.12, 'y': 0.52, 'size': 70, 'color': '#E69F00'}
                     ],
     'scale_bar': {'full_px': 890, 'orig_um': 100, 'new_um': 10, 'color': 'white', 'thickness': 12, 'number': False},
     'border': {'color': '#556E9B', 'thickness': 12}},

    "D":     {'path': 'c:\\Users\\hejaz\\OneDrive - The Rockefeller University\\2024-Present\\skeletonize\\MIP_boosted_contrast_5to99.png',
     'view': {'zoom': 1.0, 'x': 1.0, 'y': 0.34, 'rotate': 90},
     'label_color': 'white',
     'scale_bar': {'full_px': 890, 'orig_um': 100, 'new_um': 15, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [],
     'border': {'color': '#8A9562', 'thickness': 12}},

    "E":     {'path': 'D:\\P22_ir25a_20250421\\P22_ir25a_20250421\\ZSeries-04212025-1801-015\\MAX_ZSeries-04212025-1801-015_papillae_max2.png',
     'view': {'zoom': 1.7, 'x': 0.01, 'y': 0.6, 'rotate': 0},
     'label_color': 'white',
     'scale_bar': {'full_px': 890, 'orig_um': 100, 'new_um': 10, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [{'text': '*', 'x': 0.23, 'y': 0.02, 'size': 120, 'color': '#E69F00'},
                     {'text': '°', 'x': 0.24, 'y': 0.67, 'size': 120, 'color': '#E69F00'}],
     'border': {'color': '#8A9562', 'thickness': 12}},

    "F":     {'path': 'c:\\Users\\hejaz\\OneDrive - The Rockefeller University\\2024-Present\\skeletonize\\MIP_moreBoost_1to99.9.png',
     'view': {'zoom': 1.7, 'x': 0.85, 'y': 0.08, 'rotate': 270},
     'label_color': 'white',
     'scale_bar': {'full_px': 890, 'orig_um': 100, 'new_um': 10, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [{'text': '*', 'x': 0.15, 'y': 0.12, 'size': 120, 'color': '#E69F00'}],
     'border': {'color': '#8A9562', 'thickness': 12}},

    "G":     {'path': 'D:\\P22_ir25a_20250421\\P22_ir25a_20250421\\ZSeries-04212025-1801-010\\PNG_export\\MIP_full_enhanced.png',
     'view': {'zoom': 1.5, 'x': 0.9, 'y': 0.60},
     'label_color': 'white',
     'scale_bar': {'full_px': 890, 'orig_um': 100, 'new_um': 20, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [{'text': '*', 'x': 0.72, 'y': 0.55, 'size': 120, 'color': '#E69F00'},
                     {'text': '°', 'x': 0.065, 'y': 0.19, 'size': 120, 'color': '#E69F00'}],
     'border': {'color': '#d7c7a0', 'thickness': 12}},

    "H":     {'path': 'D:\\P22_ir25a_20250421\\P22_ir25a_20250421\\ZSeries-04212025-1801-006\\ZSeries-04212025-1801-006_labial_palp.png',
     'view': {'zoom': 2.0, 'x': 0.74, 'y': 0.88, 'rotate': 180},
     'label_color': 'white',
     'scale_bar': {'full_px': 890, 'orig_um': 100, 'new_um': 5, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [{'text': '*', 'x': 0.69, 'y': 0.35, 'size': 120, 'color': '#E69F00'}],
     'border': {'color': '#d7c7a0', 'thickness': 12}},

    "I":     {'path': 'D:\\20250922\\4\\antenna\\Experiment-1497_antenna_combined.png',
     'view': {'zoom': 8.0, 'x': 0.17, 'y': 0.83, 'rotate': -90},
     'scale_bar': {'full_px': 251, 'orig_um': 20, 'new_um': 5, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [{'text': 'DAPI', 'x': 0.05, 'y': 0.86, 'size': 60, 'color': 'white'},
                     {'text': 'Ir317.2', 'x': 0.05, 'y': 0.75, 'size': 60, 'color': 'magenta'},
                     {'text': 'Ir75f.3', 'x': 0.05, 'y': 0.64, 'size': 60, 'color': 'cyan'},
                     {'text': 'Ir25a', 'x': 0.05, 'y': 0.53, 'size': 60, 'color': 'yellow'}],
     'label_color': 'white',
     'border': {'color': '#B85C4F', 'thickness': 12}},

    "J":     {'path': 'D:\\20250922\\4\\labrum\\Experiment-1497_labrum_composite2.png',
     'view': {'zoom': 6.0, 'x': 0.23, 'y': 0.64, 'rotate': 10},
     'scale_bar': {'full_px': 251, 'orig_um': 20, 'new_um': 15, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [],
     'label_color': 'white',
     'border': {'color': '#556E9B', 'thickness': 12}},

    "K":     {'path': 'D:\\20250922\\4\\maxilla\\Labrum_1496_max_combined.png',
     'view': {'zoom': 4.0, 'x': 0.1, 'y': 0.69, 'rotate': -90},
     'scale_bar': {'full_px': 251, 'orig_um': 20, 'new_um': 10, 'color': 'white', 'thickness': 12, 'number': False},
     'annotations': [],
     'label_color': 'white',
     'border': {'color': '#8A9562', 'thickness': 12}},

    "L":     {'path': 'D:\\20250922\\4\\labium2\\Labrum_1496_2_combined.png',
     'view': {'zoom': 5.0, 'x': 0.2, 'y': 0.3, 'rotate': -90},
     'scale_bar': {'full_px': 251, 'orig_um': 20, 'new_um': 15, 'color': 'white', 'thickness': 12, 'number': False},
     #'annotations': [#{'text': '1', 'x': 0.12, 'y': 0.40, 'size': 60, 'color': '#E69F00'}],
     'label_color': 'white',
     'border': {'color': '#d7c7a0', 'thickness': 12}},

    "M":     {'path': 'D:\\20250922\\4\\Labrum_1496_Composite.png',
    'view': {'zoom': 5.0, 'x': 0.2, 'y': 0.3, 'rotate': -90},
     'scale_bar': {'full_px': 251, 'orig_um': 20, 'new_um': 15, 'color': 'white', 'thickness': 12, 'number': False},
     #'annotations': [#{'text': '2', 'x': 0.55, 'y': 0.60, 'size': 60, 'color': '#E69F00'}],
     'label_color': 'white',
     'border': {'color': '#d7c7a0', 'thickness': 12}},
}

figure3_copy_rows = [
    (("A", "B", "C", "D", "E"), "stacked"),
    (("F", "G", "H", "I"), "mixed"),
    (("J", "K", "L", "M"), "mixed"),
]

figure3_copy_border_groups = [
    (("B",), FRONS_BORDER_COLOR),
    (("C",), LABRUM_BORDER_COLOR),
    (("D", "E"), MAXILLA_BORDER_COLOR),
    (("F",), MAXILLA_BORDER_COLOR),
    (("G", "H"), LABIUM_BORDER_COLOR),
    (("I",), FRONS_BORDER_COLOR),
    (("J",), LABRUM_BORDER_COLOR),
    (("K",), MAXILLA_BORDER_COLOR),
    (("L", "M"), LABIUM_BORDER_COLOR),
]

figure3_copy_panel_tuning = {
    # Use this block to tune images and annotations in this layout only.
    # View and scale-bar updates merge with the explicit panel settings above.
}

panels = apply_panel_tuning(panels, figure3_copy_panel_tuning)

# The former C-N panels are now keyed directly as B-M; no display-label mapping is needed.

for panel_ids, _border_color in figure3_copy_rows:
    for panel_id in panel_ids:
        panels[panel_id].pop("border", None)

# Draw borders inward so their visible edges do not extend into adjacent gaps.
# This makes the requested gap the actual white space seen between panels.
border_thickness_px = 12
figure3_copy_label_border_gap_px = 12
figure3_copy_label_x_pad_px = border_thickness_px + figure3_copy_label_border_gap_px
figure3_copy_label_y_pad_px = inches_to_px(LABEL_PAD_IN, DPI)

panel_gap_px = 6
row_gap_px = panel_gap_px
stack_gap_px = panel_gap_px
figure3_copy_fig_h_px = fig_h_px


def add_inset_group_border(canvas, layout, panel_ids, color="black", thickness=4):
    boxes = [layout[panel_id] for panel_id in panel_ids]
    x0 = min(box[0] for box in boxes)
    y0 = min(box[1] for box in boxes)
    x1 = max(box[0] + box[2] for box in boxes) - 1
    y1 = max(box[1] + box[3] for box in boxes) - 1
    draw = ImageDraw.Draw(canvas)
    max_thickness = max(0, min(int(thickness), (x1 - x0 + 1) // 2, (y1 - y0 + 1) // 2))
    for inset in range(max_thickness):
        draw.rectangle(
            [x0 + inset, y0 + inset, x1 - inset, y1 - inset],
            outline=color,
        )
    return canvas


def build_figure3_copy_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, panel_gap_px
):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    usable_h = inner_h - 2 * row_gap_px
    row_height_ratios = (0.52, 0.24, 0.24)
    row_heights = [int(round(usable_h * ratio)) for ratio in row_height_ratios]
    row_heights[-1] += usable_h - sum(row_heights)

    layout = {}
    row1_y = mt
    row2_y = row1_y + row_heights[0] + row_gap_px
    row3_y = row2_y + row_heights[1] + row_gap_px

    # Row 1: A at left; B-E form a four-panel vertical stack at right.
    top_gap_px = panel_gap_px
    top_available_w = inner_w - top_gap_px
    a_w = int(round(top_available_w * 0.70))
    stack_w = top_available_w - a_w
    stack_x = ml + a_w + top_gap_px
    layout["A"] = (ml, row1_y, a_w, row_heights[0])

    available_stack_h = row_heights[0] - 3 * stack_gap_px
    stack_heights = [available_stack_h // 4] * 4
    stack_heights[-1] += available_stack_h - sum(stack_heights)
    stack_y = row1_y
    for panel_id, panel_h in zip(("B", "C", "D", "E"), stack_heights):
        layout[panel_id] = (stack_x, stack_y, stack_w, panel_h)
        stack_y += panel_h + stack_gap_px

    # Rows 2 and 3 contain F-I and J-M respectively.
    for panel_ids, y, row_h, gaps in (
        (("F", "G", "H", "I"), row2_y, row_heights[1], (panel_gap_px,) * 3),
        (("J", "K", "L", "M"), row3_y, row_heights[2], (panel_gap_px,) * 3),
    ):
        available_w = inner_w - sum(gaps)
        widths = [available_w // 4] * 4
        widths[-1] += available_w - sum(widths)
        x = ml
        for panel_index, (panel_id, panel_w) in enumerate(zip(panel_ids, widths)):
            layout[panel_id] = (x, y, panel_w, row_h)
            if panel_index < len(gaps):
                x += panel_w + gaps[panel_index]

    return layout


panel_layout = build_figure3_copy_layout(
    fig_w_px,
    figure3_copy_fig_h_px,
    ml,
    mr,
    mt,
    mb,
    row_gap_px,
    panel_gap_px,
)

# Cache resized panel tiles and scale calculations across repeated runs of this cell.
# The source modification time is part of each key, so edited image files refresh automatically.
import os

figure3_copy_tile_cache = globals().get("figure3_copy_tile_cache", {})
figure3_copy_scale_cache = globals().get("figure3_copy_scale_cache", {})


def _figure3_copy_freeze(value):
    if isinstance(value, dict):
        return tuple(sorted((key, _figure3_copy_freeze(item)) for key, item in value.items()))
    if isinstance(value, (list, tuple)):
        return tuple(_figure3_copy_freeze(item) for item in value)
    return value


def _figure3_copy_source_stamp(img_path):
    try:
        return os.path.getmtime(os.fspath(img_path))
    except OSError:
        return None


_figure3_copy_base_paste_with_view = paste_with_view
_figure3_copy_base_compute_panel_scale = compute_panel_scale


def _figure3_copy_cached_paste_with_view(
    img_path, panel_w, panel_h, view=None, crop_bottom_px=0, rotate=None,
    padding_px=0, padding_color="white", fit="cover", background="white",
):
    key = (
        os.fspath(img_path),
        _figure3_copy_source_stamp(img_path),
        int(panel_w),
        int(panel_h),
        _figure3_copy_freeze(view),
        int(crop_bottom_px),
        rotate,
        _figure3_copy_freeze(padding_px),
        padding_color,
        fit,
        background,
    )
    cached = figure3_copy_tile_cache.get(key)
    if cached is not None:
        return cached.copy()
    tile = _figure3_copy_base_paste_with_view(
        img_path,
        panel_w,
        panel_h,
        view=view,
        crop_bottom_px=crop_bottom_px,
        rotate=rotate,
        padding_px=padding_px,
        padding_color=padding_color,
        fit=fit,
        background=background,
    )
    if len(figure3_copy_tile_cache) >= 32:
        figure3_copy_tile_cache.pop(next(iter(figure3_copy_tile_cache)))
    figure3_copy_tile_cache[key] = tile.copy()
    return tile


def _figure3_copy_cached_compute_panel_scale(
    img_path, panel_w, panel_h, view=None, crop_bottom_px=0, rotate=None,
    padding_px=0, padding_color="white",
):
    key = (
        os.fspath(img_path),
        _figure3_copy_source_stamp(img_path),
        int(panel_w),
        int(panel_h),
        _figure3_copy_freeze(view),
        int(crop_bottom_px),
        rotate,
        _figure3_copy_freeze(padding_px),
        padding_color,
    )
    if key not in figure3_copy_scale_cache:
        if len(figure3_copy_scale_cache) >= 64:
            figure3_copy_scale_cache.pop(next(iter(figure3_copy_scale_cache)))
        figure3_copy_scale_cache[key] = _figure3_copy_base_compute_panel_scale(
            img_path,
            panel_w,
            panel_h,
            view=view,
            crop_bottom_px=crop_bottom_px,
            rotate=rotate,
            padding_px=padding_px,
            padding_color=padding_color,
        )
    return figure3_copy_scale_cache[key]


# Render without the default panel labels, then draw them below with separate
# horizontal and vertical insets. This keeps the shift self-contained in this
# figure cell, even when the shared helper cell has not been rerun.
figure3_copy_render_panels = {
    panel_id: {**panel_config, "label": ""}
    for panel_id, panel_config in panels.items()
}

# Temporarily use the cached wrappers only for this figure, then restore the shared helpers.
paste_with_view = _figure3_copy_cached_paste_with_view
compute_panel_scale = _figure3_copy_cached_compute_panel_scale
try:
    canvas = render_artboard(
        fig_w_px,
        figure3_copy_fig_h_px,
        panel_layout,
        figure3_copy_render_panels,
        dpi=DPI,
    )
finally:
    paste_with_view = _figure3_copy_base_paste_with_view
    compute_panel_scale = _figure3_copy_base_compute_panel_scale

for panel_ids, border_color in figure3_copy_border_groups:
    canvas = add_inset_group_border(
        canvas,
        panel_layout,
        panel_ids,
        color=border_color,
        thickness=border_thickness_px,
    )

figure3_copy_label_draw = ImageDraw.Draw(canvas)
figure3_copy_label_fonts = {}
for panel_id, panel_config in panels.items():
    x0, y0, _panel_w, _panel_h = panel_layout[panel_id]
    label_text = panel_config.get("label", panel_id)
    if not label_text:
        continue
    label_font_size = int(panel_config.get("label_font_size", 96))
    if label_font_size not in figure3_copy_label_fonts:
        figure3_copy_label_fonts[label_font_size] = load_font_safe("arial.ttf", label_font_size)
    figure3_copy_label_draw.text(
        (x0 + figure3_copy_label_x_pad_px, y0 + figure3_copy_label_y_pad_px),
        label_text,
        fill=panel_config.get("label_color", "black"),
        font=figure3_copy_label_fonts[label_font_size],
    )

canvas = register_figure_canvas('Ir25a sensory organ and cells copy', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

# Figure 4

## Figure 4: Central Brain Signal


In [ ]:
panels = {
    "A": {
        "path": r"D:\larval chemosensory\General Brain Anatomy\L3dapiphalloidinsynorf_2_composite.png",
        "view": {"zoom": 1.7, "x": 0.5, "y": 0.7, "rotate": 30}, 
        "scale_bar": {"full_px": 61, "orig_um": 50, "new_um": 50, "color": "white", "thickness": 12, "number": False},
        "fit": "contain",
        "background": "black",
        "label_color": "white",
        "annotations": [
            {"text": "CB", "x": 0.46, "y": 0.03, "size": 70, "color": "white"},
            #{"text": "Brain", "x": 0.39, "y": 0.08, "size": 80, "color": "white"},
            {"text": "SEG", "x": 0.63, "y": 0.81, "size": 70, "color": "white"},
            {"text": "DC", "x": 0.68, "y": 0.42, "size": 70, "color": "white"},
            {"text": "TC", "x": 0.68, "y": 0.57, "size": 70, "color": "white"},
            {"text": "*", "x": 0.6, "y": 0.54, "size": 170, "color": "white"},
            {"text": "◦", "x": 0.35, "y": 0.35, "size": 170, "color": "white"},
            {"text": "◦", "x": 0.6, "y": 0.35, "size": 170, "color": "white"},
            {"text": "Synorf", "x": 0.05, "y": 0.88, "size": 70, "color": "cyan"},
            {"text": "Phalloidin", "x": 0.05, "y": 0.79, "size": 70, "color": "red"},
            {"text": "DAPI", "x": 0.05, "y": 0.70, "size": 70, "color": "blue"},
                ],
    },
    "B": {
        "path": r"D:\larval chemosensory\central brain cartoon_smaller copy.png",
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label_color": "black",

    },
    "F": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\20250630_2\20250630_2\P22_l4_ir25a_larvalhead\MAX_Experiment-1428_TCC.png",
        "view": {"zoom": 4.5, "x": 0.48, "y": 0.52, "rotate": 240},
        "label_color": "white",
        "scale_bar": {"full_px": 269, "orig_um": 50, "new_um": 25, "color": "white", "thickness": 12, "number": False},
        "annotations": [
            {"text": "TCC", "x": 0.40, "y": 0.72, "size": 70, "color": "white"},
            #{"text": "Mandibles", "x": 0.30, "y": 0.15, "size": 80, "color": "white"},
            {"text": "CB", "x": 0.42, "y": 0.05, "size": 90, "color": "white"},
            #{"text": "Brain", "x": 0.40, "y": 0.55, "size": 80, "color": "white"},
            #{"text": "Antenna", "x": 0.2, "y": 0.29, "size": 70, "color": "white"},
            #{"text": "Anti-GFP", "x": 0.05, "y": 0.88, "size": 70, "color": "white"},
        ],
        #"border": {"color": "#B85C4F", "thickness": 12},
    },

    "D": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\20250630_2\20250630_2\P22_l4_ir25a_larvalhead\MAX_Experiment-1428_Deuto.png",
        "view": {"zoom": 4.4, "x": 0.48, "y": 0.42, "rotate": 237},
        "label_color": "white",
        "scale_bar": {"full_px": 269, "orig_um": 50, "new_um": 25, "color": "white", "thickness": 12, "number": False},
        "annotations": [
            {"text": "DC", "x": 0.42, "y": 0.45, "size": 80, "color": "white"},
             {"text": "CB", "x": 0.42, "y": 0.05, "size": 90, "color": "white"},
              #{"text": "Brain", "x": 0.41, "y": 0.35, "size": 80, "color": "white"},
              #{"text": "Towards Mouthparts", "x": 0.23, "y": 0.05, "size": 60, "color": "white"},
            #{"text": "GCaMP", "x": 0.05, "y": 0.88, "size": 70, "color": "white"},
        ],
        #"border": {"color": "#B85C4F", "thickness": 12},
    },

    "E": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\20250630_2\20250630_2\P22_l4_ir25a_larvalhead\MAX_Experiment-1428_trito.png",
        "view": {"zoom": 4.2, "x": 0.47, "y": 0.42, "rotate": 240},
        "scale_bar": {"full_px": 269, "orig_um": 50, "new_um": 25, "color": "white", "thickness": 12, "number": False},
        "label_color": "white",
        "annotations": [
            {"text": "*", "x": 0.17, "y": 0.48, "size": 150, "color": "white"},
            {"text": "*", "x": 0.8, "y": 0.48, "size": 150, "color": "white"},
            {"text": "", "x": 0.38, "y": 0.50, "size": 80, "color": "white"},
            {"text": "CB", "x": 0.43, "y": 0.05, "size": 90, "color": "white"},
            #{"text": "Labrum", "x": 0.38, "y": 0.05, "size": 80, "color": "white"},
            {"text": "TC", "x": 0.44, "y": 0.50, "size": 80, "color": "white"},
            {"text": "Antn", "x": 0.16, "y": 0.19, "size": 60, "color": "white"},
            #{"text": "Cells", "x": 0.12, "y": 0.87, "size": 60, "color": "white"},
           
        ],
        #"border": {"color": "#556E9B", "thickness": 12},
        
        },



    "C": {
        "path": r"C:\Users\hejaz\OneDrive - The Rockefeller University\2024-Present\Downloads\20250630_2\20250630_2\P22_l4_ir25a_larvalhead\MAX_Experiment-1428_SEG.png",
        "view": {"zoom": 3, "x": 0.55, "y": 0.45, "rotate": 57},
        "label_color": "white",
        "scale_bar": {"full_px": 269, "orig_um": 50, "new_um": 25, "color": "white", "thickness": 12, "number": False},
        "annotations": [
            {"text": "α-GFP", "x": 0.40, "y": 0.07, "size": 60, "color": "magenta"},
            {"text": "Ir25a>GCaMP", "x": 0.3, "y": 0.15, "size": 60, "color": "magenta"},
            {"text": "*", "x": 0.30, "y": 0.54, "size": 150, "color": "white"},
            {"text": "*", "x": 0.30, "y": 0.68, "size": 150, "color": "white"},
            {"text": "*", "x": 0.57, "y": 0.54, "size": 150, "color": "white"},
            {"text": "*", "x": 0.58, "y": 0.68, "size": 150, "color": "white"},
            {"text": "CEC", "x": 0.37, "y": 0.30, "size": 70, "color": "white"},
            {"text": "SEG", "x": 0.41, "y": 0.83, "size": 70, "color": "white"},
            #{"text": "Anti-GFP", "x": 0.05, "y": 0.80, "size": 70, "color": "#cd5ea4"},
            #{"text": "Ir25a>GcaMP", "x": 0.05, "y": 0.88, "size": 70, "color": "#cd5ea4"},
            #{"text": "Anti-GFP", "x": 0.05, "y": 0.88, "size": 90, "color": "magenta"},
            #{"text": "°", "x": 0.55, "y": 0.62, "size": 120, "color": "white"},
            #{"text": "Anti-GFP", "x": 0.05, "y": 0.88, "size": 70, "color": "white"},
        ],
        #"border": {"color": "#8A9562", "thickness": 12},
        },



}

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = inches_to_px(COL_GAP_IN, DPI)

row_panel_orders = [
    ("A", "B"),
    ("C", "D"),
    ("E", "F"),
]

panel_layout = build_mixed_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px, row_panel_orders,
    row_height_ratios=(1, 1, 1),
    row_width_fracs=(1.0, 1.0, 1.0),
)


def close_internal_2col_gaps_small_space(layout, row_panel_orders, min_gap_px=6):
    adjusted = dict(layout)

    for row_ids in row_panel_orders:
        left_id, right_id = row_ids
        lx, ly, lw, lh = adjusted[left_id]
        rx, ry, rw, rh = adjusted[right_id]
        gap = rx - (lx + lw)
        if gap > min_gap_px:
            close_px = gap - min_gap_px
            left_extra = close_px // 2
            right_extra = close_px - left_extra
            adjusted[left_id] = (lx, ly, lw + left_extra, lh)
            adjusted[right_id] = (rx - right_extra, ry, rw + right_extra, rh)

    for top_row, bottom_row in zip(row_panel_orders, row_panel_orders[1:]):
        top_bottom = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in top_row)
        bottom_top = min(adjusted[panel_id][1] for panel_id in bottom_row)
        gap = bottom_top - top_bottom
        if gap > min_gap_px:
            close_px = gap - min_gap_px
            top_extra = close_px // 2
            bottom_extra = close_px - top_extra
            for panel_id in top_row:
                x, y, w, h = adjusted[panel_id]
                adjusted[panel_id] = (x, y, w, h + top_extra)
            for panel_id in bottom_row:
                x, y, w, h = adjusted[panel_id]
                adjusted[panel_id] = (x, y - bottom_extra, w, h + bottom_extra)

    return adjusted


panel_layout = close_internal_2col_gaps_small_space(panel_layout, row_panel_orders, min_gap_px=6)

canvas = render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels, dpi=DPI,
)

def add_panel_image_annotation(canvas, panel_layout, panel_id, cfg):
    x0, y0, pw, ph = panel_layout[panel_id]
    ann_w = int(round(pw * cfg.get("w_frac", 0.22)))
    ann_h = int(round(ph * cfg.get("h_frac", 0.22)))

    annotation = Image.open(cfg["path"]).convert("RGBA")
    img_w, img_h = annotation.size
    scale = min(ann_w / img_w, ann_h / img_h)
    resized_w = max(1, int(round(img_w * scale)))
    resized_h = max(1, int(round(img_h * scale)))
    annotation = annotation.resize((resized_w, resized_h), Image.Resampling.LANCZOS)

    right_offset_px = int(round(pw * cfg.get("right_offset_frac", 0.03)))
    bottom_offset_px = int(round(ph * cfg.get("bottom_offset_frac", 0.03)))
    ann_x = x0 + pw - resized_w - right_offset_px
    ann_y = y0 + ph - resized_h - bottom_offset_px
    canvas.paste(annotation, (ann_x, ann_y), annotation)

    border_color = cfg.get("border_color")
    border_thickness = int(cfg.get("border_thickness", 0))
    if border_color and border_thickness > 0:
        draw = ImageDraw.Draw(canvas)
        for i in range(border_thickness):
            draw.rectangle(
                [ann_x + i, ann_y + i, ann_x + resized_w - 1 - i, ann_y + resized_h - 1 - i],
                outline=border_color,
            )
    return canvas


seg_compass_annotation = {
    "path": r"D:\larval chemosensory\SEG_compass.png",
    "w_frac": 0.24,
    "h_frac": 0.24,
    "right_offset_frac": 0.03,
    "bottom_offset_frac": 0.03,
    "fit": "contain",
    "background": "black",
    "padding_color": "black",
}

seg_compass_annotation = {
    "path": r"D:\larval chemosensory\CB_compass.png",
    "w_frac": 0.24,
    "h_frac": 0.24,
    "right_offset_frac": 0.03,
    "bottom_offset_frac": 0.03,
    "fit": "contain",
    "background": "black",
    "padding_color": "black",
}

for panel_id in ("C"):
    canvas = add_panel_image_annotation(canvas, panel_layout, panel_id, seg_compass_annotation)


for panel_id in ("D", "E", "F"):
    canvas = add_panel_image_annotation(canvas, panel_layout, panel_id, seg_compass_annotation)

draw = ImageDraw.Draw(canvas)
a_x, a_y, a_w, _a_h = panel_layout["A"]
title_font_size = 54
species_font = load_font_safe("ariali.ttf", title_font_size)
title_font = load_font_safe("arial.ttf", title_font_size)
species_text = "Ooceraea biroi"
suffix_text = " 4th instar brain"
species_w, species_h = text_size(draw, species_text, species_font)
suffix_w, suffix_h = text_size(draw, suffix_text, title_font)
title_w = species_w + suffix_w
title_h = max(species_h, suffix_h)
title_x = a_x + (a_w - title_w) // 2
title_y = max(8, a_y - title_h - inches_to_px(0.06, DPI))
#draw.text((title_x, title_y), species_text, fill="black", font=species_font)
#draw.text((title_x + species_w, title_y), suffix_text, fill="black", font=title_font)

canvas = register_figure_canvas('Central Brain Signal', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

# Figure 5: Summary Figure

##  Figure 5: Summary Figure

In [ ]:
panels = {
    "A": {
        "path": r"D:\larval chemosensory\Summaryfigure_wholecartoon.png",
        "view": {"zoom": 1, "x": 0.5, "y": 0.1},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label_color": "black",
                "label": "",
    },
}

panel_layout = {
    "A": (0, 0, fig_w_px, int(fig_h_px * 0.75)),
}

canvas = render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels, dpi=DPI,
)
canvas = register_figure_canvas('Summary Figure', canvas, panel_configs=panels, panel_layout=panel_layout)
display(canvas)

# Supplements


## Figure S1: L1 SEM Developmental


In [ ]:
thickness_border = 14
scale_bar_thickness = 12

dome_color = "#0093A9"
papilla_color = "#A64382"
rod_color = "#DDAA00"

panels = {
    "A": {
        "path": r"D:\larval chemosensory\22-03_Daniel_L1H_0001.tif",
        "view": {"zoom": 1.5, "x": 0.7, "y": 0.1},
        "crop_bottom_px": 270,
        "label_color": "white",
        "scale_bar": {"full_px": 475, "orig_um": 20, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, 'number': False},
    },
    "B": {
        #"path": r"D:\larval chemosensory\22-03_Daniel_L2H_0004.tif",
        "view": {"zoom": 1.0, "x": 0.66, "y": 0.9, "rotate": -15},
        "label_color": "white",
    },
    "G": {
        "path": r"D:\larval chemosensory\22-03_Daniel_L1H_0001.tif",
        "view": {"zoom": 11.0, "x": 0.44, "y": 0.275},
                "scale_bar": {"full_px": 475, "orig_um": 20, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False},

        "label_color": "white",
        #"scale_bar": {"full_px": 1002, "orig_um": 5, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            #{"text": "Papillae", "x": 0.10, "y": 0.80, "color": "white", "size": 70},
        ],
        "border": {"color": LABIUM_BORDER_COLOR, "thickness": thickness_border},
    },
    "H": {
        "path": r"D:\larval chemosensory\22-03_Daniel_L1H_0001.tif",
        "view": {"zoom": 17.0, "x": 0.66, "y": 0.33},
                "scale_bar": {"full_px": 475, "orig_um": 20, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False},

        "label_color": "white",
        #"scale_bar": {"full_px": 620, "orig_um": 2, "new_um": 4, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            #{"text": "Rod", "x": 0.10, "y": 0.80, "color": "white", "size": 70},
        ],
        "border": {"color": LABIUM_BORDER_COLOR, "thickness": thickness_border},
    },
    "E": {
        "path": r"D:\larval chemosensory\22-03_Daniel_L1H_0001.tif",
        "view": {"zoom": 8.0, "x": 0.26, "y": 0.20},   
        "label_color": "white",
        "scale_bar": {"full_px": 475, "orig_um": 20, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False},
        "annotations": [
            #{"text": "Dome", "x": 0.10, "y": 0.80, "color": "white", "size": 70},
        ],
     "border": {"color": MAXILLA_BORDER_COLOR, "thickness": thickness_border},
    },
    "F": {
        "path": r"D:\larval chemosensory\22-03_Daniel_L1H_0001.tif",
        "view": {"zoom": 8.0, "x": 0.83, "y": 0.12},   "label_color": "white",
        "crop_bottom_px": 260,
        "scale_bar": {"full_px": 475, "orig_um": 20, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False},

        "label_color": "black",
     "border": {"color": MAXILLA_BORDER_COLOR, "thickness": thickness_border},

    },
    "D": {
        "path": r"D:\larval chemosensory\22-03_Daniel_L1H_0001.tif",
        "view": {"zoom": 6.0, "x": 0.54, "y": 0.12},    "crop_bottom_px": 250,
        "label_color": "white",
        "border": {"color": LABRUM_BORDER_COLOR, "thickness": thickness_border},
        "scale_bar": {"full_px": 475, "orig_um": 20, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, 'number': False},

    },
    "C": {
        "path": r"D:\larval chemosensory\22-03_Daniel_L2V_0003.tif",
        "view": {"zoom": 10.0, "x": 0.61, "y": 0.315},
        "label_color": "white",
        "border": {"color": FRONS_BORDER_COLOR, "thickness": thickness_border},
        "scale_bar": {"full_px": 865, "orig_um": 50, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, 'number': False},


    },
    "I": {
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5},
        "crop_bottom_px": 255,
        "label_color": "white",
        "border": {"color": MAXILLA_BORDER_COLOR, "thickness": thickness_border},
        "scale_bar": {"full_px": 712, "orig_um": 10, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "▼", "x": 0.67, "y": 0.685, "color": papilla_color, "size": 70},
        ],
    },
    "J": {
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5},
        "crop_bottom_px": 255,
        "label_color": "white",
        "scale_bar": {"full_px": 474, "orig_um": 2, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "▼", "x": 0.50, "y": 0.5, "color": papilla_color, "size": 70},
            {"text": "▼", "x": 0.34, "y": 0.6, "color": dome_color, "size": 70},
            {"text": "▼", "x": 0.4, "y": 0.3, "color": dome_color, "size": 70},
            {"text": "▼", "x": 0.65, "y": 0.14, "color": rod_color, "size": 70},
        ],
    },
    "K": {
        "view": {"zoom": 3.5, "x": 0.75, "y": 0.85},
        "crop_bottom_px": 180,
        "label_color": "white",
        "scale_bar": {"full_px": 520, "orig_um": 10, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, 'number': False,},
        "annotations": [
            {"text": "▼", "x": 0.40, "y": 0.40, "color": papilla_color, "size": 70},
        ],
    },
}

for panel_id, label in {
    "A": "A",
    "C": "B",
    "D": "C",
    "E": "D",
    "F": "E",
    "G": "F",
    "H": "G",
}.items():
    panels[panel_id]["label"] = label

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = inches_to_px(COL_GAP_IN, DPI)
stack_gap_px = col_gap_px

# Use a dedicated landscape canvas for the two-row SEM layout.
figure1_l1_fig_w_px = fig_w_px
figure1_l1_fig_h_px = int(round(figure1_l1_fig_w_px * 0.80))
content_x = ml
content_y = mt
content_w = figure1_l1_fig_w_px - ml - mr
content_h = figure1_l1_fig_h_px - mt - mb

# Row 1 is taller: A is at left and B/C/D form a vertical stack at right.
# Row 2 contains E/F/G at equal widths.
available_h = content_h - row_gap_px
row1_h = int(round(available_h * 0.62))
row2_h = available_h - row1_h
row2_y = content_y + row1_h + row_gap_px

available_row1_w = content_w - col_gap_px
a_w = int(round(available_row1_w * 0.68))
bcd_w = available_row1_w - a_w
bcd_x = content_x + a_w + col_gap_px

available_stack_h = row1_h - 2 * stack_gap_px
b_h = available_stack_h // 3
c_h = available_stack_h // 3
d_h = available_stack_h - b_h - c_h

available_row2_w = content_w - 2 * col_gap_px
e_w = available_row2_w // 3
f_w = available_row2_w // 3
g_w = available_row2_w - e_w - f_w

# Configuration keys C/D/E display as B/C/D; F/G/H display as E/F/G.
panel_layout = {
    "A": (content_x, content_y, a_w, row1_h),
    "C": (bcd_x, content_y, bcd_w, b_h),
    "D": (bcd_x, content_y + b_h + stack_gap_px, bcd_w, c_h),
    "E": (bcd_x, content_y + b_h + stack_gap_px + c_h + stack_gap_px, bcd_w, d_h),
    "F": (content_x, row2_y, e_w, row2_h),
    "G": (content_x + e_w + col_gap_px, row2_y, f_w, row2_h),
    "H": (content_x + e_w + col_gap_px + f_w + col_gap_px, row2_y, g_w, row2_h),
}

canvas = render_artboard(
    figure1_l1_fig_w_px,
    figure1_l1_fig_h_px,
    panel_layout,
    panels,
    dpi=DPI,
    scale_font_size=60,
    axis_font_size=60,
)
canvas = register_figure_canvas('L1 SEM Developmental', canvas, panel_configs=panels, panel_layout=panel_layout)
display(canvas)

## Figure S11: Orco GCaMP Validation


In [ ]:
border_thickness = 14
scale_bar_thickness = 12

panels = {
    "A": {  
        "path": r"D:\larval chemosensory\MAX_CompositeOrco_GCAMP_large.png",
        "view": {"zoom": 1.6, "x": 0.35, "y": 0.0},
        "crop_bottom_px": 50,
        "label_color": "white",
        #"border": {"color": "#000000", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            {"text": "α-Orco", "x": 0.05, "y": 0.74, "size": 65, "color": "magenta"},
            {"text": "α-GFP", "x": 0.05, "y": 0.86, "size": 65, "color": "cyan"},
            {"text": "Labium", "x": 0.34, "y": 0.15, "size": 80, "color": "white"},
            #{"text": "Maxilla", "x": 0.4, "y": 0.7, "size": 100, "color": "white"},
        ],
    },

    "B": {
        "path": r"D:\larval chemosensory\MAX_Composite_ Maxilla_full.png",
        "view": {"zoom": 2.3, "x": 0.15, "y": 0.75},
        "crop_bottom_px": 50,
        "label_color": "white",
        #"border": {"color": "#000000", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            #{"text": "Orco", "x": 0.05, "y": 0.78, "size": 80, "color": "magenta"},
            #{"text": "Anti-GFP", "x": 0.05, "y": 0.88, "size": 80, "color": "cyan"},
            #{"text": "Labium", "x": 0.33, "y": 0.1, "size": 100, "color": "white"},
            {"text": " R Maxilla", "x": 0.10, "y": 0.8, "size": 70, "color": "white"},
        ],
    },

    "C": {
        "path": r"D:\larval chemosensory\MAX_Composite_ Maxilla_full.png",
        "view": {"zoom": 2.3, "x": 0.85, "y": 0.60},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "label_color": "white",
        "annotations": [
            {"text": "*", "x": 0.72, "y": 0.47, "size": 80, "color": "white"},
            {"text": "°", "x": 0.44, "y": 0.24, "size": 80, "color": "white"},
            #{"text": "Orco", "x": 0.05, "y": 0.78, "size": 80, "color": "magenta"},
            #{"text": "Anti-GFP", "x": 0.05, "y": 0.88, "size": 80, "color": "cyan"},
            #{"text": "Labium", "x": 0.33, "y": 0.1, "size": 100, "color": "white"},
            {"text": "L Maxilla", "x": 0.15, "y": 0.8, "size": 70, "color": "white"},
        ],
    },

    "D": {
        "path": r"D:\larval chemosensory\MAX_Composite-1-6.png",
        "view": {"zoom": 2.5, "x": 0.4, "y": 0.1},
        "label_color": "white",
        #"border": {"color": "#8A9562", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        # "annotations": [
        #     {"text": "Maxillary", "x": 0.08, "y": 0.8, "size": 60, "color": "#8A9562"},
        #     {"text": "Palp", "x": 0.29, "y": 0.9, "size": 60, "color": "#8A9562"},
        # ],
    },

    "E": {
        "path": r"D:\larval chemosensory\MAX_Composite-6-12.png",
        "view": {"zoom": 2.5, "x": 0.4, "y": 0.13},
        "label_color": "white",
        #"border": {"color": "#d7c7a0", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        # "annotations": [
        #     {"text": "Labial", "x": 0.22, "y": 0.8, "size": 60, "color": "#d7c7a0"},
        #     {"text": "Palp", "x": 0.29, "y": 0.9, "size": 60, "color": "#d7c7a0"},
        # ],
    },

    "F": {
        "path": r"D:\larval chemosensory\MAX_Composite-16-23.png",
        "view": {"zoom": 2.5, "x": 0.4, "y": 0.1},
        "label_color": "white",
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        # "annotations": [
        #     {"text": "Labium", "x": 0.34, "y": 0.38, "size": 55, "color": "black"},
        #     #{"text": "Maxilla", "x": 0.0, "y": 0.58, "size": 55, "color": "black"},
        #     {"text": "Front View", "x": 0.22, "y": 0.88, "size": 65, "color": "black"},
        # ],
    },

    "G": {
        "path": r"D:\larval chemosensory\MAX_Composite-25-26.png",
        "view": {"zoom": 6.0, "x": 0.57, "y": 0.205, "rotate": 0},
        "label_color": "white",
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        # "annotations": [
        #     {"text": "Labium", "x": 0.4, "y": 0.38, "size": 55, "color": "black"},
        #     {"text": "Maxilla", "x": -0.14, "y": 0.28, "size": 55, "color": "black"},
        #     {"text": "Back View", "x": 0.32, "y": 0.88, "size": 65, "color": "black"},
        # ],
    },

    "H": {
        "path": r"D:\larval chemosensory\MAX_Composite-max-left-19-26.png",
        "view": {"zoom": 3.7, "x": 0.13, "y": 0.66},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "label_color": "white",
    },

    "I": {
        "path": r"D:\larval chemosensory\MAX_Composite-max-left-26-34.png",
        "view": {"zoom": 3.7, "x": 0.13, "y": 0.64},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "label_color": "white",
        "label": "J",
    },

    "J": {
        "path": r"D:\larval chemosensory\MAX_Composite-max-left-19-26.png",
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "view": {"zoom": 3.2, "x": 0.91, "y": 0.58},
        "label_color": "white",
        "label": "I",       
    },

    "K": {
        "path": r"D:\larval chemosensory\MAX_Composite-max-left-26-34.png",
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "view": {"zoom": 3.2, "x": 0.91, "y": 0.55},
        "label_color": "white",
   
    },

    "L": {
        "path": r"D:\larval chemosensory\MAX_Composite_Orco_GCAMP_SEG.png",
        "view": {"zoom": 3.0, "x": 0.36, "y": 0.60, "rotate": -10},
        "label_color": "white",
        #"border": {"color": "#8A9562", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            #{"text": "Orco", "x": 0.05, "y": 0.82, "size": 65, "color": "magenta"},
            #{"text": "Anti-GFP", "x": 0.05, "y": 0.90, "size": 65, "color": "cyan"},
            {"text": "SEG", "x": 0.35, "y": 0.57, "size": 60, "color": "white"},
            {"text": "α-Orco", "x": 0.05, "y": 0.74, "size": 60, "color": "magenta"},
            {"text": "α-GFP", "x": 0.05, "y": 0.86, "size": 60, "color": "cyan"},
        ],
    },

    "M": {
        "path": r"D:\larval chemosensory\MAX_Composite-OrcoGCAMP_deuto.png",
        "view": {"zoom": 3.0, "x": 0.37, "y": 0.48, "rotate": -10},
        "label_color": "white",
        #"border": {"color": "#d7c7a0", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            {"text": "DC", "x": 0.38, "y": 0.52, "size": 60, "color": "white"},
            {"text": "CB", "x": 0.35, "y": 0.76, "size": 90, "color": "white"},
            #{"text": "Brain", "x": 0.38, "y": 0.35, "size": 60, "color": "white"},
        ],
    
    },

    "O": {
        "path": r"D:\larval chemosensory\MAX_Composite-OrcoGCAMP_trio.png",
        "view": {"zoom": 3.5, "x": 0.37, "y": 0.51, "rotate": -10},
        "label_color": "white",
        #"border": {"color": "#d7c7a0", "thickness": border_thickness},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
             {"text": "TC", "x": 0.38, "y": 0.12, "size": 60, "color": "white"},
            {"text": "CB", "x": 0.35, "y": 0.76, "size": 90, "color": "white"},
            #{"text": "Brain", "x": 0.38, "y": 0.56, "size": 60, "color": "white"},
            {"text": "*", "x": 0.22, "y": 0.22, "size": 120, "color": "white"},
            {"text": "*", "x": 0.62, "y": 0.22, "size": 120, "color": "white"},
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 80, "color": "cyan"},
        ],
    },

    "N": {
        "path": r"D:\larval chemosensory\MAX_Composite-OrcoGCAMP_trio.png",
         "view": {"zoom": 2.7, "x": 0.37, "y": 0.05, "rotate": -10},
        "scale_bar": {"full_px": 119, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "label_color": "white",
        "annotations": [
            {"text": "Labrum", "x": 0.26, "y": 0.3, "size": 60, "color": "white"},
            #{"text": "Central", "x": 0.28, "y": 0.42, "size": 80, "color": "white"},
            #{"text": "Brain", "x": 0.35, "y": 0.55, "size": 80, "color": "white"},
            #{"text": "OrE3", "x": 0.05, "y": 0.85, "size": 80, "color": "cyan"},
        ],
    },
}

import copy

_old_l_panel = copy.deepcopy(panels["L"])
_old_m_panel = copy.deepcopy(panels["M"])
_old_n_panel = copy.deepcopy(panels["N"])
_old_o_panel = copy.deepcopy(panels["O"])

panels["L"] = {
    "path": r"D:\larval chemosensory\central brain cartoon_smaller copy_orco.png",
    "view": {"zoom": 1.0, "x": 0.5, "y": 0.5},
    "fit": "contain",
    "background": "white",
    "padding_color": "white",
    "label_color": "black",
    "annotations": [],
}
panels["M"] = _old_l_panel
panels["N"] = _old_m_panel
panels["O"] = _old_n_panel
panels["P"] = _old_o_panel

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = inches_to_px(COL_GAP_IN, DPI)
figure2_supp_small_gap_px = 6

row_panel_orders = [
    ("A", "B"),
    ("D", "E", "H", "J"),
    ("L", "M", "N"),
]

panel_layout = build_mixed_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, figure2_supp_small_gap_px, figure2_supp_small_gap_px,
    row_panel_orders,
    row_height_ratios=(0.3, 0.35, 0.45),
    row_width_fracs=(1.0, 1.0, 1.0),
)

# Split the original B slot into side-by-side B and C panels.
x_b, y_b, w_b, h_b = panel_layout["B"]
b_w = (w_b - figure2_supp_small_gap_px) // 2
panel_layout["B"] = (x_b, y_b, b_w, h_b)
panel_layout["C"] = (x_b + b_w + figure2_supp_small_gap_px, y_b, w_b - b_w - figure2_supp_small_gap_px, h_b)

# Split row-2 columns into stacked panels with internal gaps.
stack_gap_px = figure2_supp_small_gap_px

x_d, y_d, w_d, h_d = panel_layout["D"]
d_h = (h_d - stack_gap_px) // 2
panel_layout["D"] = (x_d, y_d, w_d, d_h)
panel_layout["F"] = (x_d, y_d + d_h + stack_gap_px, w_d, h_d - d_h - stack_gap_px)

x_e, y_e, w_e, h_e = panel_layout["E"]
e_h = (h_e - stack_gap_px) // 2
panel_layout["E"] = (x_e, y_e, w_e, e_h)
panel_layout["G"] = (x_e, y_e + e_h + stack_gap_px, w_e, h_e - e_h - stack_gap_px)

x_h, y_h, w_h, h_h = panel_layout["H"]
h_half = (h_h - stack_gap_px) // 2
panel_layout["H"] = (x_h, y_h, w_h, h_half)
panel_layout["I"] = (x_h, y_h + h_half + stack_gap_px, w_h, h_h - h_half - stack_gap_px)

x_i, y_i, w_i, h_i = panel_layout["J"]
i_half = (h_i - stack_gap_px) // 2
panel_layout["J"] = (x_i, y_i, w_i, i_half)
panel_layout["K"] = (x_i, y_i + i_half + stack_gap_px, w_i, h_i - i_half - stack_gap_px)

# Recut the last row: L is 50% wide, M/N 25%, and O/P 25%.
x_l, y_l, w_l, h_l = panel_layout["L"]
x_m, y_m, w_m, h_m = panel_layout["M"]
x_n, y_n, w_n, h_n = panel_layout["N"]
row_left = x_l
row_top = min(y_l, y_m, y_n)
row_right = max(x_l + w_l, x_m + w_m, x_n + w_n)
row_h = max(y_l + h_l, y_m + h_m, y_n + h_n) - row_top
available_w = (row_right - row_left) - 2 * stack_gap_px
l_w = int(round(available_w * 0.50))
mn_w = int(round(available_w * 0.25))
op_w = available_w - l_w - mn_w
stack_h = (row_h - stack_gap_px) // 2

panel_layout["L"] = (row_left, row_top, l_w, row_h)
mn_x = row_left + l_w + stack_gap_px
op_x = mn_x + mn_w + stack_gap_px
panel_layout["M"] = (mn_x, row_top, mn_w, stack_h)
panel_layout["N"] = (op_x, row_top, op_w, stack_h)
panel_layout["O"] = (mn_x, row_top + stack_h + stack_gap_px, mn_w, stack_h)
panel_layout["P"] = (op_x, row_top + stack_h + stack_gap_px, op_w, stack_h)

figure2_supp_border_thickness = 14


def add_group_border_inside(canvas, layout, panel_ids, color="black", thickness=4):
    boxes = [layout[panel_id] for panel_id in panel_ids]
    x0 = min(box[0] for box in boxes)
    y0 = min(box[1] for box in boxes)
    x1 = max(box[0] + box[2] for box in boxes)
    y1 = max(box[1] + box[3] for box in boxes)
    draw = ImageDraw.Draw(canvas)
    for i in range(int(thickness)):
        draw.rectangle([x0 + i, y0 + i, x1 - 1 - i, y1 - 1 - i], outline=color)
    return canvas


figure2_supp_group_borders = [
    (("A", "D", "E", "F", "G"), LABRUM_BORDER_COLOR),
    (("B", "C", "H", "I", "J", "K"), MAXILLA_BORDER_COLOR),
    (("O",), LABRUM_BORDER_COLOR),
]

for panel_ids, _border_color in figure2_supp_group_borders:
    for panel_id in panel_ids:
        panels[panel_id].pop("border", None)

figure2_supp_label_pad_px = figure2_supp_border_thickness + 8
for panel_ids, _border_color in figure2_supp_group_borders:
    for panel_id in panel_ids:
        panels[panel_id]["label_pad_px"] = figure2_supp_label_pad_px

canvas = render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels, dpi=DPI, scale_font_size=60
)

for panel_ids, border_color in figure2_supp_group_borders:
    canvas = add_group_border_inside(
        canvas,
        panel_layout,
        panel_ids,
        color=border_color,
        thickness=figure2_supp_border_thickness,
    )

def add_figure2_supp_lower_right_insets(
    canvas,
    panel_layout,
    inset_cfgs,
    inset_w_frac=0.45,
    inset_h_frac=0.28,
    right_offset_frac=0.0,
    bottom_offset_frac=0.0,
    border_color="black",
    border_thickness=4,
):
    draw = ImageDraw.Draw(canvas)
    for panel_id, inset_cfg in inset_cfgs.items():
        x0, y0, pw, ph = panel_layout[panel_id]
        inset_w = int(round(pw * inset_cfg.get("w_frac", inset_w_frac)))
        inset_h = int(round(ph * inset_cfg.get("h_frac", inset_h_frac)))
        inset = paste_with_view(
            inset_cfg["path"],
            inset_w,
            inset_h,
            view=inset_cfg.get("view"),
            fit=inset_cfg.get("fit", "contain"),
            background=inset_cfg.get("background", "white"),
            padding_color=inset_cfg.get("padding_color", "white"),
        )
        offset_px = int(round(pw * inset_cfg.get("right_offset_frac", right_offset_frac)))
        bottom_offset_px = int(round(ph * inset_cfg.get("bottom_offset_frac", bottom_offset_frac)))
        inset_x = x0 + pw - inset_w - offset_px
        inset_y = y0 + ph - inset_h - bottom_offset_px
        canvas.paste(inset, (inset_x, inset_y))

        inset_border_color = inset_cfg.get("border_color", border_color)
        inset_border_thickness = int(inset_cfg.get("border_thickness", border_thickness))
        if inset_border_color and inset_border_thickness > 0:
            for i in range(inset_border_thickness):
                draw.rectangle(
                    [inset_x + i, inset_y + i, inset_x + inset_w - 1 - i, inset_y + inset_h - 1 - i],
                    outline=inset_border_color,
                )

    return canvas


figure2_supp_inset_paths = {
    "L": r"D:\larval chemosensory\central brain cartoon_smaller copy_greenSEG.png",
    "M": r"D:\larval chemosensory\central brain cartoon_smaller copy_greenDC.png",
    "O": r"D:\larval chemosensory\central brain cartoon_smaller copy_greenTC.png",
}

figure2_supp_bordered_insets = {
    "L": {
        "path": figure2_supp_inset_paths["L"] or panels["L"]["path"],
        "view": {"zoom": 1, "x": 0.5, "y": 0.5},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "border_color": "black",
        "border_thickness": 4,
    },
    "M": {
        "path": figure2_supp_inset_paths["M"] or panels["M"]["path"],
        "view": {"zoom": 1, "x": 0.5, "y": 0.5},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "border_color": "black",
        "border_thickness": 4,
    },
    "O": {
        "path": figure2_supp_inset_paths["O"] or panels["O"]["path"],
        "view": {"zoom": 1, "x": 0.5, "y": 0.5},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "w_frac": 0.32,
        "h_frac": 0.32,
        "border_color": "black",
        "border_thickness": 4,
    },
}

# Insets removed in Draft 2; cartoon panels are now full panels.

canvas = register_figure_canvas('Orco GCaMP Validation', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure S6: Ir25a GCaMP Validation


In [ ]:
border_thickness = 12
scale_bar_thickness = 12
figure2_ir25a_visible_group_gap_px = 4
# Detail groups need room for their 12 px borders, which are drawn outward.
figure2_ir25a_detail_gap_px = (
    2 * border_thickness + figure2_ir25a_visible_group_gap_px
)
# A-B use a narrow divider independent of the thicker anatomical borders.
figure2_ir25a_overview_gap_px = 12
figure2_ir25a_overview_border_thickness_px = 2
figure2_ir25a_subpanel_gap_px = 4

# Single source of truth for panels A-G.
# Overview groups A-B contain one panel; detail groups C-F contain three channels.
ir25a_gcamp_groups = [
    {
        "label": "B",
        "border_color": "white",
        "view": {"zoom": 2.0, "x": 0.5, "y": 0.20, "rotate": 233},
        "annotations": [
            {"text": "Labium", "x": 0.42, "y": 0.03, "size": 60, "color": "white"},
            {"text": "CB", "x": 0.42, "y": 0.82, "size": 105, "color": "white"},
            {"text": "Maxilla", "x": 0.38, "y": 0.42, "size": 60, "color": "white"},

        ],
        "scale_bar": {
            "full_px": 760, "orig_um": 100, "new_um": 100,
            "color": "white", "thickness": scale_bar_thickness, "number": False,
        },
        "channels": [
            ("B1", "", r"D:\larval chemosensory\-00001-04(7)\MIP z218-318\Full image - MIP z218-318\-00001-04_7_mip_z218-318_full_ch0_Ir25a_ch1_Red.png", "white"),  # Add the first overview image path here.
        ],
    },
    {
        "label": "A",
        "border_color": "white",
        "view": {"zoom": 1.7, "x": 0.62, "y": 0.42, "rotate": 197},
        "annotations": [
            {"text": "CB", "x": 0.44, "y": 0.80, "size": 105, "color": "white"},
            {"text": "Labrum", "x": 0.42, "y": 0.03, "size": 60, "color": "white"},
            #{"text": "Antn", "x": 0.07, "y": 0.60, "size": 60, "color": "white"},
            {"text": "Antn", "x": 0.72, "y": 0.72, "size": 60, "color": "white"},
            {"text": "Ir25a RNA", "x": 0.03, "y": 0.88, "size": 60, "color": "cyan"},
            {"text": "GCaMP", "x": 0.03, "y": 0.93, "size": 60, "color": "yellow"},
        ],
        "scale_bar": {
            "full_px": 760, "orig_um": 100, "new_um": 100,
            "color": "white", "thickness": scale_bar_thickness, "number": False,
        },
        "channels": [
            ("A1", "", r"D:\larval chemosensory\-00001-04(2)\MIP z143-247\Full image - MIP z143-247\-00001-04_2_mip_z143-247_full_ch0_Ir25a_ch1_Red.png", "white"),  # Add the second overview image path here.
        ],
    },
    {
        "label": "C",
        "border_color": FRONS_BORDER_COLOR,
        "view": {"zoom": 1.4, "x": 0.5, "y": 0.5, "rotate": 197},
        "annotations": [
          
        ],
        "scale_bar": {
            "full_px": 3040, "orig_um": 100, "new_um": 10,
            "color": "white", "thickness": scale_bar_thickness, "number": False,
        },
        "channels": [
            ("C1","", r"D:\larval chemosensory\-00001-04(2)\ROI 01 - antenna-magenta - z157 of 477\-00001-04_2_z156_roi01_combined_ch0_Ir25a_ch1_Red_ch3_DAPI.png","white",),
            ("C2", "", r"D:\larval chemosensory\-00001-04(2)\ROI 01 - antenna-magenta - z157 of 477\-00001-04_2_z156_roi01_ch0_Ir25a_ch3_DAPI.png", "white"),
            ("C3", "", r"D:\larval chemosensory\-00001-04(2)\ROI 01 - antenna-magenta - z157 of 477\-00001-04_2_z156_roi01_ch1_Red_ch3_DAPI.png", "white"),
        ],
    },
    {
        "label": "D",
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 1.4, "x": 0.55, "y": 0.5, "rotate": 197},
        "annotations": [],
        "scale_bar": {
            "full_px": 3040, "orig_um": 100, "new_um": 10,
            "color": "white", "thickness": scale_bar_thickness, "number": False,
        },
        "channels": [
            ("D1", "", r"D:\larval chemosensory\-00001-04(2)\ROI 01 - labrum-magenta - z225 of 477\-00001-04_2_z224_roi01_combined_ch0_Ir25a_ch1_Red_ch3_DAPI.png", "white"),
            ("D2", "", r"D:\larval chemosensory\-00001-04(2)\ROI 01 - labrum-magenta - z225 of 477\-00001-04_2_z224_roi01_ch0_Ir25a_ch3_DAPI.png", "white"),
            ("D3", "", r"D:\larval chemosensory\-00001-04(2)\ROI 01 - labrum-magenta - z225 of 477\-00001-04_2_z224_roi01_ch1_Red_ch3_DAPI.png", "white"),
        ],
    },
    {
        "label": "E",
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 1.4, "x": 0.6, "y": 0.5, "rotate": 233},
        "annotations": [],
        "scale_bar": {
             "full_px": 3040, "orig_um": 100, "new_um": 10,
            "color": "white", "thickness": scale_bar_thickness, "number": False,
        },
        "channels": [
            ("E1", "", r"D:\larval chemosensory\-00001-04(7)\ROI 01 - maxilla-magenta-new - z265 of 477\-00001-04_7_z264_roi01_combined_ch0_Ir25a_ch1_Red_ch3_DAPI.png" , "white"),
            ("E2", "", r"D:\larval chemosensory\-00001-04(7)\ROI 01 - maxilla-magenta-new - z265 of 477\-00001-04_7_z264_roi01_ch0_Ir25a_ch3_DAPI.png" , "white"),
            ("E3", "", r"D:\larval chemosensory\-00001-04(7)\ROI 01 - maxilla-magenta-new - z265 of 477\-00001-04_7_z264_roi01_ch1_Red_ch3_DAPI.png" , "white"),
        ],
    },
    {
        "label": "F",
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 1.4, "x": 0.5, "y": 0.5, "rotate": 233},
        "annotations": [],
        "scale_bar": {
             "full_px": 3040, "orig_um": 100, "new_um": 10,
            "color": "white", "thickness": scale_bar_thickness, "number": False,
        },
        "channels": [
            ("F1", "", r"D:\larval chemosensory\-00001-04(7)\ROI 01 - labium-magenta-new - z231 of 477\-00001-04_7_z230_roi01_combined_ch0_Ir25a_ch1_Red_ch3_DAPI.png", "white"),
            ("F2", "", r"D:\larval chemosensory\-00001-04(7)\ROI 01 - labium-magenta-new - z231 of 477\-00001-04_7_z230_roi01_ch0_Ir25a_ch3_DAPI.png", "white"),
            ("F3", "", r"D:\larval chemosensory\-00001-04(7)\ROI 01 - labium-magenta-new - z231 of 477\-00001-04_7_z230_roi01_ch1_Red_ch3_DAPI.png", "white"),
        ],
    },

]

# Overview boxes use normalized panel coordinates (0-1).
# x/y set the upper-left corner; w/h set the box width and height.
# Add as many dictionaries as needed to either list.
figure2_ir25a_overview_boxes = {
    "A1": [
        {"x": 0.13, "y": 0.64, "w": 0.15, "h": 0.22,
        "color": FRONS_BORDER_COLOR, "thickness": 8},
        {"x": 0.34, "y": 0.16, "w": 0.15, "h": 0.22,
        "color": LABRUM_BORDER_COLOR, "thickness": 8},
    ],
    "B1": [
        {"x": 0.63, "y": 0.32, "w": 0.17, "h": 0.23,
        "color": MAXILLA_BORDER_COLOR, "thickness": 8},
        {"x": 0.38, "y": 0.12, "w": 0.15, "h": 0.21,
        "color": LABIUM_BORDER_COLOR, "thickness": 8},
    ],
}

# Annotations here apply only to the named subpanel.
figure2_ir25a_panel_annotations = {
    "C1": [
            {"text": "Ir25a RNA", "x": 0.03, "y": 0.84, "size": 40, "color": "cyan"},
            {"text": "GCaMP", "x": 0.03, "y": 0.92, "size": 40, "color": "yellow"},
            {"text": "DAPI", "x": 0.03, "y": 0.76, "size": 40, "color": "magenta"},
    ],
}

figure2_ir25a_fig_h_px = fig_h_px

content_y = mt
content_h = figure2_ir25a_fig_h_px - mt - mb

# The colored C-F borders extend outward by border_thickness. Position their
# panel boxes inward by that amount so the visible borders are flush with the
# canvas edges. A-B images themselves extend directly to those same edges.
detail_content_x = border_thickness
detail_content_w = fig_w_px - 2 * border_thickness
overview_content_x = 0
overview_content_w = fig_w_px

# The overview row is twice as tall as either detail row, matching the reference.
available_h = content_h - 2 * figure2_ir25a_detail_gap_px
detail_row_h_1 = available_h // 4
detail_row_h_2 = available_h // 4
overview_row_h = available_h - detail_row_h_1 - detail_row_h_2

detail_col_w_left = (
    detail_content_w - figure2_ir25a_detail_gap_px
) // 2
detail_col_w_right = (
    detail_content_w - figure2_ir25a_detail_gap_px - detail_col_w_left
)
detail_right_x = (
    detail_content_x + detail_col_w_left + figure2_ir25a_detail_gap_px
)

overview_col_w_left = (
    overview_content_w - figure2_ir25a_overview_gap_px
) // 2
overview_col_w_right = (
    overview_content_w - figure2_ir25a_overview_gap_px - overview_col_w_left
)
overview_right_x = (
    overview_content_x + overview_col_w_left + figure2_ir25a_overview_gap_px
)

row_1_y = content_y
row_2_y = row_1_y + overview_row_h + figure2_ir25a_detail_gap_px
row_3_y = row_2_y + detail_row_h_1 + figure2_ir25a_detail_gap_px

group_boxes_by_label = {
    "A": (
        overview_content_x, row_1_y, overview_col_w_left, overview_row_h
    ),
    "B": (
        overview_right_x, row_1_y, overview_col_w_right, overview_row_h
    ),
    "C": (
        detail_content_x, row_2_y, detail_col_w_left, detail_row_h_1
    ),
    "D": (
        detail_right_x, row_2_y, detail_col_w_right, detail_row_h_1
    ),
    "E": (
        detail_content_x, row_3_y, detail_col_w_left, detail_row_h_2
    ),
    "F": (
        detail_right_x, row_3_y, detail_col_w_right, detail_row_h_2
    ),
}


def split_group_box(group_box, channel_count, internal_gap_px):
    if channel_count == 1:
        return (group_box,)
    if channel_count != 3:
        raise ValueError("Ir25a GCaMP groups must contain either 1 or 3 channels")

    x0, y0, group_w, group_h = group_box
    available_w = group_w - 2 * internal_gap_px
    widths = [available_w // 3] * 3
    widths[-1] += available_w - sum(widths)
    boxes = []
    x = x0
    for width in widths:
        boxes.append((x, y0, width, group_h))
        x += width + internal_gap_px
    return tuple(boxes)


panel_layout = {}
figure2_ir25a_panels = {}

for group in ir25a_gcamp_groups:
    group_box = group_boxes_by_label[group["label"]]
    sub_boxes = split_group_box(
        group_box,
        len(group["channels"]),
        figure2_ir25a_subpanel_gap_px,
    )

    for channel, sub_box in zip(group["channels"], sub_boxes):
        subpanel_id, channel_name, image_path, channel_color = channel
        annotations = [annotation.copy() for annotation in group.get("annotations", [])]
        annotations.extend(
            annotation.copy()
            for annotation in figure2_ir25a_panel_annotations.get(subpanel_id, [])
        )
        if channel_name:
            annotations.append(
                channel_text_label(
                    channel_name,
                    x=0.95,
                    y=0.85,
                    size=50,
                    anchor="right",
                    color=channel_color,
                )
            )

        panel = {
            "path": image_path,
            "view": group["view"].copy(),
            "label": "",
            "label_color": "white",
            "annotations": annotations,
        }
        for optional_key in ("fit", "background", "padding_color"):
            if optional_key in group:
                panel[optional_key] = group[optional_key]
        if group.get("scale_bar"):
            panel["scale_bar"] = group["scale_bar"].copy()
        if len(group["channels"]) == 3:
            panel["border"] = {"color": "black", "thickness": 2}

        figure2_ir25a_panels[subpanel_id] = panel
        panel_layout[subpanel_id] = sub_box

# Cache finished image tiles and scale calculations across repeated runs.
# File modification times are part of the keys, so edited source images refresh.
import os

figure2_ir25a_tile_cache = globals().get("figure2_ir25a_tile_cache", {})
figure2_ir25a_scale_cache = globals().get("figure2_ir25a_scale_cache", {})


def _figure2_ir25a_freeze(value):
    if isinstance(value, dict):
        return tuple(sorted(
            (key, _figure2_ir25a_freeze(item)) for key, item in value.items()
        ))
    if isinstance(value, (list, tuple)):
        return tuple(_figure2_ir25a_freeze(item) for item in value)
    return value


def _figure2_ir25a_source_stamp(img_path):
    try:
        return os.path.getmtime(os.fspath(img_path))
    except OSError:
        return None


_figure2_ir25a_base_paste_with_view = paste_with_view
_figure2_ir25a_base_compute_panel_scale = compute_panel_scale


def _figure2_ir25a_cached_paste_with_view(
    img_path, panel_w, panel_h, view=None, crop_bottom_px=0, rotate=None,
    padding_px=0, padding_color="white", fit="cover", background="white",
):
    key = (
        os.fspath(img_path),
        _figure2_ir25a_source_stamp(img_path),
        int(panel_w),
        int(panel_h),
        _figure2_ir25a_freeze(view),
        int(crop_bottom_px),
        rotate,
        _figure2_ir25a_freeze(padding_px),
        padding_color,
        fit,
        background,
    )
    cached = figure2_ir25a_tile_cache.get(key)
    if cached is not None:
        return cached.copy()

    tile = _figure2_ir25a_base_paste_with_view(
        img_path,
        panel_w,
        panel_h,
        view=view,
        crop_bottom_px=crop_bottom_px,
        rotate=rotate,
        padding_px=padding_px,
        padding_color=padding_color,
        fit=fit,
        background=background,
    )
    if len(figure2_ir25a_tile_cache) >= 32:
        figure2_ir25a_tile_cache.pop(next(iter(figure2_ir25a_tile_cache)))
    figure2_ir25a_tile_cache[key] = tile.copy()
    return tile


def _figure2_ir25a_cached_compute_panel_scale(
    img_path, panel_w, panel_h, view=None, crop_bottom_px=0, rotate=None,
    padding_px=0, padding_color="white",
):
    key = (
        os.fspath(img_path),
        _figure2_ir25a_source_stamp(img_path),
        int(panel_w),
        int(panel_h),
        _figure2_ir25a_freeze(view),
        int(crop_bottom_px),
        rotate,
        _figure2_ir25a_freeze(padding_px),
        padding_color,
    )
    if key not in figure2_ir25a_scale_cache:
        if len(figure2_ir25a_scale_cache) >= 64:
            figure2_ir25a_scale_cache.pop(next(iter(figure2_ir25a_scale_cache)))
        figure2_ir25a_scale_cache[key] = _figure2_ir25a_base_compute_panel_scale(
            img_path,
            panel_w,
            panel_h,
            view=view,
            crop_bottom_px=crop_bottom_px,
            rotate=rotate,
            padding_px=padding_px,
            padding_color=padding_color,
        )
    return figure2_ir25a_scale_cache[key]


# Use cached wrappers only while this figure renders, then restore shared helpers.
paste_with_view = _figure2_ir25a_cached_paste_with_view
compute_panel_scale = _figure2_ir25a_cached_compute_panel_scale
try:
    canvas = render_artboard(
        fig_w_px,
        figure2_ir25a_fig_h_px,
        panel_layout,
        figure2_ir25a_panels,
        dpi=DPI,
        scale_font_size=60,
    )
finally:
    paste_with_view = _figure2_ir25a_base_paste_with_view
    compute_panel_scale = _figure2_ir25a_base_compute_panel_scale

# Draw overview boxes after rendering so they remain crisp and are not cached
# into the underlying image tiles.
for overview_panel_id, boxes in figure2_ir25a_overview_boxes.items():
    for box in boxes:
        canvas = add_panel_box(
            canvas,
            panel_layout,
            overview_panel_id,
            x=box["x"],
            y=box["y"],
            w=box["w"],
            h=box["h"],
            color=box.get("color", "white"),
            thickness=box.get("thickness", 8),
        )

draw = ImageDraw.Draw(canvas)
group_label_font = load_font_safe("arial.ttf", 72)
group_label_pad_px = 8


def draw_border_outside(draw, box, color, thickness):
    """Draw a border entirely outside a panel/group's image bounds."""
    x0, y0, width, height = box
    for offset in range(thickness):
        draw.rectangle(
            [
                x0 - 1 - offset,
                y0 - 1 - offset,
                x0 + width + offset,
                y0 + height + offset,
            ],
            outline=color,
        )


for group in ir25a_gcamp_groups:
    group_id = group["label"]
    group_box = group_boxes_by_label[group_id]
    x0, y0, _group_w, _group_h = group_box
    group_border_thickness = (
        figure2_ir25a_overview_border_thickness_px
        if group_id in {"A", "B"}
        else border_thickness
    )
    draw_border_outside(
        draw,
        group_box,
        group["border_color"],
        group_border_thickness,
    )
    label_x = x0 + group_label_pad_px
    label_y = y0 + group_label_pad_px

    # Mask the blue image mark behind the A label with an opaque black box.
    if group_id == "A":
        label_bbox = draw.textbbox(
            (label_x, label_y), group_id, font=group_label_font
        )
        label_background_pad_px = 8
        draw.rectangle(
            (
                x0 + 1,
                y0 + 1,
                label_bbox[2] + label_background_pad_px,
                label_bbox[3] + label_background_pad_px,
            ),
            fill="black",
        )

    draw.text(
        (label_x, label_y),
        group_id,
        fill="white",
        font=group_label_font,
    )

canvas = register_figure_canvas('Ir25a GCaMP Validation', canvas, panel_configs=figure2_ir25a_panels, panel_layout=panel_layout)

display(canvas)

## Figure S2: RNA seq


In [ ]:
# Combined RNA-seq/RNA-FISH figure shown in the reference image.
figure2_rnaseq_panels = {
    "A": {
        "path": r"D:\larval chemosensory\RNAseq_2026_09.png",  # Add the combined figure path here.
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5, "rotate": 0},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label": "",
        "label_color": "black",
        "annotations": [],
        "border": {"color": "black", "thickness": 0},
    },
}

# Preserve the reference image proportions: 1452 px wide by 1139 px high.
figure2_rnaseq_reference_w = 1452
figure2_rnaseq_reference_h = 1139
figure2_rnaseq_aspect = figure2_rnaseq_reference_w / figure2_rnaseq_reference_h

# Use a dedicated landscape canvas with the same aspect ratio as the image.
# Panel A fills it edge-to-edge, so there is no outer whitespace on any side.
figure2_rnaseq_fig_w_px = fig_w_px
figure2_rnaseq_fig_h_px = int(round(
    figure2_rnaseq_fig_w_px / figure2_rnaseq_aspect
))
figure2_rnaseq_layout = {
    "A": (0, 0, figure2_rnaseq_fig_w_px, figure2_rnaseq_fig_h_px),
}

figure2_rnaseq_canvas = render_artboard(
    figure2_rnaseq_fig_w_px,
    figure2_rnaseq_fig_h_px,
    figure2_rnaseq_layout,
    figure2_rnaseq_panels,
    dpi=DPI,
)

figure2_rnaseq_canvas = register_figure_canvas('RNA seq', figure2_rnaseq_canvas, panel_configs=figure2_rnaseq_panels, panel_layout=figure2_rnaseq_layout)

display(figure2_rnaseq_canvas)

##  Figure S7: Ir75f.3 and Ir317.2 Individual Channels


In [ ]:
separate_channel_scale_bar = {
    "full_px": 251,
    "orig_um": 20,
    "new_um": 15,
}

separate_channel_label_style = {
    "x": 0.92,
    "y": 0.83,
    "size": 60,
    "anchor": "right"
}

separate_channel_groups = [
    {
        "label": "A",
        "border_color": FRONS_BORDER_COLOR,
        "view": {"zoom": 4.0, "x": 0.15, "y": 0.83, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.55, "y": 0.80, "angle": 180, "length": 0.16, "color": "#E69F00", "width": 12},
            #{"type": "arrow", "x": 0.48, "y": 0.42, "angle": 180, "length": 0.16, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("F1", "DAPI", r"D:\20250922\4\antenna\Experiment-1497_antenna_DAPI.png", "white"),
            ("F2", "Ir75f.3", r"D:\20250922\4\antenna\Experiment-1497_antenna_ir75f3.png", "cyan"),
            ("F3", "Ir317.2", r"D:\20250922\4\antenna\Experiment-1497_antenna_ir3172.png", "magenta"),
            ("F4", "Ir25a", r"D:\20250922\4\antenna\Experiment-1497_antenna_ir25a.png", "yellow"),
        ],
    },
    {
        "label": "B",
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 5.0, "x": 0.2, "y": 0.38, "rotate": 0},
        "scale_bar_new_um": 10,
        "annotations": [
            #{"type": "arrow", "x": 0.57, "y": 0.56, "angle": 270, "length": 0.16, "color": "#E69F00", "width": 12},
            #{"type": "arrow", "x": 0.35, "y": 0.52, "angle": 270, "length": 0.16, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("E1", "DAPI", r"D:\20250922\4\labrum2\Experiment-1497_labrum2_DAPI.png", "white"),
            ("E2", "Ir75f.3", r"D:\20250922\4\labrum2\Experiment-1497_labrum2_ir75f3.png", "cyan"),
            ("E3", "Ir317.2", r"D:\20250922\4\labrum2\Experiment-1497_labrum2_ir3172.png", "magenta"),
            ("E4", "Ir25a", r"D:\20250922\4\labrum2\Experiment-1497_labrum2_ir25a.png", "yellow"),
        ],
    },
    {
        "label": "C",
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 5.0, "x": 0.25, "y": 0.35, "rotate": 0},
        "annotations": [
            #{"type": "arrow", "x": 0.26, "y": 0.55, "angle": 35, "length": 0.16, "color": "#E69F00", "width": 12},
            #{"type": "arrow", "x": 0.62, "y": 0.35, "angle": 90, "length": 0.16, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("D1", "DAPI", r"D:\20250922\4\labrum\Experiment-1497_labrum_DAPI.png", "white"),
            ("D2", "Ir75f.3", r"D:\20250922\4\labrum\Experiment-1497_labrum_ir75f3_2.png", "cyan"),
            ("D3", "Ir315.2", r"D:\20250922\4\labrum\Experiment-1497_labrum_ir3152.png", "magenta"),
            ("D4", "Ir25a", r"D:\20250922\4\labrum\Experiment-1497_labrum_ir25a.png", "yellow"),
        ],
    },
    {
        "label": "D",
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 3.0, "x": 0.0, "y": 0.69, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.26, "y": 0.55, "angle": 35, "length": 0.16, "color": "#E69F00", "width": 12},
            #{"type": "arrow", "x": 0.54, "y": 0.53, "angle": 90, "length": 0.16, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("C1", "DAPI", r"D:\20250922\4\maxilla\Labrum_1496_max_DAPI.png", "white"),
            ("C2", "Ir75f.3", r"D:\20250922\4\maxilla\Labrum_1496_max_ir75f3.png", "cyan"),
            ("C3", "Ir315.2", r"D:\20250922\4\maxilla\Labrum_1496_max_ir3152.png", "magenta"),
            ("C4", "Ir25a", r"D:\20250922\4\maxilla\Labrum_1496_max_ir25a.png", "yellow"),
        ],
    },
    {
        "label": "E",
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 3.0, "x": 0.20, "y": 0.35, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.59, "y": 0.62, "angle": -145, "length": 0.16, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("A1", "DAPI", r"d:\20250922\4\Labrum_1496_ir75f3_DAPI_only.png", "white"),
            ("A2", "Ir75f.3", r"D:\20250922\4\Labrum_1496_ir75f3.png", "cyan"),
            ("A3", "Ir317.2", r"D:\20250922\4\Labrum_1496_ir3172.png", "magenta"),
            ("A4", "Ir25a", r"D:\20250922\4\Labrum_1496_ir25a.png", "yellow"),
        ],
    },
    {
        "label": "F",
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 3.0, "x": 0.15, "y": 0.35, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.48, "y": 0.54, "angle": -145, "length": 0.16, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("B1", "DAPI", r"d:\20250922\4\labium2\Labrum_1496_2_DAPI.png", "white"),
            ("B2", "Ir75f.3", r"d:\20250922\4\labium2\Labrum_1496_2_ir75f3.png", "cyan"),
            ("B3", "Ir317.2", r"d:\20250922\4\labium2\Labrum_1496_2_ir3172.png", "magenta"),
            ("B4", "Ir25a", r"d:\20250922\4\labium2\Labrum_1496_2_ir25a.png", "yellow"),
        ],
    },
]

panels = {}
separate_channel_group_specs = []

for group in separate_channel_groups:
    panel_ids = []
    for panel_id, channel_name, image_path, channel_color in group["channels"]:
        channel_label = None
        if group["label"] == "A":
            channel_label = channel_text_label(
                channel_name,
                color=channel_color,
                **separate_channel_label_style,
            )
        panels[panel_id] = make_channel_panel(
            image_path,
            group["view"],
            group["border_color"],
            channel_label=channel_label,
            scale_bar_full_px=separate_channel_scale_bar["full_px"],
            scale_bar_orig_um=separate_channel_scale_bar["orig_um"],
            scale_bar_new_um=group.get("scale_bar_new_um", separate_channel_scale_bar["new_um"]),
        )
        panels[panel_id]["annotations"].extend(group.get("annotations", []))
        panel_ids.append(panel_id)

    separate_channel_group_specs.append(
        (group["label"], tuple(panel_ids), group["border_color"])
    )

for _group_label, panel_ids, _border_color in separate_channel_group_specs:
    for panel_id in panel_ids:
        panels[panel_id]["label"] = ""
        panels[panel_id].pop("border", None)

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = inches_to_px(COL_GAP_IN, DPI)

panel_layout = build_channel_group_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px,
    [panel_ids for _group_label, panel_ids, _border_color in separate_channel_group_specs],
)

def add_gaps_to_2x2_channel_groups(
    layout,
    group_specs,
    internal_gap_px,
    border_thickness=12,
    between_group_visible_gap_by_row=None,
    between_group_content_gap_by_row=None,
    between_row_visible_gap_px=0,
):
    adjusted = dict(layout)
    between_group_visible_gap_by_row = between_group_visible_gap_by_row or {}
    between_group_content_gap_by_row = between_group_content_gap_by_row or {}
    border_clearance_px = int(border_thickness) * 2

    def group_bbox(panel_ids):
        x0 = min(adjusted[panel_id][0] for panel_id in panel_ids)
        y0 = min(adjusted[panel_id][1] for panel_id in panel_ids)
        x1 = max(adjusted[panel_id][0] + adjusted[panel_id][2] for panel_id in panel_ids)
        y1 = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in panel_ids)
        return x0, y0, x1 - x0, y1 - y0

    def place_2x2_group(panel_ids, x, y, w, h):
        left_w = (w - internal_gap_px) // 2
        right_w = w - internal_gap_px - left_w
        top_h = (h - internal_gap_px) // 2
        bottom_h = h - internal_gap_px - top_h
        top_left_id, top_right_id, bottom_left_id, bottom_right_id = panel_ids
        adjusted[top_left_id] = (x, y, left_w, top_h)
        adjusted[top_right_id] = (x + left_w + internal_gap_px, y, right_w, top_h)
        adjusted[bottom_left_id] = (x, y + top_h + internal_gap_px, left_w, bottom_h)
        adjusted[bottom_right_id] = (x + left_w + internal_gap_px, y + top_h + internal_gap_px, right_w, bottom_h)

    row_group_specs = [
        group_specs[row_start:row_start + 2]
        for row_start in range(0, len(group_specs), 2)
    ]
    row_panel_ids = [
        tuple(panel_id for _label, ids, _color in row_groups for panel_id in ids)
        for row_groups in row_group_specs
    ]
    row_boxes = [group_bbox(panel_ids) for panel_ids in row_panel_ids]
    row_y0 = min(y for _x, y, _w, _h in row_boxes)
    row_y1 = max(y + h for _x, y, _w, h in row_boxes)
    row_gap_content_px = border_clearance_px + between_row_visible_gap_px
    available_h = row_y1 - row_y0 - row_gap_content_px * (len(row_boxes) - 1)
    base_row_h = available_h // len(row_boxes)
    new_row_heights = [base_row_h] * len(row_boxes)
    new_row_heights[-1] += available_h - sum(new_row_heights)
    next_y = row_y0
    for panel_ids, (_old_x, old_y, _old_w, old_h), new_h in zip(row_panel_ids, row_boxes, new_row_heights):
        for panel_id in panel_ids:
            x, y, w, h = adjusted[panel_id]
            rel_y = (y - old_y) / old_h
            rel_h = h / old_h
            adjusted[panel_id] = (x, int(round(next_y + rel_y * new_h)), w, int(round(rel_h * new_h)))
        next_y += new_h + row_gap_content_px

    for row_start in range(0, len(group_specs), 2):
        row_idx = row_start // 2
        _left_label, left_ids, _left_color = group_specs[row_start]
        left_x, left_y, left_w, left_h = group_bbox(left_ids)
        if row_start + 1 >= len(group_specs):
            place_2x2_group(left_ids, left_x, left_y, left_w, left_h)
            continue

        _right_label, right_ids, _right_color = group_specs[row_start + 1]
        right_x, right_y, right_w, right_h = group_bbox(right_ids)
        if row_idx in between_group_content_gap_by_row:
            between_group_gap_px = between_group_content_gap_by_row[row_idx]
        else:
            visible_gap_px = between_group_visible_gap_by_row.get(row_idx, internal_gap_px)
            between_group_gap_px = border_clearance_px + visible_gap_px
        row_x = min(left_x, right_x)
        row_right = max(left_x + left_w, right_x + right_w)
        row_w = row_right - row_x
        new_left_w = (row_w - between_group_gap_px) // 2
        new_right_w = row_w - between_group_gap_px - new_left_w
        place_2x2_group(left_ids, row_x, left_y, new_left_w, left_h)
        place_2x2_group(right_ids, row_x + new_left_w + between_group_gap_px, right_y, new_right_w, right_h)

    return adjusted


separate_channels_arrows_border_thickness = 12
separate_channels_arrows_panel_gap_px = inches_to_px(0.02, DPI)
panel_layout = add_gaps_to_2x2_channel_groups(
    panel_layout,
    separate_channel_group_specs,
    separate_channels_arrows_panel_gap_px,
    border_thickness=separate_channels_arrows_border_thickness,
    between_group_visible_gap_by_row={0: 0, 1: 0, 2: separate_channels_arrows_panel_gap_px},
    between_group_content_gap_by_row={2: separate_channels_arrows_panel_gap_px},
    between_row_visible_gap_px=0,
)

separate_channels_arrows_group_rows = [
    separate_channel_group_specs[row_start:row_start + 2]
    for row_start in range(0, len(separate_channel_group_specs), 2)
]
separate_channels_arrows_row_panel_ids = {
    f"row{row_idx}": tuple(
        panel_id
        for _group_label, group_panel_ids, _border_color in row_groups
        for panel_id in group_panel_ids
    )
    for row_idx, row_groups in enumerate(separate_channels_arrows_group_rows)
}
separate_channels_arrows_row_labels = {
    # row0 = A/B, row1 = C/D, row2 = E/F
    "row0-row2": "Ir75f.3 and Ir317.2",
}

canvas = render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels, dpi=DPI,
    scale_font_size=60,
    axis_font_size=60,
)

for _group_label, panel_ids, border_color in separate_channel_group_specs:
    if _group_label in {"E", "F"}:
        continue
    canvas = add_group_border(canvas, panel_layout, panel_ids, color=border_color, thickness=separate_channels_arrows_border_thickness)

canvas = add_group_border(
    canvas,
    panel_layout,
    separate_channels_arrows_row_panel_ids["row2"],
    color=LABIUM_BORDER_COLOR,
    thickness=separate_channels_arrows_border_thickness,
)

draw = ImageDraw.Draw(canvas)
group_label_font = load_font_safe("arial.ttf", 96)
group_label_pad_px = inches_to_px(LABEL_PAD_IN, DPI)

for group_label, panel_ids, _border_color in separate_channel_group_specs:
    x0 = min(panel_layout[panel_id][0] for panel_id in panel_ids)
    y0 = min(panel_layout[panel_id][1] for panel_id in panel_ids)
    label_color = panels[panel_ids[0]].get("label_color", "white")
    draw.text((x0 + group_label_pad_px, y0 + group_label_pad_px), group_label, fill=label_color, font=group_label_font)

canvas = register_figure_canvas('Ir75f.3 and Ir317.2 Individual Channels', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure S8: Ir308 and Ir75u.1


In [ ]:
scale_bar_thickness = 12

figure3S_test_scale_bar = {
    "full_px": 251,
    "orig_um": 20,
    "new_um": 15,
    "color": "white",
    "thickness": 12,
    "number": False,
}

separate_channel_label_style_ab = make_channel_label_style(x=0.94, y=0.60, size=60)

separate_channel_label_style_cj = make_channel_label_style(x=0.94, y=0.80, size=60)

separate_channel_groups = [
    {
        "label": "A",
        "scale_bar": {"full_px": 768, "orig_um": 100, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 4.0, "x": 0.32, "y": 0.32, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.40, "y": 0.53, "angle": -145, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("A1", "", r"D:\20250922\5\Composite_1498.png", "white"),
            ("A2", "DAPI", r"D:\20250922\5\Composite_1498_DAPI_higher.png", "white"),
            ("A3", "Ir75u.1", r"D:\20250922\5\Composite_1498_IR.png", "magenta"),
            ("A4", "Ir25a", r"D:\20250922\5\Composite_1498_IR25a.png", "yellow"),
        ],
    },
    {
        "label": "B",
        "scale_bar": {"full_px": 768, "orig_um": 100, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 4.0, "x": 0.30, "y": 0.675, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.54, "y": 0.56, "angle": -145, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("B1", "", r"D:\20250922\5\Composite_1498.png", "white"),
            ("B2", "DAPI", r"D:\20250922\5\Composite_1498_DAPI_higher.png", "white"),
            ("B3", "Ir75u.1", r"D:\20250922\5\Composite_1498_IR.png", "magenta"),
            ("B4", "Ir25a", r"D:\20250922\5\Composite_1498_IR25a.png", "yellow"),
        ],
    },
    {
        "label": "C",
        "channel_labels": True,
        "scale_bar": {"full_px": 762, "orig_um": 100, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": FRONS_BORDER_COLOR,
        "view": {"zoom": 7.0, "x": 0.7, "y": 0.73, "rotate": -12},
        "annotations": [
            #{"type": "arrow", "x": 0.47, "y": 0.50, "angle": 305, "length": 0, "color": "#E69F00", "width": 12},
            #{"type": "arrow", "x": 0.54, "y": 0.53, "angle": 90, "length": 0.0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("C1", "", r"D:\20260601\20260601\2\Experiment-1682_composite_antenna.png", "white"),
            ("C2", "DAPI", r"D:\20260601\20260601\2\Experiment-1682_DAPI_antenna.png", "white"),
            ("C3", "Ir308", r"D:\20260601\20260601\2\Experiment-1682_Ir308_antenna.png", "magenta"),
        ],
    },
    {
        "label": "D",
        "scale_bar": {"full_px": 762, "orig_um": 100, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": "#999999",
        "view": {"zoom": 7.0, "x": 0.7, "y": 0.63, "rotate": -12},
        "annotations": [],
        "channels": [
            ("D1", "", r"D:\20260601\20260601\2\Experiment-1682_composite_antenna2.png", "white"),
            ("D2", "DAPI", r"D:\20260601\20260601\2\Experiment-1682_DAPI_antenna2.png", "white"),
            ("D3", "Ir308", r"D:\20260601\20260601\2\Experiment-1682_Ir308_antenna2.png", "magenta"),
        ],
    },
    {
        "label": "E",
        "scale_bar": {"full_px": 762, "orig_um": 100, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 5.0, "x": 0.56, "y": 0.42, "rotate": 0},
        "annotations": [
            #{"type": "arrow", "x": 0.20, "y": 0.58, "angle": 35, "length": 0.16, "color": "#E69F00", "width": 12},
            #{"type": "arrow", "x": 0.67, "y": 0.645, "angle": -135, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("E1", "Composite", r"D:\20260601\20260601\2\Experiment-1682_composite_esophagus.png", "white"),
            ("E2", "DAPI", r"D:\20260601\20260601\2\Experiment-1682_DAPI_esophagus.png", "cyan"),
            ("E3", "Ir308", r"D:\20260601\20260601\2\Experiment-1682_Ir308_esophagus.png", "magenta"),
        ],
    },
    {
        "label": "F",
        "scale_bar": {"full_px": 762, "orig_um": 100, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 4.0, "x": 0.56, "y": 0.27, "rotate": -12},
        "annotations": [
            #{"type": "arrow", "x": 0.39, "y": 0.50, "angle": 305, "length": 0, "color": "#E69F00", "width": 12},
            #{"type": "arrow", "x": 0.56, "y": 0.53, "angle": 90, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("F1", "Composite", r"D:\20260601\20260601\2\Experiment-1682_composite.png", "white"),
            ("F2", "DAPI", r"D:\20260601\20260601\2\Experiment-1682_DAPI.png", "cyan"),
            ("F3", "Ir308", r"D:\20260601\20260601\2\Experiment-1682_Ir308.png", "magenta"),
        ],
    },
    {
        "label": "G",
        "scale_bar": {"full_px": 639, "orig_um": 100, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 12.0, "x": 0.61, "y": 0.27, "rotate": 12},
        "annotations": [
            #{"type": "arrow", "x": 0.57, "y": 0.78, "angle": 180, "length": 0.0, "color": "#E69F00", "width": 12},
            #{"type": "arrow", "x": 0.3, "y": 0.8, "angle": 305, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("G1", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_composite_maxpalpl.png", "white"),
            ("G2", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_DAPI_maxpalpl.png", "white"),
            ("G3", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_Ir308_maxpalpl.png", "white"),
        ],
    },
    {
        "label": "H",
        "border_color": "#999999",
        "scale_bar": {"full_px": 639, "orig_um": 100, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},

        "view": {"zoom": 12.0, "x": 0.61, "y": 0.28, "rotate": 12},
        "annotations": [],
        "channels": [
            ("H1", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_composite_maxgal.png", "white"),
            ("H2", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_DAPI_maxgal.png", "white"),
            ("H3", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_Ir308_maxgal.png", "white"),
        ],
    },
    {
        "label": "I",
        "border_color": LABIUM_BORDER_COLOR,
        "scale_bar": {"full_px": 639, "orig_um": 100, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},

        "view": {"zoom": 10.0, "x": 0.5, "y": 0.13, "rotate": 12},
        "annotations": [],
        "channels": [
            ("I1", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_composite_labv.png", "white"),
            ("I2", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_DAPI_labv.png", "white"),
            ("I3", "", r"D:\20260601\20260601\2\Experiment-1683_VD large_Ir308_labv.png", "white"),
        ],

    },
    {
        "label": "J",
        "border_color": LABIUM_BORDER_COLOR,
        "scale_bar": {"full_px": 639, "orig_um": 100, "new_um": 5, "color": "white", "thickness": scale_bar_thickness, "number": False},

        "view": {"zoom": 12.0, "x": 0.6, "y": 0.14, "rotate": 12},
        "annotations": [],
        "channels": [
            ("J1", "", r"D:\20260601\20260601\2\Labium 1683 VD large\Experiment-1683_VD large_composite_labium_new.png", "white"),
            ("J2", "", r"D:\20260601\20260601\2\Labium 1683 VD large\Experiment-1683_VD large_DAPI_labium_new.png", "white"),
            ("J3", "", r"D:\20260601\20260601\2\Labium 1683 VD large\Experiment-1683_VD large_Ir308_labium_new.png", "white"),
        ],
    },
]

panels = {}
separate_channel_group_specs = []

for group in separate_channel_groups:
    panel_ids = []
    group_scale_bar = group.get("scale_bar")
    show_channel_labels = group.get("channel_labels", group["label"] in {"A", "C"})
    channel_label_style = (
        separate_channel_label_style_ab
        if group["label"] in {"A", "B"}
        else separate_channel_label_style_cj
    )

    for panel_id, channel_name, image_path, channel_color in group["channels"]:
        channel_label = None
        if show_channel_labels and channel_name:
            channel_label = channel_text_label(
                channel_name,
                color=channel_color,
                **channel_label_style,
            )
        panels[panel_id] = make_channel_panel(
            image_path,
            group["view"],
            group["border_color"],
            channel_label=channel_label,
        )
        if group_scale_bar:
            panels[panel_id]["scale_bar"] = group_scale_bar.copy()
        else:
            panels[panel_id].pop("scale_bar", None)
        panels[panel_id]["annotations"].extend(group.get("annotations", []))
        panel_ids.append(panel_id)

    separate_channel_group_specs.append(
        (group["label"], tuple(panel_ids), group["border_color"])
    )

for _group_label, panel_ids, _border_color in separate_channel_group_specs:
    for panel_id in panel_ids:
        panels[panel_id]["label"] = ""
        panels[panel_id].pop("border", None)

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = 0

panel_layout = build_variable_channel_group_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px,
    [panel_ids for _group_label, panel_ids, _border_color in separate_channel_group_specs],
)

def add_gaps_to_channel_group_layout(layout, group_specs, gap_px):
    adjusted = dict(layout)

    def group_bbox(panel_ids):
        x0 = min(adjusted[panel_id][0] for panel_id in panel_ids)
        y0 = min(adjusted[panel_id][1] for panel_id in panel_ids)
        x1 = max(adjusted[panel_id][0] + adjusted[panel_id][2] for panel_id in panel_ids)
        y1 = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in panel_ids)
        return x0, y0, x1 - x0, y1 - y0

    def place_group(panel_ids, x, y, w, h):
        if len(panel_ids) == 4:
            left_w = (w - gap_px) // 2
            right_w = w - gap_px - left_w
            top_h = (h - gap_px) // 2
            bottom_h = h - gap_px - top_h
            top_left_id, top_right_id, bottom_left_id, bottom_right_id = panel_ids
            adjusted[top_left_id] = (x, y, left_w, top_h)
            adjusted[top_right_id] = (x + left_w + gap_px, y, right_w, top_h)
            adjusted[bottom_left_id] = (x, y + top_h + gap_px, left_w, bottom_h)
            adjusted[bottom_right_id] = (x + left_w + gap_px, y + top_h + gap_px, right_w, bottom_h)
            return

        if len(panel_ids) == 3:
            available_w = w - gap_px * 2
            first_w = available_w // 3
            second_w = available_w // 3
            third_w = available_w - first_w - second_w
            panel_x = x
            for panel_id, panel_w in zip(panel_ids, (first_w, second_w, third_w)):
                adjusted[panel_id] = (panel_x, y, panel_w, h)
                panel_x += panel_w + gap_px
            return

        if len(panel_ids) == 1:
            adjusted[panel_ids[0]] = (x, y, w, h)
            return

        raise ValueError(f"Expected 1, 3, or 4 panels per group, got {len(panel_ids)}: {panel_ids}")

    for row_start in range(0, len(group_specs), 2):
        _left_label, left_ids, _left_color = group_specs[row_start]
        left_x, left_y, left_w, left_h = group_bbox(left_ids)
        if row_start + 1 >= len(group_specs):
            place_group(left_ids, left_x, left_y, left_w, left_h)
            continue

        _right_label, right_ids, _right_color = group_specs[row_start + 1]
        right_x, right_y, right_w, right_h = group_bbox(right_ids)
        left_gap_share = gap_px // 2
        right_gap_share = gap_px - left_gap_share
        place_group(left_ids, left_x, left_y, left_w - left_gap_share, left_h)
        place_group(right_ids, right_x + right_gap_share, right_y, right_w - right_gap_share, right_h)

    return adjusted


figure3S_test_panel_gap_px = inches_to_px(0.03, DPI)
panel_layout = add_gaps_to_channel_group_layout(
    panel_layout,
    separate_channel_group_specs,
    figure3S_test_panel_gap_px,
)

figure3S_test_group_rows = [
    separate_channel_group_specs[row_start:row_start + 2]
    for row_start in range(0, len(separate_channel_group_specs), 2)
]
figure3S_test_row_panel_ids = {
    f"row{row_idx}": tuple(
        panel_id
        for _group_label, group_panel_ids, _border_color in row_groups
        for panel_id in group_panel_ids
    )
    for row_idx, row_groups in enumerate(figure3S_test_group_rows)
}

figure3S_test_row_labels = {
    # row0 = A/B, row1 = C/D, row2 = E/F, row3 = G/H, row4 = I/J
    # Use "row1-row2" to center one label across rows 1 and 2.
    "row0": "Ir75u.1",
    "row1-row4": "Ir308",
}

canvas = render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels, dpi=DPI,
    scale_font_size=60,
    axis_font_size=60,
)

figure3S_test_row_border_colors = {
    "row0": MAXILLA_BORDER_COLOR,
    "row1": FRONS_BORDER_COLOR,
    "row2": LABRUM_BORDER_COLOR,
    "row3": MAXILLA_BORDER_COLOR,
    "row4": LABIUM_BORDER_COLOR,
}

for row_key, panel_ids in figure3S_test_row_panel_ids.items():
    canvas = add_group_border(
        canvas,
        panel_layout,
        panel_ids,
        color=figure3S_test_row_border_colors[row_key],
        thickness=12,
    )

draw = ImageDraw.Draw(canvas)
group_label_font = load_font_safe("arial.ttf", 96)
group_label_pad_px = inches_to_px(LABEL_PAD_IN, DPI)

for group_label, panel_ids, _border_color in separate_channel_group_specs:
    x0 = min(panel_layout[panel_id][0] for panel_id in panel_ids)
    y0 = min(panel_layout[panel_id][1] for panel_id in panel_ids)
    label_color = panels[panel_ids[0]].get("label_color", "white")
    draw.text((x0 + group_label_pad_px, y0 + group_label_pad_px), group_label, fill=label_color, font=group_label_font)

canvas = register_figure_canvas('Ir308 and Ir75u.1 Combined', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure S3: Ir25a and Ir8a

In [ ]:
scale_bar_thickness = 8
irco_all_regions_fig_h_px = fig_h_px
irco_placeholder_color = "#999999"
irco_panel_gap_px = inches_to_px(0.03, DPI)

irco_channel_label_style = {
    "x": 0.94,
    "y": 0.84,
    "size": 60,
    "anchor": "right",
}


def make_supplement_channel_group_panels(channel_groups, channel_label_style, placeholder_color):
    panels = {}
    group_specs = []

    for group in channel_groups:
        panel_ids = []
        group_scale_bar = group.get("scale_bar")
        show_channel_labels = group.get("channel_labels", False)

        for panel_id, channel_name, image_path, channel_color in group["channels"]:
            channel_label = None
            if show_channel_labels and channel_name:
                channel_label = channel_text_label(
                    channel_name,
                    color=channel_color,
                    **channel_label_style,
                )
            panels[panel_id] = make_channel_panel(
                image_path,
                group.get("view", {"zoom": 1.0, "x": 0.5, "y": 0.5, "rotate": 0}),
                group.get("border_color", placeholder_color),
                channel_label=channel_label,
                label_color=group.get("label_color", "black"),
            )
            if group_scale_bar:
                panels[panel_id]["scale_bar"] = group_scale_bar.copy()
            else:
                panels[panel_id].pop("scale_bar", None)
            panels[panel_id]["annotations"].extend(group.get("annotations", []))
            panel_ids.append(panel_id)

        group_specs.append(
            (group["label"], tuple(panel_ids), group.get("border_color", placeholder_color))
        )

    for _group_label, panel_ids, _border_color in group_specs:
        for panel_id in panel_ids:
            panels[panel_id]["label"] = ""
            panels[panel_id].pop("border", None)

    return panels, group_specs


def build_channel_row_groups_layout(fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, panel_gap_px, group_panel_ids):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    n_rows = len(group_panel_ids)
    usable_h = inner_h - row_gap_px * (n_rows - 1)
    row_h = usable_h // n_rows
    row_heights = [row_h] * n_rows
    row_heights[-1] += usable_h - sum(row_heights)

    layout = {}
    y = mt
    for row_idx, panel_ids in enumerate(group_panel_ids):
        panel_ids = tuple(panel_ids)
        row_h_i = row_heights[row_idx]
        available_w = inner_w - panel_gap_px * (len(panel_ids) - 1)
        base_w = available_w // len(panel_ids)
        remainder = available_w - base_w * len(panel_ids)

        x = ml
        for panel_idx, panel_id in enumerate(panel_ids):
            panel_w = base_w + (1 if panel_idx < remainder else 0)
            layout[panel_id] = (x, y, panel_w, row_h_i)
            x += panel_w + panel_gap_px
        y += row_h_i + row_gap_px

    return layout


def draw_supplement_group_labels(canvas, layout, group_specs, panels, font_size=96):
    draw = ImageDraw.Draw(canvas)
    group_label_font = load_font_safe("arial.ttf", font_size)
    group_label_pad_px = inches_to_px(LABEL_PAD_IN, DPI)

    for group_label, panel_ids, _border_color in group_specs:
        x0 = min(layout[panel_id][0] for panel_id in panel_ids)
        y0 = min(layout[panel_id][1] for panel_id in panel_ids)
        label_color = panels[panel_ids[0]].get("label_color", "black")
        draw.text((x0 + group_label_pad_px, y0 + group_label_pad_px), group_label, fill=label_color, font=group_label_font)

    return canvas


def render_supplement_channel_groups(fig_h, layout, panels, group_specs, row_panel_ids, row_labels, span_key):
    canvas = render_artboard(
        fig_w_px, fig_h, layout, panels, dpi=DPI,
        scale_font_size=60,
        axis_font_size=60,
    )

    for _group_label, panel_ids, border_color in group_specs:
        canvas = add_group_border(
            canvas,
            layout,
            panel_ids,
            color=border_color,
            thickness=12,
        )

    return draw_supplement_group_labels(canvas, layout, group_specs, panels)


irco_all_regions_groups = [
    {
        "label": "A",
        "label_color": "white",
        "channel_labels": True,
        "scale_bar": {"full_px": 380, "orig_um": 50, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": FRONS_BORDER_COLOR,
        "view": {"zoom": 7.0, "x": 0.48, "y": 0.285, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.54, "y": 0.56, "angle": -145, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("A1", "Composite", r"D:\larval chemosensory\20260626\new\Experiment-1735_ant1_composite.png", "white"),
            ("A2", "Ir25a", r"D:\larval chemosensory\20260626\new\Experiment-1735_ant1_ir25a.png", "magenta"),
            ("A3", "Ir8a", r"D:\larval chemosensory\20260626\new\Experiment-1735_ant1_ir8a.png", "yellow"),
        ],
    },
    {
        "label": "B",
        "label_color": "white",
        "channel_labels": False,
        "scale_bar": {"full_px": 380, "orig_um": 50, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},        "border_color": FRONS_BORDER_COLOR,
        "view": {"zoom": 7.0, "x": 0.37, "y": 0.70, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.54, "y": 0.56, "angle": -145, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("B1", "Composite", r"D:\larval chemosensory\20260626\new\Experiment-1735_labr_composite.png", "black"),
            ("B2", "Ir25a", r"D:\larval chemosensory\20260626\new\Experiment-1735_labr_ir25a.png", "magenta"),
            ("B3", "Ir8a", r"D:\larval chemosensory\20260626\new\Experiment-1735_labr_ir8a.png", "yellow"),
        ],
    },
    {
        "label": "C",
        "label_color": "white",
        "channel_labels": False,
        "scale_bar": {"full_px": 380, "orig_um": 50, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 5.0, "x": 0.20, "y": 0.80, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.54, "y": 0.56, "angle": -145, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("C1", "Composite", r"D:\larval chemosensory\20260626\new\Experiment-1735_labr_composite.png", "black"),
            ("C2", "Ir25a", r"D:\larval chemosensory\20260626\new\Experiment-1735_labr_ir25a.png", "magenta"),
            ("C3", "Ir8a", r"D:\larval chemosensory\20260626\new\Experiment-1735_labr_ir8a.png", "yellow"),
        ],
    },
    {
        "label": "D",
        "label_color": "white",
        "channel_labels": False,
        "scale_bar": {"full_px": 613, "orig_um": 50, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 4.0, "x": 0.82, "y": 0.28, "rotate": -90},
        "annotations": [
            #{"type": "arrow", "x": 0.54, "y": 0.56, "angle": -145, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("D1", "Composite", r"D:\larval chemosensory\20260626\new\Experiment-1734_max_composite.png", "black"),
            ("D2", "Ir25a", r"D:\larval chemosensory\20260626\new\Experiment-1734_max_ir25a.png", "magenta"),
            ("D3", "Ir8a", r"D:\larval chemosensory\20260626\new\Experiment-1734_max_ir8a.png", "yellow"),
        ],
    },
    {
        "label": "E",
        "label_color": "white",
        "channel_labels": False,
        "scale_bar": {"full_px": 613, "orig_um": 50, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 5.0, "x": 0.4, "y": 0.58, "rotate": -40},
        "annotations": [
            #{"type": "arrow", "x": 0.54, "y": 0.56, "angle": -145, "length": 0, "color": "#E69F00", "width": 12},
        ],
        "channels": [
            ("E1", "Composite", r"D:\larval chemosensory\20260626\new\Experiment-1734_composite.png", "black"),
            ("E2", "Ir25a", r"D:\larval chemosensory\20260626\new\Experiment-1734_ir25a.png", "magenta"),
            ("E3", "Ir8a", r"D:\larval chemosensory\20260626\new\Experiment-1734_ir8a.png", "yellow"),
        ],
    }
]

panels, irco_all_regions_group_specs = make_supplement_channel_group_panels(
    irco_all_regions_groups,
    irco_channel_label_style,
    irco_placeholder_color,
)

# Add 0.02 in (6 px at 300 DPI) beyond the shared row gap so adjacent
# 12 px group borders retain a small visible separation.
irco_row_gap_px = inches_to_px(ROW_GAP_IN + 0.02, DPI)
panel_layout = build_channel_row_groups_layout(
    fig_w_px,
    irco_all_regions_fig_h_px,
    ml,
    mr,
    mt,
    mb,
    irco_row_gap_px,
    irco_panel_gap_px,
    [panel_ids for _group_label, panel_ids, _border_color in irco_all_regions_group_specs],
)

irco_all_regions_row_panel_ids = {
    f"row{row_idx}": panel_ids
    for row_idx, (_group_label, panel_ids, _border_color) in enumerate(irco_all_regions_group_specs)
}
irco_all_regions_row_labels = {
    "row0-row4": "Ir25a/Ir8a",
}

canvas = render_supplement_channel_groups(
    irco_all_regions_fig_h_px,
    panel_layout,
    panels,
    irco_all_regions_group_specs,
    irco_all_regions_row_panel_ids,
    irco_all_regions_row_labels,
    "row0-row4",
)

canvas = register_figure_canvas('Irco all regions', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure S10: Gr13 and Gr8


In [ ]:
scale_bar_thickness = 8
figure4_grs_placeholder_color = "#999999"
figure4_grs_13_8_h_px = int(round(fig_h_px * 0.70))

# Match the Ir25a GCaMP validation channel-label styling.
separate_channel_label_style_ab = make_channel_label_style(x=0.95, y=0.85, size=50)

separate_channel_label_style_cj = make_channel_label_style(x=0.95, y=0.85, size=50)

separate_channel_groups = [
    {
        "label": "A",
        "channel_labels": True,
        "scale_bar": {"full_px": 245, "orig_um": 10, "new_um": 10, "color": "white", "thickness": 12, "number": False},
        "border_color": "#556E9B",
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.9, "rotate": 0},
        "annotations": [
            #{"text": "*", "x": 0.38, "y": 0.72, "color": "#E69F00", "size": 70},
        ],
        "channels": [
            ("A1", "Composite", r"D:\larval chemosensory\Gr8Gr13-01-VD\ROI 01 - labrum - z071 of 115\Gr8Gr13-01-VD_z070_roi01_combined_ch0_Gr8_ch1_Gr13_ch2_DAPI.png", "white"),
            ("A2", "DAPI", r"D:\larval chemosensory\Gr8Gr13-01-VD\ROI 01 - labrum - z071 of 115\Gr8Gr13-01-VD_z070_roi01_ch2_DAPI.png", "magenta"),
            ("A3", "Gr13", r"D:\larval chemosensory\Gr8Gr13-01-VD\ROI 01 - labrum - z071 of 115\Gr8Gr13-01-VD_z070_roi01_ch1_Gr13.png", "yellow"),
            ("A4", "Gr8", r"D:\larval chemosensory\Gr8Gr13-01-VD\ROI 01 - labrum - z071 of 115\Gr8Gr13-01-VD_z070_roi01_ch0_Gr8.png", "cyan"),
        ],
    },
    {
        "label": "B",
        "channel_labels": False,
        "scale_bar": {"full_px": 251, "orig_um": 10, "new_um": 10, "color": "white", "thickness": 12, "number": False},
        "border_color": "#556E9B",
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.3, "rotate": 0},
        "annotations": [
        ],
        "channels": [
            ("B1", "", r"D:\larval chemosensory\Gr8Gr13-01-VD\ROI 01 - labrum 2 - z109 of 115\Gr8Gr13-01-VD_z108_roi01_combined_ch0_Gr8_ch1_Gr13_ch2_DAPI.png", "white"),
            ("B2", "DAPI", r"D:\larval chemosensory\Gr8Gr13-01-VD\ROI 01 - labrum 2 - z109 of 115\Gr8Gr13-01-VD_z108_roi01_ch2_DAPI.png", "white"),
            ("B3", "Gr13", r"D:\larval chemosensory\Gr8Gr13-01-VD\ROI 01 - labrum 2 - z109 of 115\Gr8Gr13-01-VD_z108_roi01_ch1_Gr13.png", "magenta"),
            ("B4", "Gr8", r"D:\larval chemosensory\Gr8Gr13-01-VD\ROI 01 - labrum 2 - z109 of 115\Gr8Gr13-01-VD_z108_roi01_ch0_Gr8.png", "yellow"),
        ],
    },
    {
        "label": "C",
        "channel_labels": False,
        "scale_bar": {"full_px": 258, "orig_um": 10, "new_um": 10, "color": "white", "thickness": 12, "number": False},
        "border_color": "#8A9562",
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.32, "rotate": 0},
        "annotations": [
            #{"text": "*", "x": 0.74, "y": 0.27, "color": "white", "size": 70},
        ],
        "channels": [
            ("C1", "", r"D:\larval chemosensory\Gr8Gr13-01-DV\ROI 01 - maxilla - z152 of 239\Gr8Gr13-01-DV_z151_roi01_combined_ch0_Gr8_ch1_Gr13_ch2_DAPI.png", "white"),
            ("C2", "DAPI", r"D:\larval chemosensory\Gr8Gr13-01-DV\ROI 01 - maxilla - z152 of 239\Gr8Gr13-01-DV_z151_roi01_ch2_DAPI.png", "white"),
            ("C3", "Gr13", r"D:\larval chemosensory\Gr8Gr13-01-DV\ROI 01 - maxilla - z152 of 239\Gr8Gr13-01-DV_z151_roi01_ch1_Gr13.png", "magenta"),
            ("C4", "Gr8", r"D:\larval chemosensory\Gr8Gr13-01-DV\ROI 01 - maxilla - z152 of 239\Gr8Gr13-01-DV_z151_roi01_ch0_Gr8.png", "yellow"),
        ],
    },
    {
        "label": "D",
        "scale_bar": {"full_px": 252, "orig_um": 10, "new_um": 10, "color": "white", "thickness": 12, "number": False},
        "border_color": "#d7c7a0",
        "view": {"zoom": 1.0, "x": 0.8, "y": 0.35, "rotate": 0},
        "annotations": [
            #{"text": "*", "x": 0.42, "y": 0.22, "color": "#E69F00", "size": 70},
        ],
        "channels": [
            ("D1", "", r"D:\larval chemosensory\Gr8Gr13-01-DV\ROI 01 - labium-2 - z145 of 239\Gr8Gr13-01-DV_z144_roi01_combined_ch0_Gr8_ch1_Gr13_ch2_DAPI.png", "white"),
            ("D2", "DAPI", r"D:\larval chemosensory\Gr8Gr13-01-DV\ROI 01 - labium-2 - z145 of 239\Gr8Gr13-01-DV_z144_roi01_ch2_DAPI.png", "white"),
            ("D3", "Gr13", r"D:\larval chemosensory\Gr8Gr13-01-DV\ROI 01 - labium-2 - z145 of 239\Gr8Gr13-01-DV_z144_roi01_ch1_Gr13.png", "magenta"),
            ("D4", "Gr8", r"D:\larval chemosensory\Gr8Gr13-01-DV\ROI 01 - labium-2 - z145 of 239\Gr8Gr13-01-DV_z144_roi01_ch0_Gr8.png", "yellow"),
        ],
    },
]

panels = {}
separate_channel_group_specs = []

for group in separate_channel_groups:
    panel_ids = []
    group_scale_bar = group.get("scale_bar")
    show_channel_labels = group.get("channel_labels", group["label"] in {"A", "C"})
    channel_label_style = (
        separate_channel_label_style_ab
        if group["label"] in {"A", "B"}
        else separate_channel_label_style_cj
    )

    for panel_id, channel_name, image_path, channel_color in group["channels"]:
        channel_label = None
        if show_channel_labels and channel_name:
            channel_label = channel_text_label(
                channel_name,
                color=channel_color,
                **channel_label_style,
            )
        panels[panel_id] = make_channel_panel(
            image_path,
            group.get("view", {'zoom': 1.0, 'x': 0.5, 'y': 0.5, 'rotate': 0}),
            group.get("border_color", "#999999"),
            channel_label=channel_label,
        )
        if group_scale_bar:
            panels[panel_id]["scale_bar"] = group_scale_bar.copy()
        else:
            panels[panel_id].pop("scale_bar", None)
        panels[panel_id]["annotations"].extend(group.get("annotations", []))
        panel_ids.append(panel_id)

    separate_channel_group_specs.append(
        (group["label"], tuple(panel_ids), group.get("border_color", "#999999"))
    )

for _group_label, panel_ids, _border_color in separate_channel_group_specs:
    for panel_id in panel_ids:
        panels[panel_id]["label"] = ""
        panels[panel_id].pop("border", None)

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = 0

panel_layout = build_variable_channel_group_layout(
    fig_w_px, figure4_grs_13_8_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px,
    [panel_ids for _group_label, panel_ids, _border_color in separate_channel_group_specs],
)

figure4_grs_border_thickness = 12

def add_gaps_to_figure4_grs_2x2_groups(
    layout,
    group_specs,
    internal_gap_px,
    border_thickness=12,
    between_group_visible_gap_by_row=None,
    between_group_content_gap_by_row=None,
    between_row_visible_gap_px=0,
    between_row_content_gap_by_pair=None,
):
    adjusted = dict(layout)
    between_group_visible_gap_by_row = between_group_visible_gap_by_row or {}
    between_group_content_gap_by_row = between_group_content_gap_by_row or {}
    between_row_content_gap_by_pair = between_row_content_gap_by_pair or {}
    border_clearance_px = int(border_thickness) * 2

    def group_bbox(panel_ids):
        x0 = min(adjusted[panel_id][0] for panel_id in panel_ids)
        y0 = min(adjusted[panel_id][1] for panel_id in panel_ids)
        x1 = max(adjusted[panel_id][0] + adjusted[panel_id][2] for panel_id in panel_ids)
        y1 = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in panel_ids)
        return x0, y0, x1 - x0, y1 - y0

    def place_2x2_group(panel_ids, x, y, w, h):
        left_w = (w - internal_gap_px) // 2
        right_w = w - internal_gap_px - left_w
        top_h = (h - internal_gap_px) // 2
        bottom_h = h - internal_gap_px - top_h
        top_left_id, top_right_id, bottom_left_id, bottom_right_id = panel_ids
        adjusted[top_left_id] = (x, y, left_w, top_h)
        adjusted[top_right_id] = (x + left_w + internal_gap_px, y, right_w, top_h)
        adjusted[bottom_left_id] = (x, y + top_h + internal_gap_px, left_w, bottom_h)
        adjusted[bottom_right_id] = (x + left_w + internal_gap_px, y + top_h + internal_gap_px, right_w, bottom_h)

    row_group_specs = [group_specs[row_start:row_start + 2] for row_start in range(0, len(group_specs), 2)]
    row_panel_ids = [
        tuple(panel_id for _label, ids, _color in row_groups for panel_id in ids)
        for row_groups in row_group_specs
    ]
    row_boxes = [group_bbox(panel_ids) for panel_ids in row_panel_ids]
    row_y0 = min(y for _x, y, _w, _h in row_boxes)
    row_y1 = max(y + h for _x, y, _w, h in row_boxes)
    row_gap_content_by_pair = [
        between_row_content_gap_by_pair.get(row_idx, border_clearance_px + between_row_visible_gap_px)
        for row_idx in range(len(row_boxes) - 1)
    ]
    available_h = row_y1 - row_y0 - sum(row_gap_content_by_pair)
    base_row_h = available_h // len(row_boxes)
    new_row_heights = [base_row_h] * len(row_boxes)
    new_row_heights[-1] += available_h - sum(new_row_heights)
    next_y = row_y0
    for row_idx, (panel_ids, (_old_x, old_y, _old_w, old_h), new_h) in enumerate(zip(row_panel_ids, row_boxes, new_row_heights)):
        for panel_id in panel_ids:
            x, y, w, h = adjusted[panel_id]
            rel_y = (y - old_y) / old_h
            rel_h = h / old_h
            adjusted[panel_id] = (x, int(round(next_y + rel_y * new_h)), w, int(round(rel_h * new_h)))
        if row_idx < len(row_gap_content_by_pair):
            next_y += new_h + row_gap_content_by_pair[row_idx]

    for row_start in range(0, len(group_specs), 2):
        row_idx = row_start // 2
        _left_label, left_ids, _left_color = group_specs[row_start]
        left_x, left_y, left_w, left_h = group_bbox(left_ids)
        if row_start + 1 >= len(group_specs):
            place_2x2_group(left_ids, left_x, left_y, left_w, left_h)
            continue

        _right_label, right_ids, _right_color = group_specs[row_start + 1]
        right_x, right_y, right_w, right_h = group_bbox(right_ids)
        if row_idx in between_group_content_gap_by_row:
            between_group_gap_px = between_group_content_gap_by_row[row_idx]
        else:
            visible_gap_px = between_group_visible_gap_by_row.get(row_idx, internal_gap_px)
            between_group_gap_px = border_clearance_px + visible_gap_px
        row_x = min(left_x, right_x)
        row_right = max(left_x + left_w, right_x + right_w)
        row_w = row_right - row_x
        new_left_w = (row_w - between_group_gap_px) // 2
        new_right_w = row_w - between_group_gap_px - new_left_w
        place_2x2_group(left_ids, row_x, left_y, new_left_w, left_h)
        place_2x2_group(right_ids, row_x + new_left_w + between_group_gap_px, right_y, new_right_w, right_h)

    return adjusted


# Match the subtle 4 px separators in Ir25a GCaMP validation.
figure4_grs_panel_gap_px = 4
panel_layout = add_gaps_to_figure4_grs_2x2_groups(
    panel_layout,
    separate_channel_group_specs,
    figure4_grs_panel_gap_px,
    border_thickness=figure4_grs_border_thickness,
    between_group_visible_gap_by_row={0: figure4_grs_panel_gap_px, 1: figure4_grs_panel_gap_px},
    between_group_content_gap_by_row={0: figure4_grs_panel_gap_px},
    between_row_visible_gap_px=figure4_grs_panel_gap_px,
)

figure4_grs_group_rows = [
    separate_channel_group_specs[row_start:row_start + 2]
    for row_start in range(0, len(separate_channel_group_specs), 2)
]
figure4_grs_row_panel_ids = {
    f"row{row_idx}": tuple(
        panel_id
        for _group_label, group_panel_ids, _border_color in row_groups
        for panel_id in group_panel_ids
    )
    for row_idx, row_groups in enumerate(figure4_grs_group_rows)
}

figure4_grs_row_labels = {
    "row0-row1": "Gr13 and Gr8",
}

canvas = render_artboard(
    fig_w_px, figure4_grs_13_8_h_px, panel_layout, panels, dpi=DPI,
    scale_font_size=60,
    axis_font_size=60,
)

figure4_grs_row_border_colors = {
    "row0": LABIUM_BORDER_COLOR,
    "row1": figure4_grs_placeholder_color,
    "row2": figure4_grs_placeholder_color,
    "row3": figure4_grs_placeholder_color,
    "row4": figure4_grs_placeholder_color,
}

for row_key, panel_ids in figure4_grs_row_panel_ids.items():
    if row_key == "row0":
        canvas = add_group_border(
            canvas,
            panel_layout,
            panel_ids,
            color=LABRUM_BORDER_COLOR,
            thickness=figure4_grs_border_thickness,
        )

for group_label, panel_ids, border_color in separate_channel_group_specs:
    if group_label in {"A", "B"}:
        continue
    canvas = add_group_border(
        canvas,
        panel_layout,
        panel_ids,
        color=border_color,
        thickness=figure4_grs_border_thickness,
    )

draw = ImageDraw.Draw(canvas)
group_label_font = load_font_safe("arial.ttf", 72)
group_label_pad_px = 8

for group_label, panel_ids, _border_color in separate_channel_group_specs:
    x0 = min(panel_layout[panel_id][0] for panel_id in panel_ids)
    y0 = min(panel_layout[panel_id][1] for panel_id in panel_ids)
    label_color = panels[panel_ids[0]].get("label_color", "white")
    draw.text((x0 + group_label_pad_px, y0 + group_label_pad_px), group_label, fill=label_color, font=group_label_font)

canvas = register_figure_canvas('Grs (13 and 8)', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure S5: Ir25a and Orco


In [ ]:
scale_bar_thickness = 8
ir25a_orco_fig_h_px = int(round(fig_h_px * 2 / 4))
ir25a_orco_placeholder_color = "#999999"

ir25a_orco_channel_label_style = {
    "x": 0.94,
    "y": 0.86,
    "size": 50,
    "anchor": "right",
}

ir25a_orco_channel_groups = [
    {
        "label": "A",
        "channel_labels": True,
        "scale_bar": {"full_px": 320, "orig_um": 50, "new_um": 15, "color": "white", "thickness": 12, "number": False},
        "border_color": FRONS_BORDER_COLOR,
        "view": {"zoom": 8.4, "x": 0.55, "y": 0.68, "rotate": 100},
        "annotations": [
            #{"text": "*", "x": 0.38, "y": 0.72, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.49, "y": 0.41, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("A1", "Composite", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1749_ant_composite.png", "white"),
            ("A2", "Ir25a", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1749_ant_ir25a.png", "magenta"),
            ("A3", "Orco", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1749_ant_orco.png", "cyan"),
        ],
    },
    {
        "label": "B",
        "scale_bar": {"full_px": 320, "orig_um": 50, "new_um": 15, "color": "white", "thickness": 12, "number": False},
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 4.4, "x": 0.6, "y": 0.27, "rotate": 100},
        "annotations": [
            #{"type": "arrow", "x": 0.43, "y": 0.6, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("B1", "Composite", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1749_labr_composite.png", "white"),
            ("B2", "Ir25a", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1749_labr_ir25a.png", "yellow"),
            ("B3", "Orco", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1749_labr_orco.png", "magenta"),
        ],
    },
    {
        "label": "C",
        "scale_bar": {"full_px": 318, "orig_um": 50, "new_um": 15, "color": "white", "thickness": 12, "number": False},
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 7.4, "x": 0.32, "y": 0.27, "rotate": 245},
        "annotations": [
            #{"type": "arrow", "x": 0.43, "y": 0.6, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("C1", "Composite", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1750_max_composite.png", "white"),
            ("C2", "Ir25a", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1750_max_ir25a.png", "yellow"),
            ("C3", "Orco", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1750_max_orco.png", "magenta"),
        ],
    },
    {
        "label": "D",
        "channel_labels": True,
        "scale_bar": {"full_px": 318, "orig_um": 50, "new_um": 15, "color": "white", "thickness": 12, "number": False},
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 7.4, "x": 0.40, "y": 0.14, "rotate": 245},
        "annotations": [
            #{"text": "*", "x": 0.38, "y": 0.72, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.49, "y": 0.41, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("D1", "", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1750_composite.png", "white"),
            ("D2", "", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1750_ir25a.png", "magenta"),
            ("D3", "", r"D:\larval chemosensory\Orco, Gr25, Ir25a\Orco, Gr25, Ir25a\Experiment-1750_orco.png", "cyan"),
        ],
    },
]

panels = {}
ir25a_orco_group_specs = []

for group in ir25a_orco_channel_groups:
    panel_ids = []
    group_scale_bar = group.get("scale_bar")
    show_channel_labels = group.get("channel_labels", group["label"] == "A")

    for panel_id, channel_name, image_path, channel_color in group["channels"]:
        channel_label = None
        if show_channel_labels and channel_name:
            channel_label = channel_text_label(
                channel_name,
                color=channel_color,
                **ir25a_orco_channel_label_style,
            )
        panels[panel_id] = make_channel_panel(
            image_path,
            group.get("view", {"zoom": 1.0, "x": 0.5, "y": 0.5, "rotate": 0}),
            group.get("border_color", ir25a_orco_placeholder_color),
            channel_label=channel_label,
        )
        if group_scale_bar:
            panels[panel_id]["scale_bar"] = group_scale_bar.copy()
        else:
            panels[panel_id].pop("scale_bar", None)
        panels[panel_id]["annotations"].extend(group.get("annotations", []))
        panel_ids.append(panel_id)

    ir25a_orco_group_specs.append(
        (group["label"], tuple(panel_ids), group.get("border_color", ir25a_orco_placeholder_color))
    )

for _group_label, panel_ids, _border_color in ir25a_orco_group_specs:
    for panel_id in panel_ids:
        panels[panel_id]["label"] = ""
        panels[panel_id].pop("border", None)

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = 0

panel_layout = build_variable_channel_group_layout(
    fig_w_px, ir25a_orco_fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px,
    [panel_ids for _group_label, panel_ids, _border_color in ir25a_orco_group_specs],
)

def add_gaps_to_ir25a_orco_groups(
    layout,
    group_specs,
    internal_gap_px,
    border_thickness,
    visible_group_gap_px,
    visible_row_gap_px,
):
    adjusted = dict(layout)
    border_clearance_px = int(border_thickness) * 2

    def group_bbox(panel_ids):
        x0 = min(adjusted[panel_id][0] for panel_id in panel_ids)
        y0 = min(adjusted[panel_id][1] for panel_id in panel_ids)
        x1 = max(adjusted[panel_id][0] + adjusted[panel_id][2] for panel_id in panel_ids)
        y1 = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in panel_ids)
        return x0, y0, x1 - x0, y1 - y0

    def place_3_panel_group(panel_ids, x, y, w, h):
        available_w = w - internal_gap_px * 2
        first_w = available_w // 3
        second_w = available_w // 3
        third_w = available_w - first_w - second_w
        panel_x = x
        for panel_id, panel_w in zip(panel_ids, (first_w, second_w, third_w)):
            adjusted[panel_id] = (panel_x, y, panel_w, h)
            panel_x += panel_w + internal_gap_px

    # Group borders expand outward. Reserve two border widths plus a small
    # visible gap so the upper and lower border strokes cannot overlap.
    row_group_specs = [
        group_specs[row_start:row_start + 2]
        for row_start in range(0, len(group_specs), 2)
    ]
    row_panel_ids = [
        tuple(panel_id for _label, ids, _color in row_groups for panel_id in ids)
        for row_groups in row_group_specs
    ]
    row_boxes = [group_bbox(panel_ids) for panel_ids in row_panel_ids]
    row_y0 = min(y for _x, y, _w, _h in row_boxes)
    row_y1 = max(y + h for _x, y, _w, h in row_boxes)
    between_row_content_gap_px = border_clearance_px + visible_row_gap_px
    available_h = row_y1 - row_y0 - between_row_content_gap_px * (len(row_boxes) - 1)
    base_row_h = available_h // len(row_boxes)
    new_row_heights = [base_row_h] * len(row_boxes)
    new_row_heights[-1] += available_h - sum(new_row_heights)

    next_y = row_y0
    for row_idx, (panel_ids, (_old_x, old_y, _old_w, old_h), new_h) in enumerate(
        zip(row_panel_ids, row_boxes, new_row_heights)
    ):
        for panel_id in panel_ids:
            x, y, w, h = adjusted[panel_id]
            rel_y = (y - old_y) / old_h
            rel_h = h / old_h
            adjusted[panel_id] = (
                x,
                int(round(next_y + rel_y * new_h)),
                w,
                int(round(rel_h * new_h)),
            )
        if row_idx < len(row_boxes) - 1:
            next_y += new_h + between_row_content_gap_px

    # Apply the same clearance between the left and right groups while
    # retaining a tiny visible gap between their border strokes.
    between_group_content_gap_px = border_clearance_px + visible_group_gap_px
    for row_start in range(0, len(group_specs), 2):
        _left_label, left_ids, _left_color = group_specs[row_start]
        left_x, left_y, left_w, left_h = group_bbox(left_ids)
        if row_start + 1 >= len(group_specs):
            place_3_panel_group(left_ids, left_x, left_y, left_w, left_h)
            continue

        _right_label, right_ids, _right_color = group_specs[row_start + 1]
        right_x, right_y, right_w, right_h = group_bbox(right_ids)
        row_x = min(left_x, right_x)
        row_right = max(left_x + left_w, right_x + right_w)
        row_w = row_right - row_x
        new_left_w = (row_w - between_group_content_gap_px) // 2
        new_right_w = row_w - between_group_content_gap_px - new_left_w
        place_3_panel_group(left_ids, row_x, left_y, new_left_w, left_h)
        place_3_panel_group(
            right_ids,
            row_x + new_left_w + between_group_content_gap_px,
            right_y,
            new_right_w,
            right_h,
        )

    return adjusted


ir25a_orco_border_thickness = 12
ir25a_orco_panel_gap_px = inches_to_px(0.02, DPI)
ir25a_orco_border_gap_px = inches_to_px(0.02, DPI)
panel_layout = add_gaps_to_ir25a_orco_groups(
    panel_layout,
    ir25a_orco_group_specs,
    ir25a_orco_panel_gap_px,
    border_thickness=ir25a_orco_border_thickness,
    visible_group_gap_px=ir25a_orco_border_gap_px,
    visible_row_gap_px=ir25a_orco_border_gap_px,
)

ir25a_orco_group_rows = [
    ir25a_orco_group_specs[row_start:row_start + 2]
    for row_start in range(0, len(ir25a_orco_group_specs), 2)
]
ir25a_orco_row_panel_ids = {
    f"row{row_idx}": tuple(
        panel_id
        for _group_label, group_panel_ids, _border_color in row_groups
        for panel_id in group_panel_ids
    )
    for row_idx, row_groups in enumerate(ir25a_orco_group_rows)
}

ir25a_orco_row_labels = {
    "row0-row1": "Ir25a and Orco",
}

canvas = render_artboard(
    fig_w_px, ir25a_orco_fig_h_px, panel_layout, panels, dpi=DPI,
    scale_font_size=60,
    axis_font_size=60,
)

for _group_label, panel_ids, border_color in ir25a_orco_group_specs:
    canvas = add_group_border(
        canvas,
        panel_layout,
        panel_ids,
        color=border_color,
        thickness=ir25a_orco_border_thickness,
    )

draw = ImageDraw.Draw(canvas)
group_label_font = load_font_safe("arial.ttf", 96)
group_label_pad_px = inches_to_px(LABEL_PAD_IN, DPI)

for group_label, panel_ids, _border_color in ir25a_orco_group_specs:
    x0 = min(panel_layout[panel_id][0] for panel_id in panel_ids)
    y0 = min(panel_layout[panel_id][1] for panel_id in panel_ids)
    label_color = panels[panel_ids[0]].get("label_color", "white")
    draw.text((x0 + group_label_pad_px, y0 + group_label_pad_px), group_label, fill=label_color, font=group_label_font)

canvas = register_figure_canvas('Ir25a and Orco', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure S4: Ir25a and Ir8a and Ir76b


In [ ]:
scale_bar_thickness = 8
ir25a_ir8a_ir76b_fig_h_px = int(round(fig_h_px * 2 / 4))
ir25a_ir8a_ir76b_placeholder_color = "#999999"

ir25a_ir8a_ir76b_channel_label_style = {
    "x": 0.94,
    "y": 0.86,
    "size": 50,
    "anchor": "right",
}

# Add image paths for each channel when the exported panels are ready.
ir25a_ir8a_ir76b_channel_groups = [
    {
        "label": "A",
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5, "rotate": 0},
            "scale_bar": {"full_px": 3460, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        # Applied to every channel panel in row A.
        "annotations": [
            # {"text": "*", "x": 0.50, "y": 0.50, "size": 70, "color": "white"},
            # {"type": "arrow", "x": 0.50, "y": 0.50, "angle": -45, "length": 0.18, "color": "white", "width": 8},
        ],
        # Applied only to the named panel.
        "panel_annotations": {
            "A1": [],
            "A2": [],
            "A3": [],
            "A4": [],
        },
        "channels": [
            ("A1", "Composite", r"D:\larval chemosensory\Experiment-1374\ROI 01 - maxillary - z004 of 8\Experiment-1374_z003_roi01_combined_ch0_Ir25a_ch1_Ir76b_ch2_Ir8a_ch3_DAPI-T4.png", "white"),
            ("A2", "Ir25a", r"D:\larval chemosensory\Experiment-1374\ROI 01 - maxillary - z004 of 8\Experiment-1374_z003_roi01_ch0_Ir25a_ch3_DAPI-T4.png", "cyan"),
            ("A3", "Ir8a", r"D:\larval chemosensory\Experiment-1374\ROI 01 - maxillary - z004 of 8\Experiment-1374_z003_roi01_ch2_Ir8a_ch3_DAPI-T4.png", "green"),
            ("A4", "Ir76b", r"D:\larval chemosensory\Experiment-1374\ROI 01 - maxillary - z004 of 8\Experiment-1374_z003_roi01_ch1_Ir76b_ch3_DAPI-T4.png", "yellow"),
        ],
    },
    {
        "label": "B",
        "channel_labels": False,
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5, "rotate": 0},
           
        "scale_bar": {"full_px": 769, "orig_um": 20, "new_um": 15, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            {"text": "*", "x": 0.38, "y": 0.78, "color": "white", "size": 170},
            #{"type": "arrow", "x": 0.54, "y": 0.41, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        # Applied only to the named panel.
        "panel_annotations": {
            "B1": [],
            "B2": [],
            "B3": [],
            "B4": [],
        },
        "channels": [
            ("B1", "Composite", r"D:\larval chemosensory\Experiment-1374\ROI 01 - Maxillary Ir76b - z006 of 8\Experiment-1374_z005_roi01_combined_ch0_Ir25a_ch1_Ir76b_ch2_Ir8a_ch3_DAPI-T4.png", "white"),
            ("B2", "Ir25a", r"D:\larval chemosensory\Experiment-1374\ROI 01 - Maxillary Ir76b - z006 of 8\Experiment-1374_z005_roi01_ch0_Ir25a_ch3_DAPI-T4.png", "cyan"),
            ("B3", "Ir8a", r"D:\larval chemosensory\Experiment-1374\ROI 01 - Maxillary Ir76b - z006 of 8\Experiment-1374_z005_roi01_ch2_Ir8a_ch3_DAPI-T4.png", "green"),
            ("B4", "Ir76b", r"D:\larval chemosensory\Experiment-1374\ROI 01 - Maxillary Ir76b - z006 of 8\Experiment-1374_z005_roi01_ch1_Ir76b_ch3_DAPI-T4.png", "yellow"),
        ],
    },
]

panels = {}
ir25a_ir8a_ir76b_group_specs = []

for group in ir25a_ir8a_ir76b_channel_groups:
    panel_ids = []
    for panel_id, channel_name, image_path, channel_color in group["channels"]:
        channel_label = None
        if group.get("channel_labels", True):
            channel_label = channel_text_label(
                channel_name,
                color=channel_color,
                **ir25a_ir8a_ir76b_channel_label_style,
            )
        panels[panel_id] = make_channel_panel(
            image_path,
            group["view"],
            MAXILLA_BORDER_COLOR,
            channel_label=channel_label,
        )
        panels[panel_id]["label"] = ""
        panels[panel_id].pop("border", None)
        group_scale_bar = group.get("scale_bar")
        if group_scale_bar:
            panels[panel_id]["scale_bar"] = group_scale_bar.copy()
        else:
            panels[panel_id].pop("scale_bar", None)
        panels[panel_id]["annotations"].extend(group.get("annotations", []))
        panels[panel_id]["annotations"].extend(
            group.get("panel_annotations", {}).get(panel_id, [])
        )
        panel_ids.append(panel_id)
    ir25a_ir8a_ir76b_group_specs.append((group["label"], tuple(panel_ids)))

ir25a_ir8a_ir76b_gap_px = inches_to_px(0.03, DPI)


def build_ir25a_ir8a_ir76b_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, group_specs, gap_px
):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    row_h = (inner_h - gap_px) // 2
    row_heights = (row_h, inner_h - gap_px - row_h)
    layout = {}

    for row_index, ((_group_label, panel_ids), panel_h) in enumerate(
        zip(group_specs, row_heights)
    ):
        y = mt if row_index == 0 else mt + row_heights[0] + gap_px
        available_w = inner_w - gap_px * (len(panel_ids) - 1)
        widths = [available_w // len(panel_ids)] * len(panel_ids)
        widths[-1] += available_w - sum(widths)
        x = ml
        for panel_index, (panel_id, panel_w) in enumerate(zip(panel_ids, widths)):
            layout[panel_id] = (x, y, panel_w, panel_h)
            if panel_index < len(panel_ids) - 1:
                x += panel_w + gap_px

    return layout


panel_layout = build_ir25a_ir8a_ir76b_layout(
    fig_w_px,
    ir25a_ir8a_ir76b_fig_h_px,
    ml,
    mr,
    mt,
    mb,
    ir25a_ir8a_ir76b_group_specs,
    ir25a_ir8a_ir76b_gap_px,
)

canvas = render_artboard(
    fig_w_px,
    ir25a_ir8a_ir76b_fig_h_px,
    panel_layout,
    panels,
    dpi=DPI,
    scale_font_size=60,
    axis_font_size=60,
)

# One maxilla-colored border encloses the complete two-row figure.
all_ir25a_ir8a_ir76b_panel_ids = tuple(panel_layout)
canvas = add_group_border(
    canvas,
    panel_layout,
    all_ir25a_ir8a_ir76b_panel_ids,
    color=MAXILLA_BORDER_COLOR,
    thickness=12,
)

draw = ImageDraw.Draw(canvas)
group_label_font = load_font_safe("arial.ttf", 96)
group_label_pad_px = inches_to_px(LABEL_PAD_IN, DPI)

for group_label, panel_ids in ir25a_ir8a_ir76b_group_specs:
    x0 = min(panel_layout[panel_id][0] for panel_id in panel_ids)
    y0 = min(panel_layout[panel_id][1] for panel_id in panel_ids)
    draw.text(
        (x0 + group_label_pad_px, y0 + group_label_pad_px),
        group_label,
        fill="white",
        font=group_label_font,
    )

canvas = register_figure_canvas('Ir25a and Ir8a and Ir76b', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Ir25a and Orco and Gr25


In [ ]:
scale_bar_thickness = 8
ir25a_orco_gr25_fig_h_px = int(round(fig_h_px * 2 / 4))
ir25a_orco_gr25_placeholder_color = "#999999"

ir25a_orco_gr25_channel_label_style = {
    "x": 0.94,
    "y": 0.86,
    "size": 50,
    "anchor": "right",
}

# Add image paths for each channel when the exported panels are ready.
ir25a_orco_gr25_channel_groups = [
    {
        "label": "A",
        "view": {"zoom": 1.3, "x": 0.5, "y": 0.2, "rotate": -110},
            "scale_bar": {"full_px": 508, "orig_um": 20, "new_um": 20, "color": "white", "thickness": scale_bar_thickness, "number": False},
        # Applied to every channel panel in row A.
        "annotations": [
            # {"text": "*", "x": 0.50, "y": 0.50, "size": 70, "color": "white"},
            # {"type": "arrow", "x": 0.50, "y": 0.50, "angle": -45, "length": 0.18, "color": "white", "width": 8},
        ],
        # Applied only to the named panel.
        "panel_annotations": {
            "A1": [],
            "A2": [],
            "A3": [],
            "A4": [],
        },
        "channels": [
            ("A1", "Composite", r"D:\larval chemosensory\Experiment-1750\ROI 01 - labium - z080 of 219\Experiment-1750_z079_roi01_combined_ch0_Ir25a_ch1_Gr25_ch2_Orco_ch3_DAPI.png", "white"),
            ("A2", "Ir25a", r"D:\larval chemosensory\Experiment-1750\ROI 01 - labium - z080 of 219\Experiment-1750_z079_roi01_ch0_Ir25a_ch3_DAPI.png", "cyan"),
            ("A3", "Gr25", r"D:\larval chemosensory\Experiment-1750\ROI 01 - labium - z080 of 219\Experiment-1750_z079_roi01_ch1_Gr25_ch3_DAPI.png", "yellow"),
            ("A4", "Orco", r"D:\larval chemosensory\Experiment-1750\ROI 01 - labium - z080 of 219\Experiment-1750_z079_roi01_ch2_Orco_ch3_DAPI.png", "green"),
        ],
    },
    {
        "label": "B",
        "channel_labels": False,
        "view": {"zoom": 3.0, "x": 0.62, "y": 0.35, "rotate": -110},
           
        "scale_bar": {"full_px": 508, "orig_um": 20, "new_um": 10, "color": "white", "thickness": scale_bar_thickness, "number": False},
        "annotations": [
            #{"text": "*", "x": 0.38, "y": 0.78, "color": "white", "size": 170},
            #{"type": "arrow", "x": 0.54, "y": 0.41, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        # Applied only to the named panel.
        "panel_annotations": {
            "B1": [],
            "B2": [],
            "B3": [],
            "B4": [],
        },
        "channels": [
            ("B1", "Composite", r"D:\larval chemosensory\Experiment-1750\ROI 01 - labium - z080 of 219\Experiment-1750_z079_roi01_combined_ch0_Ir25a_ch1_Gr25_ch2_Orco_ch3_DAPI.png", "white"),
            ("B2", "Ir25a", r"D:\larval chemosensory\Experiment-1750\ROI 01 - labium - z080 of 219\Experiment-1750_z079_roi01_ch0_Ir25a_ch3_DAPI.png", "cyan"),
            ("B3", "Gr25", r"D:\larval chemosensory\Experiment-1750\ROI 01 - labium - z080 of 219\Experiment-1750_z079_roi01_ch1_Gr25_ch3_DAPI.png", "yellow"),
            ("B4", "Orco", r"D:\larval chemosensory\Experiment-1750\ROI 01 - labium - z080 of 219\Experiment-1750_z079_roi01_ch2_Orco_ch3_DAPI.png", "green"),
        ],
    },
]

panels = {}
ir25a_orco_gr25_group_specs = []

for group in ir25a_orco_gr25_channel_groups:
    panel_ids = []
    for panel_id, channel_name, image_path, channel_color in group["channels"]:
        channel_label = None
        if group.get("channel_labels", True):
            channel_label = channel_text_label(
                channel_name,
                color=channel_color,
                **ir25a_orco_gr25_channel_label_style,
            )
        panels[panel_id] = make_channel_panel(
            image_path,
            group["view"],
            MAXILLA_BORDER_COLOR,
            channel_label=channel_label,
        )
        panels[panel_id]["label"] = ""
        panels[panel_id].pop("border", None)
        group_scale_bar = group.get("scale_bar")
        if group_scale_bar:
            panels[panel_id]["scale_bar"] = group_scale_bar.copy()
        else:
            panels[panel_id].pop("scale_bar", None)
        panels[panel_id]["annotations"].extend(group.get("annotations", []))
        panels[panel_id]["annotations"].extend(
            group.get("panel_annotations", {}).get(panel_id, [])
        )
        panel_ids.append(panel_id)
    ir25a_orco_gr25_group_specs.append((group["label"], tuple(panel_ids)))

ir25a_orco_gr25_gap_px = inches_to_px(0.03, DPI)


def build_ir25a_orco_gr25_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, group_specs, gap_px
):
    inner_w = fig_w_px - ml - mr
    inner_h = fig_h_px - mt - mb
    row_h = (inner_h - gap_px) // 2
    row_heights = (row_h, inner_h - gap_px - row_h)
    layout = {}

    for row_index, ((_group_label, panel_ids), panel_h) in enumerate(
        zip(group_specs, row_heights)
    ):
        y = mt if row_index == 0 else mt + row_heights[0] + gap_px
        available_w = inner_w - gap_px * (len(panel_ids) - 1)
        widths = [available_w // len(panel_ids)] * len(panel_ids)
        widths[-1] += available_w - sum(widths)
        x = ml
        for panel_index, (panel_id, panel_w) in enumerate(zip(panel_ids, widths)):
            layout[panel_id] = (x, y, panel_w, panel_h)
            if panel_index < len(panel_ids) - 1:
                x += panel_w + gap_px

    return layout


panel_layout = build_ir25a_orco_gr25_layout(
    fig_w_px,
    ir25a_orco_gr25_fig_h_px,
    ml,
    mr,
    mt,
    mb,
    ir25a_orco_gr25_group_specs,
    ir25a_orco_gr25_gap_px,
)

canvas = render_artboard(
    fig_w_px,
    ir25a_orco_gr25_fig_h_px,
    panel_layout,
    panels,
    dpi=DPI,
    scale_font_size=60,
    axis_font_size=60,
)

# One maxilla-colored border encloses the complete two-row figure.
all_ir25a_orco_gr25_panel_ids = tuple(panel_layout)
canvas = add_group_border(
    canvas,
    panel_layout,
    all_ir25a_orco_gr25_panel_ids,
    color=MAXILLA_BORDER_COLOR,
    thickness=12,
)

draw = ImageDraw.Draw(canvas)
group_label_font = load_font_safe("arial.ttf", 96)
group_label_pad_px = inches_to_px(LABEL_PAD_IN, DPI)

for group_label, panel_ids in ir25a_orco_gr25_group_specs:
    x0 = min(panel_layout[panel_id][0] for panel_id in panel_ids)
    y0 = min(panel_layout[panel_id][1] for panel_id in panel_ids)
    draw.text(
        (x0 + group_label_pad_px, y0 + group_label_pad_px),
        group_label,
        fill="white",
        font=group_label_font,
    )

canvas = register_figure_canvas('Ir25a and Orco and Gr25', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure S9: Gr25 and Gr6


In [ ]:
scale_bar_thickness = 8
figure4_grs_placeholder_color = "#999999"

separate_channel_label_style_ab = make_channel_label_style(x=0.95, y=0.70, size=50)

separate_channel_label_style_cj = make_channel_label_style(x=0.95, y=0.70, size=50)

separate_channel_groups = [
    {
        "label": "A",
        "channel_labels": True,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": FRONS_BORDER_COLOR,
             "view": {"zoom": 1.7, "x": 0.5, "y": 0.58, "rotate": 0},
        "annotations": [
            #{"text": "*", "x": 0.40, "y": 0.41, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.54, "y": 0.41, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("A1", "", r"D:\larval chemosensory\Grs\20260601\6\6\669\Snap-669_composite.png", "white"),
            ("A2", "DAPI", r"D:\larval chemosensory\Grs\20260601\6\6\669\Snap-669_DAPI.png", "white"),
            ("A3", "Gr25", r"D:\larval chemosensory\Grs\20260601\6\6\669\Snap-669_Gr25.png", "magenta"),
            ("A4", "Gr6", r"D:\larval chemosensory\Grs\20260601\6\6\669\Snap-669_Gr6.png", "yellow"),
        ],
    },
    {
        "label": "B",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 2.0, "x": 0.64, "y": 0.40, "rotate": -50},
        "annotations": [
            #{"text": "*", "x": 0.54, "y": 0.41, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.54, "y": 0.36, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("B1", "Composite", r"D:\larval chemosensory\Grs\20260601\6\6\665\Snap-665_composite.png", "white"),
            ("B2", "DAPI", r"D:\larval chemosensory\Grs\20260601\6\6\665\Snap-665_DAPI.png", "cyan"),
            ("B3", "Gr25", r"D:\larval chemosensory\Grs\20260601\6\6\665\Snap-665_Gr25.png", "magenta"),
            ("B4", "Gr6", r"D:\larval chemosensory\Grs\20260601\6\6\665\Snap-665_Gr6.png", "white"),
        ],
    },
    {
        "label": "C",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 1.5, "x": 0.5, "y": 0.55, "rotate": 90},
        "annotations": [
            #{"text": "*", "x": 0.46, "y": 0.2, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.54, "y": 0.41, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("E1", "", r"D:\larval chemosensory\Grs\20260601\6\labrum_otherthan_main\Snap-674_composite.png", "white"),
            ("E2", "", r"D:\larval chemosensory\Grs\20260601\6\labrum_otherthan_main\Snap-674_DAPI.png", "cyan"),
            ("E3", "", r"D:\larval chemosensory\Grs\20260601\6\labrum_otherthan_main\Snap-674_Gr25.png", "magenta"),
            ("E4", "", r"D:\larval chemosensory\Grs\20260601\6\labrum_otherthan_main\Snap-674_Gr6.png", "yellow"),
        ],
    },
    {
        "label": "D",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": LABRUM_BORDER_COLOR,
        "view": {"zoom": 1.8, "x": 0.5, "y": 0.50, "rotate": 90},
        "annotations": [
            #{"text": "*", "x": 0.46, "y": 0.2, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.54, "y": 0.41, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("F1", "", r"D:\larval chemosensory\Grs\20260601\6\labrum_otherthan_main\Snap-676_composite.png", "white"),
            ("F2", "", r"D:\larval chemosensory\Grs\20260601\6\labrum_otherthan_main\Snap-676_DAPI.png", "cyan"),
            ("F3", "", r"D:\larval chemosensory\Grs\20260601\6\labrum_otherthan_main\Snap-676_Gr25.png", "magenta"),
            ("F4", "", r"D:\larval chemosensory\Grs\20260601\6\labrum_otherthan_main\Snap-676_Gr6.png", "yellow"),
        ],
    },
    {
        "label": "E",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 1.4, "x": 0.1, "y": 0.5, "rotate": 45},
        "annotations": [
            #{"text": "*", "x": 0.28, "y": 0.60, "color": "#E69F00", "size": 70},
            #{"text": "*", "x": 0.75, "y": 0.54, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.76, "y": 0.66, "angle": -165, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("C1", "", r"D:\larval chemosensory\Grs\20260601\6\max\Snap-697_composite.png", "white"),
            ("C2", "", r"D:\larval chemosensory\Grs\20260601\6\max\Snap-697_DAPI.png", "cyan"),
            ("C3", "", r"D:\larval chemosensory\Grs\20260601\6\max\Snap-697_Gr25.png", "magenta"),
            ("C4", "", r"D:\larval chemosensory\Grs\20260601\6\max\Snap-697_Gr6.png", "yellow"),
        ],
    },
    {
        "label": "F",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": MAXILLA_BORDER_COLOR,
        "view": {"zoom": 1.8, "x": 0.7, "y": 0.50, "rotate": 45},
        "annotations": [
            #{"text": "*", "x": 0.66, "y": 0.64, "color": "#E69F00", "size": 70},
            #{"text": "*", "x": 0.77, "y": 0.54, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.33, "y": 0.12, "angle": -165, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("D1", "", r"D:\larval chemosensory\Grs\20260601\6\max\Snap-699_composite.png", "white"),
            ("D2", "", r"D:\larval chemosensory\Grs\20260601\6\max\Snap-699_DAPI.png", "cyan"),
            ("D3", "", r"D:\larval chemosensory\Grs\20260601\6\max\Snap-699_Gr25.png", "magenta"),
            ("D4", "", r"D:\larval chemosensory\Grs\20260601\6\max\Snap-699_Gr6.png", "yellow"),
        ],
    },
    {
        "label": "G",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 1.5, "x": 0.5, "y": 0.8, "rotate": 0},
        "annotations": [
            #{"text": "*", "x": 0.46, "y": 0.21, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.4, "y": 0.71, "angle": -145, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("G1", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labium\Snap-682_composite.png", "white"),
            ("G2", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labium\Snap-682_DAPI.png", "cyan"),
            ("G3", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labium\Snap-682_Gr25.png", "magenta"),
            ("G4", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labium\Snap-682_Gr6.png", "white"),
        ],
    },
    {
        "label": "H",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 1.5, "x": 0.5, "y": 0.9, "rotate": 0},
        "annotations": [
            #{"text": "*", "x": 0.37, "y": 0.21, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.4, "y": 0.71, "angle": -165, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("H1", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labium\Snap-687_composite.png", "white"),
            ("H2", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labium\Snap-687_DAPI.png", "cyan"),
            ("H3", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labium\Snap-687_Gr25.png", "magenta"),
            ("H4", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labium\Snap-687_Gr6.png", "white"),
        ],
    },
    {
        "label": "I",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 2.0, "x": 0.27, "y": 0.5, "rotate": 0},
        "annotations": [
            #{"text": "*", "x": 0.52, "y": 0.21, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.4, "y": 0.71, "angle": -165, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("I1", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labiumother\prelabium\Snap-695_composite.png", "white"),
            ("I2", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labiumother\prelabium\Snap-695_DAPI.png", "cyan"),
            ("I3", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labiumother\prelabium\Snap-695_Gr25.png", "magenta"),
            ("I4", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labiumother\prelabium\Snap-695_Gr6.png", "yellow"),
        ],
    },
    {
        "label": "J",
        "channel_labels": False,
        "scale_bar": {"full_px": 718, "orig_um": 25, "new_um": 5, "color": "white", "thickness": 12, "number": False},
        "border_color": LABIUM_BORDER_COLOR,
        "view": {"zoom": 2.0, "x": 0.8, "y": 0.55, "rotate": 0},
        "annotations": [
            #{"text": "*", "x": 0.42, "y": 0.68, "color": "#E69F00", "size": 70},
            #{"type": "arrow", "x": 0.76, "y": 0.66, "angle": -165, "length": 0, "color": "#E69F00", "width": 10},
        ],
        "channels": [
            ("J1", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labiumother\labium2\Snap-693_composite.png", "white"),
            ("J2", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labiumother\labium2\Snap-693_DAPI.png", "cyan"),
            ("J3", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labiumother\labium2\Snap-693_Gr25.png", "magenta"),
            ("J4", "", r"D:\larval chemosensory\Grs\20260601\6\labium\labiumother\labium2\Snap-693_Gr6.png", "yellow"),
        ],
    },
]

panels = {}
separate_channel_group_specs = []

for group in separate_channel_groups:
    panel_ids = []
    group_scale_bar = group.get("scale_bar")
    show_channel_labels = group.get("channel_labels", group["label"] in {"A", "C"})
    channel_label_style = (
        separate_channel_label_style_ab
        if group["label"] in {"A", "B"}
        else separate_channel_label_style_cj
    )

    for panel_id, channel_name, image_path, channel_color in group["channels"]:
        channel_label = None
        if show_channel_labels and channel_name:
            channel_label = channel_text_label(
                channel_name,
                color=channel_color,
                **channel_label_style,
            )
        panels[panel_id] = make_channel_panel(
            image_path,
            group.get("view", {'zoom': 1.0, 'x': 0.5, 'y': 0.5, 'rotate': 0}),
            group.get("border_color", "#999999"),
            channel_label=channel_label,
        )
        if group_scale_bar:
            panels[panel_id]["scale_bar"] = group_scale_bar.copy()
        else:
            panels[panel_id].pop("scale_bar", None)
        panels[panel_id]["annotations"].extend(group.get("annotations", []))
        panel_ids.append(panel_id)

    separate_channel_group_specs.append(
        (group["label"], tuple(panel_ids), group.get("border_color", "#999999"))
    )

for _group_label, panel_ids, _border_color in separate_channel_group_specs:
    for panel_id in panel_ids:
        panels[panel_id]["label"] = ""
        panels[panel_id].pop("border", None)

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = 0

panel_layout = build_variable_channel_group_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px,
    [panel_ids for _group_label, panel_ids, _border_color in separate_channel_group_specs],
)

figure4_grs_border_thickness = 12

def add_gaps_to_figure4_grs_2x2_groups(
    layout,
    group_specs,
    internal_gap_px,
    border_thickness=12,
    between_group_visible_gap_by_row=None,
    between_group_content_gap_by_row=None,
    between_row_visible_gap_px=0,
    between_row_content_gap_by_pair=None,
):
    adjusted = dict(layout)
    between_group_visible_gap_by_row = between_group_visible_gap_by_row or {}
    between_group_content_gap_by_row = between_group_content_gap_by_row or {}
    between_row_content_gap_by_pair = between_row_content_gap_by_pair or {}
    border_clearance_px = int(border_thickness) * 2

    def group_bbox(panel_ids):
        x0 = min(adjusted[panel_id][0] for panel_id in panel_ids)
        y0 = min(adjusted[panel_id][1] for panel_id in panel_ids)
        x1 = max(adjusted[panel_id][0] + adjusted[panel_id][2] for panel_id in panel_ids)
        y1 = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in panel_ids)
        return x0, y0, x1 - x0, y1 - y0

    def place_2x2_group(panel_ids, x, y, w, h):
        left_w = (w - internal_gap_px) // 2
        right_w = w - internal_gap_px - left_w
        top_h = (h - internal_gap_px) // 2
        bottom_h = h - internal_gap_px - top_h
        top_left_id, top_right_id, bottom_left_id, bottom_right_id = panel_ids
        adjusted[top_left_id] = (x, y, left_w, top_h)
        adjusted[top_right_id] = (x + left_w + internal_gap_px, y, right_w, top_h)
        adjusted[bottom_left_id] = (x, y + top_h + internal_gap_px, left_w, bottom_h)
        adjusted[bottom_right_id] = (x + left_w + internal_gap_px, y + top_h + internal_gap_px, right_w, bottom_h)

    row_group_specs = [group_specs[row_start:row_start + 2] for row_start in range(0, len(group_specs), 2)]
    row_panel_ids = [
        tuple(panel_id for _label, ids, _color in row_groups for panel_id in ids)
        for row_groups in row_group_specs
    ]
    row_boxes = [group_bbox(panel_ids) for panel_ids in row_panel_ids]
    row_y0 = min(y for _x, y, _w, _h in row_boxes)
    row_y1 = max(y + h for _x, y, _w, h in row_boxes)
    row_gap_content_by_pair = [
        between_row_content_gap_by_pair.get(row_idx, border_clearance_px + between_row_visible_gap_px)
        for row_idx in range(len(row_boxes) - 1)
    ]
    available_h = row_y1 - row_y0 - sum(row_gap_content_by_pair)
    base_row_h = available_h // len(row_boxes)
    new_row_heights = [base_row_h] * len(row_boxes)
    new_row_heights[-1] += available_h - sum(new_row_heights)
    next_y = row_y0
    for row_idx, (panel_ids, (_old_x, old_y, _old_w, old_h), new_h) in enumerate(zip(row_panel_ids, row_boxes, new_row_heights)):
        for panel_id in panel_ids:
            x, y, w, h = adjusted[panel_id]
            rel_y = (y - old_y) / old_h
            rel_h = h / old_h
            adjusted[panel_id] = (x, int(round(next_y + rel_y * new_h)), w, int(round(rel_h * new_h)))
        if row_idx < len(row_gap_content_by_pair):
            next_y += new_h + row_gap_content_by_pair[row_idx]

    for row_start in range(0, len(group_specs), 2):
        row_idx = row_start // 2
        _left_label, left_ids, _left_color = group_specs[row_start]
        left_x, left_y, left_w, left_h = group_bbox(left_ids)
        if row_start + 1 >= len(group_specs):
            place_2x2_group(left_ids, left_x, left_y, left_w, left_h)
            continue

        _right_label, right_ids, _right_color = group_specs[row_start + 1]
        right_x, right_y, right_w, right_h = group_bbox(right_ids)
        if row_idx in between_group_content_gap_by_row:
            between_group_gap_px = between_group_content_gap_by_row[row_idx]
        else:
            visible_gap_px = between_group_visible_gap_by_row.get(row_idx, internal_gap_px)
            between_group_gap_px = border_clearance_px + visible_gap_px
        row_x = min(left_x, right_x)
        row_right = max(left_x + left_w, right_x + right_w)
        row_w = row_right - row_x
        new_left_w = (row_w - between_group_gap_px) // 2
        new_right_w = row_w - between_group_gap_px - new_left_w
        place_2x2_group(left_ids, row_x, left_y, new_left_w, left_h)
        place_2x2_group(right_ids, row_x + new_left_w + between_group_gap_px, right_y, new_right_w, right_h)

    return adjusted


figure4_grs_panel_gap_px = inches_to_px(0.02, DPI)
panel_layout = add_gaps_to_figure4_grs_2x2_groups(
    panel_layout,
    separate_channel_group_specs,
    figure4_grs_panel_gap_px,
    border_thickness=figure4_grs_border_thickness,
    between_group_visible_gap_by_row={0: 0, 1: figure4_grs_panel_gap_px, 2: figure4_grs_panel_gap_px, 3: figure4_grs_panel_gap_px, 4: figure4_grs_panel_gap_px},
    between_group_content_gap_by_row={1: figure4_grs_panel_gap_px, 2: figure4_grs_panel_gap_px, 3: figure4_grs_panel_gap_px, 4: figure4_grs_panel_gap_px},
    between_row_visible_gap_px=0,
    between_row_content_gap_by_pair={3: figure4_grs_panel_gap_px},
)

figure4_grs_group_rows = [
    separate_channel_group_specs[row_start:row_start + 2]
    for row_start in range(0, len(separate_channel_group_specs), 2)
]
figure4_grs_row_panel_ids = {
    f"row{row_idx}": tuple(
        panel_id
        for _group_label, group_panel_ids, _border_color in row_groups
        for panel_id in group_panel_ids
    )
    for row_idx, row_groups in enumerate(figure4_grs_group_rows)
}

figure4_grs_row_labels = {
    "row0-row4": "Gr25 and Gr6",
}

canvas = render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels, dpi=DPI,
    scale_font_size=60,
    axis_font_size=60,
)

figure4_grs_row_border_colors = {
    "row0": LABIUM_BORDER_COLOR,
    "row1": figure4_grs_placeholder_color,
    "row2": figure4_grs_placeholder_color,
    "row3": figure4_grs_placeholder_color,
    "row4": figure4_grs_placeholder_color,
}

figure4_grs_combined_border_groups = {
    "C-D": {"labels": ("C", "D"), "color": LABRUM_BORDER_COLOR},
    "E-F": {"labels": ("E", "F"), "color": MAXILLA_BORDER_COLOR},
    "G-J": {"labels": ("G", "H", "I", "J"), "color": LABIUM_BORDER_COLOR},
}
figure4_grs_combined_border_members = {
    group_label
    for group_spec in figure4_grs_combined_border_groups.values()
    for group_label in group_spec["labels"]
}

for group_label, panel_ids, border_color in separate_channel_group_specs:
    if group_label in figure4_grs_combined_border_members:
        continue
    canvas = add_group_border(
        canvas,
        panel_layout,
        panel_ids,
        color=border_color,
        thickness=figure4_grs_border_thickness,
    )

for _combined_label, group_spec in figure4_grs_combined_border_groups.items():
    combined_panel_ids = tuple(
        panel_id
        for group_label, panel_ids, _border_color in separate_channel_group_specs
        if group_label in group_spec["labels"]
        for panel_id in panel_ids
    )
    canvas = add_group_border(
        canvas,
        panel_layout,
        combined_panel_ids,
        color=group_spec["color"],
        thickness=figure4_grs_border_thickness,
    )

draw = ImageDraw.Draw(canvas)
group_label_font = load_font_safe("arial.ttf", 96)
group_label_pad_px = inches_to_px(LABEL_PAD_IN, DPI)

for group_label, panel_ids, _border_color in separate_channel_group_specs:
    x0 = min(panel_layout[panel_id][0] for panel_id in panel_ids)
    y0 = min(panel_layout[panel_id][1] for panel_id in panel_ids)
    label_color = panels[panel_ids[0]].get("label_color", "white")
    draw.text((x0 + group_label_pad_px, y0 + group_label_pad_px), group_label, fill=label_color, font=group_label_font)

canvas = register_figure_canvas('Grs (25 and 6)', canvas, panel_configs=panels, panel_layout=panel_layout)

display(canvas)

## Figure S13: Transgenic

In [ ]:
panels = {
    "A": {
        "path": r"D:\larval chemosensory\cartoon_transgenic_plasmid.png",  # Add the full-width row-1 image here.
        "view": {"zoom": 0.95, "x": 0.5, "y": 0.5, "rotate": 0},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label_color": "black",
        "annotations": [],
    },
    "B": {
        "path": r"D:\larval chemosensory\IR25a_GCaMP_lineB_depth.png",  # Lower-left panel (formerly E).
        "view": {"zoom": 0.98, "x": 0.5, "y": 0.5, "rotate": 0},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label_color": "black",
        "annotations": [],
    },
    "C": {
        "path": r"D:\larval chemosensory\IR25a_GCaMP_lineB_karyogram.png",  # Middle panel (formerly B).
        "view": {"zoom": 1.5, "x": 0.01, "y": 0.5, "rotate": 0},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label_color": "black",
        "annotations": [],
    },
    "D": {
        "path": r"D:\larval chemosensory\insertion_14_P22_IR25a_GCaMP_lineB.png",  # Upper-right panel (formerly C).
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5, "rotate": 0},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label_color": "black",
        "annotations": [],
    },
    "E": {
        "path": r"D:\larval chemosensory\insertion_15_P22_IR25a_GCaMP_lineB.png",  # Lower-right panel (formerly D).
        "view": {"zoom": 1.0, "x": 0.5, "y": 0.5, "rotate": 0},
        "fit": "contain",
        "background": "white",
        "padding_color": "white",
        "label_color": "black",
        "annotations": [],
    },
}

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = inches_to_px(COL_GAP_IN, DPI)


def transgenic_contain_height(panel_id, panel_width):
    """Return the uncropped contain-height for a panel at a given width."""
    with Image.open(panels[panel_id]["path"]) as source_image:
        return int(round(panel_width * source_image.height / source_image.width))


# Size the artboard to the images instead of stretching the lower row to the
# default full-page height. Row 2 has three columns: B | C | D/E.
transgenic_inner_w_px = fig_w_px - ml - mr
transgenic_row2_col_w_px = (
    transgenic_inner_w_px - 2 * col_gap_px
) // 3
transgenic_row1_h_px = transgenic_contain_height("A", transgenic_inner_w_px)
transgenic_row2_h_px = max(
    transgenic_contain_height("B", transgenic_row2_col_w_px),
    transgenic_contain_height("C", transgenic_row2_col_w_px),
    transgenic_contain_height("D", transgenic_row2_col_w_px)
    + row_gap_px
    + transgenic_contain_height("E", transgenic_row2_col_w_px),
)
transgenic_fig_h_px = (
    mt + transgenic_row1_h_px + row_gap_px + transgenic_row2_h_px + mb
)

panel_layout = build_mixed_layout(
    fig_w_px, transgenic_fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px,
    [("A",), ("B", "C", "D")],
    row_height_ratios=(transgenic_row1_h_px, transgenic_row2_h_px),
    row_width_fracs=(1.0, 1.0),
)

# Split the right column of row 2 into D above E.
x_d, y_d, w_d, h_d = panel_layout["D"]
d_h = (h_d - row_gap_px) // 2
panel_layout["D"] = (x_d, y_d, w_d, d_h)
panel_layout["E"] = (
    x_d,
    y_d + d_h + row_gap_px,
    w_d,
    h_d - d_h - row_gap_px,
)

canvas = render_artboard(
    fig_w_px, transgenic_fig_h_px, panel_layout, panels, dpi=DPI,
    label_font_size=60,
)
canvas = register_figure_canvas(
    "Transgenic", canvas, panel_configs=panels, panel_layout=panel_layout,
)
display(canvas)

## Figure S12: Central Brain Supplement


In [ ]:
panels = {
    "A": {
        "path": r"D:\larval chemosensory\-00001-03(9)\MIP z1-174\Full image - MIP z1-174\-00001-03_9_mip_z1-174_full_combined_ch1_Red_ch2_Green_ch3_DAPI.png",
        "view": {"zoom": 1.2, "x": 0.5, "y": 0.5, "rotate": 20}, 
        "scale_bar": {"full_px": 3564, "orig_um": 100, "new_um": 25, "color": "white", "thickness": 12, "number": False},
        # "fit": "contain",
        # "background": "black",
        "label_color": "white",
        "annotations": [
            {"text": "SEG", "x": 0.5, "y": 0.75, "size": 70, "color": "white"},
            {"text": "CB", "x": 0.65, "y": 0.25, "size": 100, "color": "white"},
            {"text": "Labr.", "x": 0.02, "y": 0.15, "size": 60, "color": "white"},
            #{"text": "Labm.", "x": 0.02, "y": 0.55, "size": 60, "color": "white"},
            {"text": "Antn.", "x": 0.64, "y": 0.02, "size": 60, "color": "white"},
            {"text": "Max.", "x": 0.02, "y": 0.70, "size": 60, "color": "white"},
            {"text": "GCaMP", "x": 0.02, "y": 0.88, "size": 70, "color": "yellow"},
        ],
    },
    "B": {
        "path": r"D:\larval chemosensory\-00001-03(9)\MIP z84-112\Full image - MIP z84-112\-00001-03_9_mip_z84-112_full_ch1_Red_ch3_DAPI.png",
        "view": {"zoom": 1.2, "x": 0.5, "y": 0.5, "rotate": 20}, 
        # "fit": "contain",
        # "background": "white",
        "scale_bar": {"full_px": 3564, "orig_um": 100, "new_um": 25, "color": "white", "thickness": 12, "number": False},
        "padding_color": "white",
        "label_color": "white",
                "annotations": [
            {"text": "SEG", "x": 0.5, "y": 0.75, "size": 70, "color": "white"},
            {"text": "CB", "x": 0.65, "y": 0.25, "size": 100, "color": "white"},
            {"text": "Labr.", "x": 0.02, "y": 0.15, "size": 60, "color": "white"},
            #{"text": "Labm.", "x": 0.02, "y": 0.55, "size": 60, "color": "white"},
            {"text": "Antn.", "x": 0.64, "y": 0.02, "size": 60, "color": "white"},

            {"text": "Max.", "x": 0.02, "y": 0.70, "size": 60, "color": "white"},
            #{"text": "GCaMP", "x": 0.05, "y": 0.88, "size": 70, "color": "yellow"},
        ],

    },
    "D": {
        "path": r"D:\larval chemosensory\-00001-03(9)\MIP z100-144\Full image - MIP z100-144\-00001-03_9_mip_z100-144_full_ch1_Red_ch2_Green.png",
        "view": {"zoom": 6.0, "x": 0.48, "y": 0.20, "rotate": 30},
        "label_color": "white",
        "scale_bar": {"full_px": 3564, "orig_um": 100, "new_um": 10, "color": "white", "thickness": 12, "number": False},
        "annotations": [
            {"text": "Sensilla", "x": 0.18, "y": 0.06, "size": 70, "color": "white"},
            {"text": "Antenna", "x": 0.48, "y": 0.54, "size": 75, "color": "white"},
             #{"text": "CB", "x": 0.42, "y": 0.05, "size": 90, "color": "white"},
              #{"text": "Brain", "x": 0.41, "y": 0.35, "size": 80, "color": "white"},
              #{"text": "Towards Mouthparts", "x": 0.23, "y": 0.05, "size": 60, "color": "white"},
            #{"text": "GCaMP", "x": 0.05, "y": 0.88, "size": 70, "color": "white"},
        ],
        #"border": {"color": "#B85C4F", "thickness": 12},
    },

    "C": {
        "path": r"D:\larval chemosensory\-00001-03(9)\MIP z84-112\Full image - MIP z84-112\-00001-03_9_mip_z84-112_full_ch1_Red_ch3_DAPI.png",
        "view": {"zoom": 5, "x": 0.5, "y": 0.33, "rotate": 20}, 
        # "fit": "contain",
        # "background": "white",
        "scale_bar": {"full_px": 3564, "orig_um": 100, "new_um": 10, "color": "white", "thickness": 12, "number": False},
        "padding_color": "white",
        "label_color": "white",
        "annotations": [
            {"text": "From", "x": 0.4, "y": 0.80, "size": 60, "color": "white"},
            {"text": "SEG", "x": 0.4, "y": 0.88, "size": 60, "color": "white"},
            #{"text": "CB", "x": 0.65, "y": 0.25, "size": 100, "color": "white"},
            {"text": "From.", "x": 0.05, "y": 0.14, "size": 60, "color": "white"},
            {"text": "Labr.", "x": 0.05, "y": 0.22, "size": 60, "color": "white"},
            #{"text": "Labm.", "x": 0.02, "y": 0.55, "size": 60, "color": "white"},
            {"text": "From", "x": 0.63, "y": 0.05, "size": 60, "color": "white"},
            {"text": "Antn.", "x": 0.63, "y": 0.13, "size": 60, "color": "white"},
           #{"text": "Max.", "x": 0.02, "y": 0.70, "size": 60, "color": "white"},
           #{"text": "GCaMP", "x": 0.05, "y": 0.88, "size": 70, "color": "yellow"},
        ],
        },



}

row_gap_px = inches_to_px(ROW_GAP_IN, DPI)
col_gap_px = inches_to_px(COL_GAP_IN, DPI)

# E and F are layout-only placeholders. Keeping this empty third row preserves
# the original panel sizes and full artboard instead of stretching A-D.
row_panel_orders = [
    ("A", "B"),
    ("C", "D"),
    ("E", "F"),
]

panel_layout = build_mixed_layout(
    fig_w_px, fig_h_px, ml, mr, mt, mb, row_gap_px, col_gap_px, row_panel_orders,
    row_height_ratios=(1, 1, 1),
    row_width_fracs=(1.0, 1.0, 1.0),
)


def close_internal_2col_gaps_small_space(layout, row_panel_orders, min_gap_px=6):
    adjusted = dict(layout)

    for row_ids in row_panel_orders:
        left_id, right_id = row_ids
        lx, ly, lw, lh = adjusted[left_id]
        rx, ry, rw, rh = adjusted[right_id]
        gap = rx - (lx + lw)
        if gap > min_gap_px:
            close_px = gap - min_gap_px
            left_extra = close_px // 2
            right_extra = close_px - left_extra
            adjusted[left_id] = (lx, ly, lw + left_extra, lh)
            adjusted[right_id] = (rx - right_extra, ry, rw + right_extra, rh)

    for top_row, bottom_row in zip(row_panel_orders, row_panel_orders[1:]):
        top_bottom = max(adjusted[panel_id][1] + adjusted[panel_id][3] for panel_id in top_row)
        bottom_top = min(adjusted[panel_id][1] for panel_id in bottom_row)
        gap = bottom_top - top_bottom
        if gap > min_gap_px:
            close_px = gap - min_gap_px
            top_extra = close_px // 2
            bottom_extra = close_px - top_extra
            for panel_id in top_row:
                x, y, w, h = adjusted[panel_id]
                adjusted[panel_id] = (x, y, w, h + top_extra)
            for panel_id in bottom_row:
                x, y, w, h = adjusted[panel_id]
                adjusted[panel_id] = (x, y - bottom_extra, w, h + bottom_extra)

    return adjusted


panel_layout = close_internal_2col_gaps_small_space(panel_layout, row_panel_orders, min_gap_px=6)

# Drop the empty E/F slots only after their row has reserved its original space.
panel_layout = {
    panel_id: box
    for panel_id, box in panel_layout.items()
    if panel_id in panels
}

canvas = render_artboard(
    fig_w_px, fig_h_px, panel_layout, panels, dpi=DPI,
)

# B and C use the same source image. This box is calculated automatically from
# C's current view, so it follows C whenever its zoom or position is edited.
canvas = add_zoom_box_from_zoomed_panel(
    canvas, panel_layout, panels,
    base_id="B", zoom_id="C", color="white", width=8,
)

# Second, manually editable box. x/y set its upper-left corner and w/h set its
# size, all as fractions of the selected panel.
editable_box = {
    "panel_id": "B",
    "x": 0.52,
    "y": 0.035,
    "w": 0.10,
    "h": 0.10,
    "color": "green",
    "thickness": 8,
}
canvas = add_panel_boxes(canvas, panel_layout, [editable_box])


def add_panel_image_annotation(canvas, panel_layout, panel_id, cfg):
    x0, y0, pw, ph = panel_layout[panel_id]
    ann_w = int(round(pw * cfg.get("w_frac", 0.22)))
    ann_h = int(round(ph * cfg.get("h_frac", 0.22)))

    annotation = Image.open(cfg["path"]).convert("RGBA")
    img_w, img_h = annotation.size
    scale = min(ann_w / img_w, ann_h / img_h)
    resized_w = max(1, int(round(img_w * scale)))
    resized_h = max(1, int(round(img_h * scale)))
    annotation = annotation.resize((resized_w, resized_h), Image.Resampling.LANCZOS)

    right_offset_px = int(round(pw * cfg.get("right_offset_frac", 0.03)))
    bottom_offset_px = int(round(ph * cfg.get("bottom_offset_frac", 0.03)))
    ann_x = x0 + pw - resized_w - right_offset_px
    ann_y = y0 + ph - resized_h - bottom_offset_px
    canvas.paste(annotation, (ann_x, ann_y), annotation)

    border_color = cfg.get("border_color")
    border_thickness = int(cfg.get("border_thickness", 0))
    if border_color and border_thickness > 0:
        draw = ImageDraw.Draw(canvas)
        for i in range(border_thickness):
            draw.rectangle(
                [ann_x + i, ann_y + i, ann_x + resized_w - 1 - i, ann_y + resized_h - 1 - i],
                outline=border_color,
            )
    return canvas


side_compass_annotation = {
    "path": r"D:\larval chemosensory\Side_compass copy.png",
    "w_frac": 0.24,
    "h_frac": 0.24,
    "right_offset_frac": 0.03,
    "bottom_offset_frac": 0.03,
    "fit": "contain",
    "background": "black",
    "padding_color": "black",
}

for panel_id in ("C", "D"):
    canvas = add_panel_image_annotation(
        canvas, panel_layout, panel_id, side_compass_annotation,
    )


draw = ImageDraw.Draw(canvas)
a_x, a_y, a_w, _a_h = panel_layout["A"]
title_font_size = 54
species_font = load_font_safe("ariali.ttf", title_font_size)
title_font = load_font_safe("arial.ttf", title_font_size)
species_text = "Ooceraea biroi"
suffix_text = " 4th instar brain"
species_w, species_h = text_size(draw, species_text, species_font)
suffix_w, suffix_h = text_size(draw, suffix_text, title_font)
title_w = species_w + suffix_w
title_h = max(species_h, suffix_h)
title_x = a_x + (a_w - title_w) // 2
title_y = max(8, a_y - title_h - inches_to_px(0.06, DPI))
#draw.text((title_x, title_y), species_text, fill="black", font=species_font)
#draw.text((title_x + species_w, title_y), suffix_text, fill="black", font=title_font)

canvas = register_figure_canvas(
    'Central Brain Supplement', canvas,
    panel_configs=panels, panel_layout=panel_layout,
)

display(canvas)

# Export

In [ ]:
# Run all figure cells first so their final canvases are registered.
# Filenames are taken from the Markdown heading immediately above each figure cell.
from pathlib import Path
from PIL import Image
import ast
import json
import re

EXPORT_DIR = Path(r"D:\larval chemosensory\Written\Round 8\Exports Figures")
NOTEBOOK_PATH = Path("larval_chemosensory.ipynb")


def _markdown_heading_text(markdown_source):
    for line in str(markdown_source).splitlines():
        stripped = line.strip()
        if stripped.startswith("#"):
            return stripped.lstrip("#").strip()
    return None


def _registration_key_from_call(node):
    if not isinstance(node, ast.Call):
        return None
    function = node.func
    is_register_call = (
        isinstance(function, ast.Name) and function.id == "register_figure_canvas"
    ) or (
        isinstance(function, ast.Attribute) and function.attr == "register_figure_canvas"
    )
    if not is_register_call or not node.args:
        return None
    first_arg = node.args[0]
    if isinstance(first_arg, ast.Constant) and isinstance(first_arg.value, str):
        return first_arg.value
    return None


def markdown_registration_names(notebook_path=NOTEBOOK_PATH):
    """Return (registry key, Markdown filename) pairs in notebook order."""
    notebook_path = Path(notebook_path)
    if not notebook_path.exists():
        raise FileNotFoundError(
            f"Cannot find {notebook_path}. Run Export from the notebook's directory "
            "or update NOTEBOOK_PATH."
        )

    notebook_data = json.loads(notebook_path.read_text(encoding="utf-8"))
    pending_heading = None
    registrations = []
    seen_registry_keys = set()

    for cell in notebook_data.get("cells", []):
        cell_type = cell.get("cell_type")
        source = "".join(cell.get("source", []))

        if cell_type == "markdown":
            heading = _markdown_heading_text(source)
            if heading:
                pending_heading = heading
            continue

        if cell_type != "code":
            continue

        if pending_heading is None:
            continue

        try:
            tree = ast.parse(source)
        except SyntaxError:
            pending_heading = None
            continue

        for node in ast.walk(tree):
            registry_key = _registration_key_from_call(node)
            if registry_key is None:
                continue
            if registry_key in seen_registry_keys:
                raise ValueError(
                    f"The registry key {registry_key!r} appears under more than one figure heading."
                )
            seen_registry_keys.add(registry_key)
            registrations.append((registry_key, pending_heading))

        pending_heading = None

    if not registrations:
        raise RuntimeError("No Markdown-heading/figure-registration pairs were found.")
    return registrations


def _safe_markdown_filename(markdown_heading):
    # Windows does not allow these characters in filenames; hashtags have
    # already been removed by _markdown_heading_text().
    name = re.sub(r'[<>:"/\\|?*]+', "_", str(markdown_heading)).strip(" .")
    return re.sub(r"\s+", " ", name) or "figure"


def export_registered_canvases_from_markdown(
    output_dir,
    notebook_path=NOTEBOOK_PATH,
    source_dpi=DPI,
    target_dpi=600,
):
    """Export registered figures as PNG and LZW-compressed TIFF files."""
    if not FIGURE_CANVASES:
        raise RuntimeError("No canvases are registered. Run the figure cells before Export.")

    output_path = Path(output_dir)
    png_dir = output_path / "PNG"
    tiff_dir = output_path / "TIFF"
    png_dir.mkdir(parents=True, exist_ok=True)
    tiff_dir.mkdir(parents=True, exist_ok=True)

    registrations = markdown_registration_names(notebook_path)
    available_registrations = [
        (registry_key, markdown_heading)
        for registry_key, markdown_heading in registrations
        if registry_key in FIGURE_CANVASES
    ]
    skipped_headings = [
        markdown_heading
        for registry_key, markdown_heading in registrations
        if registry_key not in FIGURE_CANVASES
    ]
    if not available_registrations:
        raise RuntimeError(
            "None of the figures found below Markdown headings are registered. "
            "Run the figure cells before Export."
        )

    filename_owners = {}
    for _registry_key, markdown_heading in available_registrations:
        safe_name = _safe_markdown_filename(markdown_heading)
        collision_key = safe_name.casefold()
        if collision_key in filename_owners:
            raise ValueError(
                f"Markdown headings {filename_owners[collision_key]!r} and "
                f"{markdown_heading!r} produce the same filename {safe_name!r}."
            )
        filename_owners[collision_key] = markdown_heading

    scale = float(target_dpi) / float(source_dpi)
    written = {"PNG": [], "TIFF": []}

    for registry_key, markdown_heading in available_registrations:
        canvas = FIGURE_CANVASES[registry_key]
        trim_options = FIGURE_CANVAS_TRIM_OPTIONS.get(registry_key, {})
        final_canvas = trim_canvas_to_artboard(canvas, **trim_options)
        if scale != 1.0:
            final_canvas = final_canvas.resize(
                (
                    int(round(final_canvas.width * scale)),
                    int(round(final_canvas.height * scale)),
                ),
                Image.Resampling.LANCZOS,
            )

        filename = _safe_markdown_filename(markdown_heading)
        png_path = png_dir / f"{filename}.png"
        tiff_path = tiff_dir / f"{filename}.tiff"

        final_canvas.save(
            png_path,
            format="PNG",
            dpi=(target_dpi, target_dpi),
        )
        final_canvas.save(
            tiff_path,
            format="TIFF",
            compression="tiff_lzw",
            dpi=(target_dpi, target_dpi),
        )
        written["PNG"].append(png_path)
        written["TIFF"].append(tiff_path)

    return written, skipped_headings


exported_figure_paths, skipped_figure_headings = export_registered_canvases_from_markdown(
    EXPORT_DIR,
    notebook_path=NOTEBOOK_PATH,
    source_dpi=DPI,
    target_dpi=600,
)

figure_count = len(exported_figure_paths["PNG"])
print(f"Exported {figure_count} figure(s) as PNG and TIFF at 600 DPI to {EXPORT_DIR}")
for format_name in ("PNG", "TIFF"):
    print(f"\n{format_name} folder: {EXPORT_DIR / format_name}")
    for exported_path in exported_figure_paths[format_name]:
        print(f"  {exported_path.name}")

if skipped_figure_headings:
    print("\nSkipped because their figure cells have not been run:")
    for heading in skipped_figure_headings:
        print(f"  {heading}")


# Scale Bars

In [ ]:
# Run the figure cells first so their final panel configurations are registered.
print_all_scale_bar_summaries()


# Figure Source Paths

In [ ]:
# Build a Markdown inventory from active absolute paths in figure cells.
# Paths in comments, helper cells, and export cells are intentionally excluded.
from collections import OrderedDict
from IPython.display import Markdown, display
import ast
import json
import ntpath
import re
from pathlib import Path

PATH_SUMMARY_NOTEBOOK = Path("larval_chemosensory.ipynb")
WINDOWS_ABSOLUTE_PATH_RE = re.compile(r"^[A-Za-z]:[\\/]")


def _path_summary_heading(markdown_source):
    for line in str(markdown_source).splitlines():
        stripped = line.strip()
        if stripped.startswith("#"):
            return stripped.lstrip("#").strip()
    return None


def _is_figure_registration(node):
    if not isinstance(node, ast.Call):
        return False
    function = node.func
    return (
        isinstance(function, ast.Name)
        and function.id == "register_figure_canvas"
    ) or (
        isinstance(function, ast.Attribute)
        and function.attr == "register_figure_canvas"
    )


def _active_absolute_paths(tree):
    paths = OrderedDict()
    for node in ast.walk(tree):
        if not (isinstance(node, ast.Constant) and isinstance(node.value, str)):
            continue
        path = node.value
        if not WINDOWS_ABSOLUTE_PATH_RE.match(path):
            continue
        canonical_path = ntpath.normcase(ntpath.normpath(path))
        paths.setdefault(canonical_path, path)
    return paths


def build_figure_path_markdown(notebook_path=PATH_SUMMARY_NOTEBOOK):
    notebook_path = Path(notebook_path)
    if not notebook_path.exists():
        raise FileNotFoundError(
            f"Cannot find {notebook_path}. Run this cell from the notebook's "
            "directory or update PATH_SUMMARY_NOTEBOOK."
        )

    notebook_data = json.loads(notebook_path.read_text(encoding="utf-8"))
    current_heading = None
    figure_paths = OrderedDict()

    for cell_index, cell in enumerate(notebook_data.get("cells", [])):
        source = "".join(cell.get("source", []))
        if cell.get("cell_type") == "markdown":
            heading = _path_summary_heading(source)
            if heading:
                current_heading = heading
            continue
        if cell.get("cell_type") != "code":
            continue

        try:
            tree = ast.parse(source)
        except SyntaxError as error:
            raise SyntaxError(
                f"Cannot build the path summary because code cell {cell_index} "
                f"does not parse: {error.msg}"
            ) from error

        if not any(_is_figure_registration(node) for node in ast.walk(tree)):
            continue

        heading = current_heading or f"Code cell {cell_index}"
        paths_for_figure = figure_paths.setdefault(heading, OrderedDict())
        paths_for_figure.update(_active_absolute_paths(tree))

    unique_paths = OrderedDict()
    for heading, paths_for_figure in figure_paths.items():
        for canonical_path, display_path in paths_for_figure.items():
            path_record = unique_paths.setdefault(
                canonical_path,
                {"path": display_path, "figures": []},
            )
            if heading not in path_record["figures"]:
                path_record["figures"].append(heading)

    if not unique_paths:
        raise RuntimeError("No absolute source paths were found in registered figure cells.")

    lines = [
        "## Unique source-image paths",
        "",
        (
            f"**{len(unique_paths)} unique source-image paths** are used across "
            f"**{len(figure_paths)} figure sections**. Paths are normalized "
            "case-insensitively and listed once, under the first figure in which "
            "they appear."
        ),
    ]

    for heading, paths_for_figure in figure_paths.items():
        first_seen_paths = [
            canonical_path
            for canonical_path in paths_for_figure
            if unique_paths[canonical_path]["figures"][0] == heading
        ]
        lines.extend(["", f"### {heading}", ""])

        if not first_seen_paths:
            shared_with = []
            for canonical_path in paths_for_figure:
                first_figure = unique_paths[canonical_path]["figures"][0]
                if first_figure not in shared_with:
                    shared_with.append(first_figure)
            lines.append(
                f"All {len(paths_for_figure)} source paths are already listed under "
                + ", ".join(f"**{name}**" for name in shared_with)
                + "."
            )
            continue

        for canonical_path in first_seen_paths:
            path_record = unique_paths[canonical_path]
            display_path = path_record["path"].replace("`", r"\`")
            also_used_by = path_record["figures"][1:]
            suffix = ""
            if also_used_by:
                suffix = " — also used by: " + ", ".join(also_used_by)
            lines.append(f"- `{display_path}`{suffix}")

    return "\n".join(lines)


figure_path_markdown = build_figure_path_markdown()
display(Markdown(figure_path_markdown))


# Raw Microscopy Provenance

In [ ]:
# Regenerate the machine-readable CSV and render the Markdown provenance table.
from IPython.display import Markdown, display
from build_figure_provenance import MARKDOWN_PATH, build_provenance

provenance_rows = build_provenance()
display(Markdown(MARKDOWN_PATH.read_text(encoding="utf-8")))
